In [1]:
# %%
import os
os.environ["OMP_NUM_THREADS"] = "4"
import random
import functools
import numpy as np
import scipy.ndimage as ndi
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ===== root / data =====
ROOT_DIR = "."
FRAMES_DIR = os.path.join(ROOT_DIR, "Frames")

TRAIN_CACHE_FOLDERS = [
    "LearnCache_seq_fast_4x4",
    "LearnCache_seq_fast_A_House",
    "LearnCache_seq_fast_AZUMA",
    "LearnCache_seq_fast_CASA",
    "LearnCache_seq_fast_House_H",
    "LearnCache_seq_fast_Smith",
    "LearnCache_seq_fast_Kidoshina",
    "LearnCache_seq_fast_Kidasaki",
    "LearnCache_seq_fast_Kidoshaki_House",
]

VAL_CACHE_FOLDERS = [
    "LearnCache_seq_fast_Eagle_Feather",
]

# ===== geometry =====
VOX_CM = 20.0
GRID_MARGIN = 1
CTX_LEN = 1
DIRECTION_NEXT_OFFSET = 1
PATCH_RADIUS = 2
CROP_MARGIN = 2
DIRECTION_DIM = 3
MAX_TRAIN_NODES = 512

# ===== bridge / frontier helper defaults =====
SURFACE_HOLE_FILL_RADIUS = 2
SURFACE_HOLE_MAX_VOXELS = 64
SURFACE_HOLE_CONNECTIVITY = 2
SURFACE_BOUNDARY_NEIGHBOR_THRESHOLD = 8

PREVIEW_HORIZON = 12
RAG_PATTERN_OVERLAP_THRESH = 0.05
RAG_PATTERN_CLEARANCE = 1
RAG_START_BAND_Q = 0.18
BRIDGE_STEP_DILATE = 0

# ===== model =====
BATCH_SIZE = 3
NUM_WORKERS = 0
PIN_MEMORY = True

D_MODEL = 240
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1

# ===== training =====
EPOCHS = 120
LR = 1e-4
WD = 1e-4

# RAG Training hyperparameters
RAG_LR = 5e-5  # Lower learning rate for RAG encoder
RAG_TEMPERATURE = 0.07  # Temperature for contrastive loss
RAG_ENABLE_TRAINING = True  # Toggle RAG encoder training
PATTERNS_PER_SCENE = 4  # Number of patterns to retrieve per scene
# Initialize BANK_EMBEDDINGS as global variable (add after line 2944)
BANK_EMBEDDINGS = None

device: cuda


In [2]:
#sub config setting



#hole filling config
ENABLE_SURFACE_HOLE_FILL = True
SURFACE_HOLE_FILL_RADIUS = 2          # closing 半径，先用 1
SURFACE_HOLE_MAX_VOXELS = 64          # 只填体素数 <= 64 的孔
SURFACE_HOLE_CONNECTIVITY = 2         # 1=>6连通, 2=>26连通

# ===== surface-boundary frontier config =====
SURFACE_BOUNDARY_NEIGHBOR_THRESHOLD = 8

# pattern overlap 
RAG_PATTERN_OVERLAP_THRESH = 0.05  # overlap ratio above which we shift the pattern
RAG_PATTERN_CLEARANCE   = 1      # extra gap voxels after shifting
RAG_START_BAND_Q =1.0            # reverse-direction start band quantile

#bridge config
BRIDGE_STEP_DILATE      = 1      # shell thickness per step (keep 1= thin shell)

MANAGER_FUTURE_HORIZON = 12 #
PREVIEW_HORIZON = max(2, MANAGER_FUTURE_HORIZON)

# ===== surface-boundary frontier config =====
SURFACE_BOUNDARY_NEIGHBOR_THRESHOLD = 8

# ===== lock-state classes =====
LOCK_UNKNOWN = 0
LOCK_SOLID = 1
LOCK_EXTERIOR = 2
LOCK_INTERIOR = 3
LOCK_BLOCKED = 4
LOCK_STATE_NAMES = (
    "unknown",
    "solid",
    "exterior_empty",
    "interior_empty",
    "blocked",
)
NUM_LOCK_CLASSES = len(LOCK_STATE_NAMES)

In [3]:
NEIGHBOR_6 = [
    (-1, 0, 0),
    ( 1, 0, 0),
    ( 0,-1, 0),
    ( 0, 1, 0),
    ( 0, 0,-1),
    ( 0, 0, 1),
]

NEIGHBOR_26 = [
    (dx, dy, dz)
    for dx in (-1, 0, 1)
    for dy in (-1, 0, 1)
    for dz in (-1, 0, 1)
    if not (dx == 0 and dy == 0 and dz == 0)
]



STRUCT6 = ndi.generate_binary_structure(rank=3, connectivity=1)
STRUCT26 = ndi.generate_binary_structure(rank=3, connectivity=3)

Data Preporcess

In [4]:
def build_learncache_sources(frames_dir, folder_names):
    sources = []
    for name in folder_names:
        seq_root = os.path.join(frames_dir, name)
        cache_dir = os.path.join(seq_root, "frame_cache")
        chain_npy = os.path.join(seq_root, "chain_indices.npy")

        if not os.path.isdir(seq_root):
            print(f"[WARN] missing seq_root: {seq_root}")
            continue
        if not os.path.isdir(cache_dir):
            print(f"[WARN] missing frame_cache: {cache_dir}")
            continue
        if not os.path.isfile(chain_npy):
            print(f"[WARN] missing chain_indices.npy: {chain_npy}")
            continue

        chain_ids = np.load(chain_npy).astype(np.int32)
        sources.append({
            "name": name,
            "seq_root": seq_root,
            "cache_dir": cache_dir,
            "chain_ids": chain_ids,
        })
        print(f"[OK] {name}: frames={len(chain_ids)}")
    return sources

def build_sequence_index(sources):
    seqs = []
    for si, src in enumerate(sources):
        seqs.append({
            "source_idx": si,
            "source_name": src["name"],
            "cache_dir": src["cache_dir"],
            "chain_ids": src["chain_ids"],
            "length": len(src["chain_ids"]),
        })
    return seqs

train_sources = build_learncache_sources(FRAMES_DIR, TRAIN_CACHE_FOLDERS)
val_sources = build_learncache_sources(FRAMES_DIR, VAL_CACHE_FOLDERS)

train_seq_items = build_sequence_index(train_sources)
val_seq_items = build_sequence_index(val_sources)

all_paths = sorted({
    os.path.join(seq["cache_dir"], f"{int(fid):06d}.npz")
    for seq in (train_seq_items + val_seq_items)
    for fid in seq["chain_ids"]
})
UID2PATH = list(all_paths)
PATH2UID = {p: i for i, p in enumerate(UID2PATH)}# make into a dict to call
#print(PATH2UID)

print("train seqs:", len(train_seq_items))
print("val seqs:", len(val_seq_items))
print("unique frame paths:", len(UID2PATH))

[OK] LearnCache_seq_fast_4x4: frames=7
[OK] LearnCache_seq_fast_A_House: frames=9
[OK] LearnCache_seq_fast_AZUMA: frames=10
[WARN] missing seq_root: .\Frames\LearnCache_seq_fast_CASA
[OK] LearnCache_seq_fast_House_H: frames=14
[WARN] missing seq_root: .\Frames\LearnCache_seq_fast_Smith
[OK] LearnCache_seq_fast_Kidoshina: frames=373
[WARN] missing seq_root: .\Frames\LearnCache_seq_fast_Kidasaki
[OK] LearnCache_seq_fast_Kidoshaki_House: frames=21
[OK] LearnCache_seq_fast_Eagle_Feather: frames=13
train seqs: 6
val seqs: 1
unique frame paths: 447


In [5]:
@functools.lru_cache(maxsize=4096)
def _load_frame_cache_lru(npz_path: str):
    z = np.load(npz_path, allow_pickle=False)
    if "empty" in z.files:
        return None, None
    centers = z["centers_cm"].astype(np.float32)
    local = z["local_pts"].astype(np.float32)
    return centers, local

def load_frame_cache_by_uid(uid: int):
    path = UID2PATH[int(uid)]
    return _load_frame_cache_lru(path)

def clear_frame_cache():
    _load_frame_cache_lru.cache_clear()
    print("frame cache cleared")

centers,_ = load_frame_cache_by_uid(0)
print(len(centers), centers[:5])

1940 [[1550.  390.  390.]
 [1550.  410.  390.]
 [1550.  430.  390.]
 [1550.  450.  390.]
 [1550.  470.  390.]]


In [6]:
# ===== 3D voxel grid building =====
def centers_to_vox_idx(centers_cm: np.ndarray, vox_cm: float = VOX_CM):# getting voxel indices in coordinate
    return np.floor(centers_cm / vox_cm).astype(np.int32)

def build_voxel_grid_from_centers(centers_cm: np.ndarray,margin: int = GRID_MARGIN):
    if centers_cm is None or len(centers_cm) == 0:
        raise ValueError("centers_cm is empty or None")
    vox_id = centers_to_vox_idx(centers_cm)

    lo = vox_id.min(axis=0) - margin
    hi = vox_id.max(axis=0) + margin + 1
    shape = tuple((hi - lo).tolist())  # [Dx, Dy, Dz] in x/y/z indexing

    grid = np.zeros(shape, dtype=np.uint8)
    shifted = vox_id - lo[None, :]
    valid = np.all((shifted >= 0) & (shifted < np.array(shape)[None, :]), axis=1)
    shifted = shifted[valid]
    grid[shifted[:, 0], shifted[:, 1], shifted[:, 2]] = 1

    meta = {
        "grid_lo": lo.astype(np.int32),
        "grid_hi": hi.astype(np.int32),
    }
    return grid, meta # grid of model , the valid region edge

    

def load_voxel_grid_by_uid(uid: int):
    centers_cm, _ = load_frame_cache_by_uid(uid)
    if centers_cm is None:          # ← empty frame, return gracefully
        return None, None
    grid, meta = build_voxel_grid_from_centers(centers_cm)

    return grid, meta

def align_many_voxel_grids(grid_meta_list):
    # Filter out any None entries from empty frames
    grid_meta_list = [(g, m) for g, m in grid_meta_list if g is not None]
    
    if len(grid_meta_list) == 0:
        g = np.zeros((1, 1, 1), dtype=np.uint8)
        meta = {
            "grid_lo": np.zeros((3,), dtype=np.int32),
            "grid_hi": np.ones((3,), dtype=np.int32),
        }
        return [g], meta

    los = np.stack([meta["grid_lo"] for _, meta in grid_meta_list], axis=0)
    his = np.stack([meta["grid_hi"] for _, meta in grid_meta_list], axis=0)
    lo = los.min(axis=0)
    hi = his.max(axis=0)
    shape = tuple((hi - lo).tolist())

    outs = []
    for grid, meta in grid_meta_list:
        out = np.zeros(shape, dtype=np.uint8)
        off = meta["grid_lo"] - lo
        x0, y0, z0 = off.tolist()
        sx, sy, sz = grid.shape
        out[x0:x0+sx, y0:y0+sy, z0:z0+sz] = grid
        outs.append(out)

    meta = {
        "grid_lo": lo.astype(np.int32),
        "grid_hi": hi.astype(np.int32),
    }
    return outs, meta

In [7]:
# %%
# ===== top-view heat pruning preprocess =====
# 放置位置：
# 建議直接放喺 build_voxel_grid_from_centers / load_voxel_grid_by_uid 後面
# 然後喺 load_voxel_grid_by_uid(...) 入面加一行去調用即可

TOPVIEW_PRUNE_ENABLE = True

# 沿哪個軸做 top-view 投影：
# 0 => yz view, 1 => xz view, 2 => xy(top view)
TOPVIEW_PRUNE_AXIS = 2

# 2D 熱力圖平滑 sigma
TOPVIEW_HEAT_SIGMA = 2.0

# 熱力閾值，相對於最大熱值
TOPVIEW_HEAT_REL_THRESH = 0.22

# 是否只保留 2D 最大連通區
TOPVIEW_KEEP_LARGEST_2D_CC = True

# 對 2D mask 做少量膨脹，避免切太狠
TOPVIEW_MASK_DILATE_ITERS = 1

# 3D 裡面再做一次最小體素量過濾，去掉殘碎小塊
TOPVIEW_MIN_3D_COMPONENT_VOXELS = 24


def keep_largest_connected_component_2d(mask2d: np.ndarray) -> np.ndarray:
    mask2d = (mask2d > 0).astype(np.uint8)
    if mask2d.sum() == 0:
        return mask2d

    struct = ndi.generate_binary_structure(2, 2)
    lab, nlab = ndi.label(mask2d, structure=struct)
    if nlab <= 1:
        return mask2d

    counts = np.bincount(lab.ravel())
    counts[0] = 0
    keep_id = int(np.argmax(counts))
    return (lab == keep_id).astype(np.uint8)


def remove_small_connected_components_3d(mask3d: np.ndarray, min_voxels: int = 24) -> np.ndarray:
    mask3d = (mask3d > 0).astype(np.uint8)
    if mask3d.sum() == 0 or min_voxels <= 1:
        return mask3d

    struct = ndi.generate_binary_structure(3, 2)
    lab, nlab = ndi.label(mask3d, structure=struct)
    if nlab == 0:
        return mask3d

    counts = np.bincount(lab.ravel())
    out = np.zeros_like(mask3d, dtype=np.uint8)
    for lid in range(1, nlab + 1):
        if counts[lid] >= int(min_voxels):
            out[lab == lid] = 1
    return out.astype(np.uint8)


def build_topview_heat_mask_2d(
    grid_u8: np.ndarray,
    axis: int = 2,
    sigma: float = 2.0,
    rel_thresh: float = 0.22,
    keep_largest_cc: bool = True,
    dilate_iters: int = 1,
) -> np.ndarray:
    """
    將 3D occupancy 沿指定軸壓成 2D density map，
    再 blur + threshold，得到主體平面熱區 mask。
    """
    occ = (np.asarray(grid_u8) > 0).astype(np.float32)
    if occ.sum() == 0:
        if axis == 0:
            return np.zeros((occ.shape[1], occ.shape[2]), dtype=np.uint8)
        elif axis == 1:
            return np.zeros((occ.shape[0], occ.shape[2]), dtype=np.uint8)
        else:
            return np.zeros((occ.shape[0], occ.shape[1]), dtype=np.uint8)

    # density map: sum 比 max 更適合做熱區
    dens2d = occ.sum(axis=int(axis))

    if sigma is not None and sigma > 0:
        dens2d = ndi.gaussian_filter(dens2d, sigma=float(sigma))

    vmax = float(dens2d.max())
    if vmax <= 1e-8:
        return np.zeros_like(dens2d, dtype=np.uint8)

    mask2d = (dens2d >= (vmax * float(rel_thresh))).astype(np.uint8)

    if keep_largest_cc:
        mask2d = keep_largest_connected_component_2d(mask2d)

    if dilate_iters > 0:
        mask2d = ndi.binary_dilation(
            mask2d > 0,
            structure=ndi.generate_binary_structure(2, 2),
            iterations=int(dilate_iters),
        ).astype(np.uint8)

    return mask2d.astype(np.uint8)


def apply_topview_mask_to_3d(grid_u8: np.ndarray, mask2d: np.ndarray, axis: int = 2) -> np.ndarray:
    """
    把 2D 平面熱區 mask 回投到 3D。
    """
    grid = (np.asarray(grid_u8) > 0).astype(np.uint8)

    if axis == 0:
        # mask2d shape = [Y, Z]
        if mask2d.shape != (grid.shape[1], grid.shape[2]):
            raise ValueError(f"mask2d shape mismatch: {mask2d.shape} vs {(grid.shape[1], grid.shape[2])}")
        keep3d = np.broadcast_to(mask2d[None, :, :], grid.shape)

    elif axis == 1:
        # mask2d shape = [X, Z]
        if mask2d.shape != (grid.shape[0], grid.shape[2]):
            raise ValueError(f"mask2d shape mismatch: {mask2d.shape} vs {(grid.shape[0], grid.shape[2])}")
        keep3d = np.broadcast_to(mask2d[:, None, :], grid.shape)

    else:
        # axis == 2, mask2d shape = [X, Y]
        if mask2d.shape != (grid.shape[0], grid.shape[1]):
            raise ValueError(f"mask2d shape mismatch: {mask2d.shape} vs {(grid.shape[0], grid.shape[1])}")
        keep3d = np.broadcast_to(mask2d[:, :, None], grid.shape)

    out = (grid * keep3d.astype(np.uint8)).astype(np.uint8)
    return out


def prune_voxel_grid_by_topview_heat(
    grid_u8: np.ndarray,
    axis: int = TOPVIEW_PRUNE_AXIS,
    sigma: float = TOPVIEW_HEAT_SIGMA,
    rel_thresh: float = TOPVIEW_HEAT_REL_THRESH,
    keep_largest_cc: bool = TOPVIEW_KEEP_LARGEST_2D_CC,
    dilate_iters: int = TOPVIEW_MASK_DILATE_ITERS,
    min_3d_component_voxels: int = TOPVIEW_MIN_3D_COMPONENT_VOXELS,
) -> np.ndarray:
    """
    主入口：
      3D grid -> top-view heat mask -> 回投 3D -> 去掉細碎 3D 小塊
    """
    grid = (np.asarray(grid_u8) > 0).astype(np.uint8)
    if grid.sum() == 0:
        return grid

    mask2d = build_topview_heat_mask_2d(
        grid_u8=grid,
        axis=axis,
        sigma=sigma,
        rel_thresh=rel_thresh,
        keep_largest_cc=keep_largest_cc,
        dilate_iters=dilate_iters,
    )

    pruned = apply_topview_mask_to_3d(
        grid_u8=grid,
        mask2d=mask2d,
        axis=axis,
    )

    pruned = remove_small_connected_components_3d(
        pruned,
        min_voxels=min_3d_component_voxels,
    )

    # fallback：如果切完幾乎冇咗，就返回原圖，避免 dataset 崩
    if pruned.sum() == 0:
        return grid

    return pruned.astype(np.uint8)

DatasetLoader

In [8]:
#frontier 3d
def make_ball_struct(radius=1):
    r = int(radius)
    if r <= 0:
        return np.ones((1, 1, 1), dtype=np.uint8)
    xs, ys, zs = np.mgrid[-r:r+1, -r:r+1, -r:r+1]
    ball = (xs**2 + ys**2 + zs**2) <= radius**2
    return ball.astype(bool)

def fill_small_surface_holes_3d(
        surface,
        radius = SURFACE_HOLE_FILL_RADIUS,
        max_hole_voxels = SURFACE_HOLE_MAX_VOXELS,
        connectivity = SURFACE_HOLE_CONNECTIVITY,
    ):
    if max_hole_voxels <= 0:
        return surface

    struct_close = make_ball_struct(radius)
    closed = ndi.binary_closing(surface, structure=struct_close).astype(np.uint8)

    fill_cand = (closed >0) & (surface == 0)
    if not fill_cand.any():
        return surface
    struct_cc = ndi.generate_binary_structure(3, connectivity)
    lab, n_lab = ndi.label(fill_cand, structure=struct_cc)
    if n_lab == 0:
        return surface
    
    keep = np.zeros_like(surface, dtype=np.uint8)
    counts = np.bincount(lab.ravel())

    for lid in range(1, n_lab + 1):
        cnt = int(counts[lid])
        if cnt <= int(max_hole_voxels):
            keep[lab == lid] = 1

    repaired = ((surface > 0) | (keep > 0)).astype(np.uint8)
    return repaired

def shift3d(x, dx, dy, dz):
    X, Y, Z = x.shape
    out = np.zeros_like(x)

    sx0 = max(0, -dx)
    sx1 = min(X, X - dx)
    sy0 = max(0, -dy)
    sy1 = min(Y, Y - dy)
    sz0 = max(0, -dz)
    sz1 = min(Z, Z - dz)

    tx0 = max(0, dx)
    tx1 = min(X, X + dx)
    ty0 = max(0, dy)
    ty1 = min(Y, Y + dy)
    tz0 = max(0, dz)
    tz1 = min(Z, Z + dz)

    out[tx0:tx1, ty0:ty1, tz0:tz1] = x[sx0:sx1, sy0:sy1, sz0:sz1]
    return out

def shift_volume_combined(volume_u8, shift_voxels, grow_direction_np, lateral_offset_np=np.array([0, 0, 0])):
    """
    結合了「沿生長方向推進」與「側向(XY)偏移」的 3D 移動函數。
    """
    if shift_voxels == 0 and np.all(lateral_offset_np == 0):
        return volume_u8.copy()
        
    # 1. 計算沿著生長方向的位移向量
    g = np.asarray(grow_direction_np, dtype=np.float64)
    g = g / (np.linalg.norm(g) + 1e-8)
    forward_shift = g * shift_voxels
    
    # 2. 加上側向的位移向量 (XY 偏移)
    total_shift = forward_shift + np.asarray(lateral_offset_np, dtype=np.float64)
    
    # 3. 轉換為整數 Grid 偏移量
    dx, dy, dz = np.round(total_shift).astype(np.int32)
    dx, dy, dz = int(dx), int(dy), int(dz)
    
    X, Y, Z = volume_u8.shape
    out = np.zeros_like(volume_u8)
    
    # 裁切邊界防呆
    sx0, sx1 = max(0, -dx), min(X, X - dx)
    sy0, sy1 = max(0, -dy), min(Y, Y - dy)
    sz0, sz1 = max(0, -dz), min(Z, Z - dz)
    tx0, tx1 = max(0,  dx), min(X, X + dx)
    ty0, ty1 = max(0,  dy), min(Y, Y + dy)
    tz0, tz1 = max(0,  dz), min(Z, Z + dz)
    
    if (sx1 > sx0) and (sy1 > sy0) and (sz1 > sz0):
        out[tx0:tx1, ty0:ty1, tz0:tz1] = volume_u8[sx0:sx1, sy0:sy1, sz0:sz1]
        
    return out.astype(np.uint8)

def surface_neighbor_count_26_np(surface):
    surface = (surface > 0).astype(np.uint8)
    cnt = np.zeros_like(surface, dtype = np.uint16)
    for dx, dy, dz in NEIGHBOR_26:
        cnt += (shift3d(surface, dx, dy, dz) > 0).astype(np.uint16)
    return cnt

def surface_boundary_from_surface_3d(surface, resolved_mask = None, threshold=SURFACE_BOUNDARY_NEIGHBOR_THRESHOLD):
    """
    surface-boundary 定义：
      先修补小孔，再对修补后的 surface 用邻居数找 boundary。
    """
    surface = (surface > 0).astype(np.uint8)
    surface = fill_small_surface_holes_3d(surface)

    nb = surface_neighbor_count_26_np(surface)
    boundary = (surface > 0) & (nb < int(threshold))

    if resolved_mask is not None:
        boundary = boundary & (resolved_mask == 0)
    
    return boundary.astype(np.uint8)


def frontier_from_known_3d(known, resolved_mask = None):
    surface =(known > 0).astype(np.uint8)
    frontier = surface_boundary_from_surface_3d(surface,resolved_mask = resolved_mask)
    return frontier.astype(np.uint8)
    

In [9]:
# pattern shifting
def compute_pattern_shift_along_direction(
    preview_known_u8,
    initial_known_u8,
    grow_direction_np,
    clearance=RAG_PATTERN_CLEARANCE,
    overlap_thresh=RAG_PATTERN_OVERLAP_THRESH, # 保留參數以防其他地方報錯，但內部不再使用
):
    """
    [升級版] 絕對停靠邏輯 (Absolute Docking)：
    不論 pattern 初始被放在哪裡，都強制計算出需要的位移量，
    將 pattern 的「車尾」精準對齊到現有建築的「車頭」。
    允許負數(向後拉)與正數(向前推)。
    """
    preview_bin = (np.asarray(preview_known_u8) > 0)
    known_bin   = (np.asarray(initial_known_u8) > 0)
 
    # 如果任一方是空的，就不用移動
    if int(preview_bin.sum()) == 0 or int(known_bin.sum()) == 0:
        return 0
    g = np.asarray(grow_direction_np, dtype=np.float64)
    gn = np.linalg.norm(g)
    g = g / gn

    # 1. 攞出灰色建築所有的 3D 坐標點
    known_coords = np.argwhere(known_bin).astype(np.float64)

    # 2. 【新增邏輯】忽略自下往上第 1, 2 層 (假設 Z 軸為垂直高度，Index = 2)
    # 只保留 Z >= 2 的點 (即係忽略 Z=0 同 Z=1 嘅底層)
    ignore_bottom_layers = 3
    valid_mask = known_coords[:, 2] >= ignore_bottom_layers
    filtered_coords = known_coords[valid_mask]

    # 4. 計算灰色建築的「車頭」(最突出點) 
    # 今次只會用過濾後嘅主體結構去計，唔會被突出的地基影響
    known_proj_max = float((filtered_coords @ g).max())
    
    # 計算紅色圖案的「車尾」(最後方點)
    pattern_back_proj = float((np.argwhere(preview_bin).astype(np.float64) @ g).min())

 
    # ❌ 刪除原本的 overlap_ratio 判斷
    # ❌ 刪除原本的 raw_shift <= 0 判斷
    
    # 強制計算兩者的距離差 (可能為正，也可能為負)
    raw_shift = known_proj_max - pattern_back_proj
    
    # 加上一點間隙 (clearance)，確保不會完全黏死
    final_shift = raw_shift + float(clearance)
    
    return int(np.round(final_shift))

# pattern frontier
def _reverse_direction_start_band_np(pattern_delta_u8, grow_direction_np, q=0.18):
    pattern_delta_u8 = (np.asarray(pattern_delta_u8) > 0).astype(np.uint8)
    xyz = np.argwhere(pattern_delta_u8 > 0)
    if xyz.shape[0] == 0:
        return np.zeros_like(pattern_delta_u8, dtype=np.uint8)

    g = np.asarray(grow_direction_np, dtype=np.float32)
    gn = float(np.linalg.norm(g))
    if gn < 1e-6:
        g = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
    else:
        g = g / gn

    proj = xyz.astype(np.float32) @ g
    #thr = float(np.quantile(proj, max(0.01, min(0.49, float(q)))))
    thr = float(np.quantile(proj, max(0.01, q)))
    keep = proj <= thr

    band = np.zeros_like(pattern_delta_u8, dtype=np.uint8)
    band_xyz = xyz[keep]
    if band_xyz.shape[0] == 0:
        band_xyz = xyz[np.argmin(proj)][None, :]
    band[band_xyz[:, 0], band_xyz[:, 1], band_xyz[:, 2]] = 1
    return band.astype(np.uint8)

In [10]:
# %%
# bridge part (NEWER: select source frontier band first, then converge to pattern start)

def _make_orthonormal_basis_from_dir_np(grow_direction_np):
    g = np.asarray(grow_direction_np, dtype=np.float32)
    gn = float(np.linalg.norm(g))
    if gn < 1e-6:
        u = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
    else:
        u = g / gn

    ref = np.asarray([0.0, 0.0, 1.0], dtype=np.float32)
    if abs(float(np.dot(u, ref))) > 0.95:
        ref = np.asarray([0.0, 1.0, 0.0], dtype=np.float32)

    v = np.cross(u, ref)
    v = v / (np.linalg.norm(v) + 1e-8)
    w = np.cross(u, v)
    w = w / (np.linalg.norm(w) + 1e-8)
    return u.astype(np.float32), v.astype(np.float32), w.astype(np.float32)


def _project_points_to_plane_np(xyz, v, w):
    xyz = xyz.astype(np.float32)
    pv = xyz @ v
    pw = xyz @ w
    return pv, pw


def _select_front_frontier_band_by_pattern_bbox_np(
    initial_known_u8,
    current_frontier_u8,
    pattern_start_u8,
    grow_direction_np,
    front_depth_q=0.90,
    bbox_margin=1.5,
):
    """
    核心：
    1) 只在 grow_direction 前端的 frontier 中找候選
    2) 用 pattern_start_frontier 在垂直 grow_direction 的平面上的 bbox
       去裁剪 source frontier band
    """
    current_xyz = np.argwhere(current_frontier_u8 > 0).astype(np.int32)
    pattern_xyz = np.argwhere(pattern_start_u8 > 0).astype(np.int32)

    out = np.zeros_like(current_frontier_u8, dtype=np.uint8)

    if current_xyz.shape[0] == 0 or pattern_xyz.shape[0] == 0:
        return out

    u, v, w = _make_orthonormal_basis_from_dir_np(grow_direction_np)

    cur_f = current_xyz.astype(np.float32)
    pat_f = pattern_xyz.astype(np.float32)

    cur_depth = cur_f @ u
    pat_depth = pat_f @ u

    # 只取 input frontier 在 grow 方向最前端 band
    front_thr = float(np.quantile(cur_depth, max(0.50, min(0.99, front_depth_q))))
    front_keep = cur_depth >= front_thr
    front_xyz = current_xyz[front_keep]
    if front_xyz.shape[0] == 0:
        front_xyz = current_xyz[np.argmax(cur_depth)][None, :]

    front_f = front_xyz.astype(np.float32)

    # pattern bbox on (v,w) plane
    pat_pv, pat_pw = _project_points_to_plane_np(pat_f, v, w)
    pv_min, pv_max = float(pat_pv.min()), float(pat_pv.max())
    pw_min, pw_max = float(pat_pw.min()), float(pat_pw.max())
    pat_cv = 0.5 * (pv_min + pv_max)
    pat_cw = 0.5 * (pw_min + pw_max)
    pat_wv = max(1e-6, pv_max - pv_min)
    pat_ww = max(1e-6, pw_max - pw_min)

    # front frontier center on (v,w)
    front_pv, front_pw = _project_points_to_plane_np(front_f, v, w)
    front_cv = float(front_pv.mean())
    front_cw = float(front_pw.mean())

    # 將 pattern bbox center 對齊 input front frontier center
    shift_v = front_cv - pat_cv
    shift_w = front_cw - pat_cw

    box_v_min = pv_min + shift_v - bbox_margin
    box_v_max = pv_max + shift_v + bbox_margin
    box_w_min = pw_min + shift_w - bbox_margin
    box_w_max = pw_max + shift_w + bbox_margin

    keep = (
        (front_pv >= box_v_min) & (front_pv <= box_v_max) &
        (front_pw >= box_w_min) & (front_pw <= box_w_max)
    )
    band_xyz = front_xyz[keep]

    if band_xyz.shape[0] == 0:
        # fallback: front frontier 裡面挑最接近對齊後 bbox center 的點
        dist2 = (front_pv - (pat_cv + shift_v)) ** 2 + (front_pw - (pat_cw + shift_w)) ** 2
        band_xyz = front_xyz[np.argmin(dist2)][None, :]

    out[band_xyz[:, 0], band_xyz[:, 1], band_xyz[:, 2]] = 1
    return out.astype(np.uint8)


def _choose_attachment_pair_np(
    selected_source_frontier_u8,
    pattern_start_u8,
    grow_direction_np,
    lateral_weight=1.5,
    depth_weight=0.35,
):
    src_xyz = np.argwhere(selected_source_frontier_u8 > 0).astype(np.int32)
    dst_xyz = np.argwhere(pattern_start_u8 > 0).astype(np.int32)

    if src_xyz.shape[0] == 0 or dst_xyz.shape[0] == 0:
        return None, None

    g = np.asarray(grow_direction_np, dtype=np.float32)
    gn = float(np.linalg.norm(g))
    if gn < 1e-6:
        g = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
    else:
        g = g / gn

    src_f = src_xyz.astype(np.float32)
    dst_f = dst_xyz.astype(np.float32)

    src_depth = src_f @ g
    dst_depth = dst_f @ g

    src_lat = src_f - src_depth[:, None] * g[None, :]
    dst_lat = dst_f - dst_depth[:, None] * g[None, :]

    lat_dist = np.linalg.norm(src_lat[:, None, :] - dst_lat[None, :, :], axis=-1)
    depth_dist = np.abs(src_depth[:, None] - dst_depth[None, :])

    cost = lateral_weight * lat_dist + depth_weight * depth_dist
    forward_bias = (dst_depth[None, :] - src_depth[:, None])
    cost = cost + 0.10 * np.maximum(-forward_bias, 0.0)

    ij = np.unravel_index(np.argmin(cost), cost.shape)
    src = src_xyz[int(ij[0])]
    dst = dst_xyz[int(ij[1])]
    return src.astype(np.int32), dst.astype(np.int32)


import numpy as np

def _draw_axis_path_np(shape, src_xyz, dst_xyz):
    vol = np.zeros(shape, dtype=np.uint8)
    src = np.asarray(src_xyz, dtype=np.int32)
    dst = np.asarray(dst_xyz, dtype=np.int32)

    def mark(p):
        x, y, z = int(p[0]), int(p[1]), int(p[2])
        if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
            vol[x, y, z] = 1

    delta = dst - src
    max_dist = int(np.max(np.abs(delta)))

    if max_dist == 0:
        mark(src)
        return vol

    # 1. 產生直連兩點的虛擬直線 (密集採樣，確保不漏點)
    # 採樣率設為 max_dist 的 2 倍，保證連續性
    t = np.linspace(0, 1, max_dist * 2 + 1)
    float_pts = src[None, :] + t[:, None] * delta[None, :]

    # 2. 往法線方向 Pull 到最近的 Matrix 網格點 (四捨五入)
    round_pts = np.round(float_pts).astype(np.int32)

    # 過濾掉連續重複的座標，提取出乾淨的「關鍵航點」
    waypoints = [round_pts[0]]
    for pt in round_pts[1:]:
        if not np.array_equal(pt, waypoints[-1]):
            waypoints.append(pt)

    # 3. 透過 Manhattan line 將這些緊密的點鏈接起來
    cur = waypoints[0].copy()
    mark(cur)

    for nxt in waypoints[1:]:
        step_delta = nxt - cur
        order = np.argsort(-np.abs(step_delta))
        
        # 微型 Manhattan 連接 (由於點跟點好近，通常只行 1-2 步)
        for ax in order:
            while cur[ax] != nxt[ax]:
                cur[ax] += 1 if nxt[ax] > cur[ax] else -1
                mark(cur)

    return vol.astype(np.uint8)

def _select_start_frontier_seed_band_np(
    pattern_start_u8,
    grow_direction_np,
    q=RAG_START_BAND_Q,
):
    """
    從 pattern_start volume 裏抽出 canonical start frontier seed。
    這個 seed 是整個局部 pattern 的起始前沿，用來：
      1) 向 -grow_direction reverse bridge 回 input frontier
      2) 向 +grow_direction 交俾 grow phase
    """
    return _reverse_direction_start_band_np(
        pattern_start_u8,
        grow_direction_np,
        q=q,
    ).astype(np.uint8)


def _choose_reverse_attachment_pair_np(
    input_frontier_u8,
    start_frontier_seed_u8,
    grow_direction_np,
    lateral_weight=1.5,
    depth_weight=0.35,
):
    """
    注意角色已經反轉：
      src = start_frontier_seed
      dst = input_frontier
    即從 start seed 沿 reverse grow direction 接返去 input frontier。
    """
    src_xyz = np.argwhere(start_frontier_seed_u8 > 0).astype(np.int32)
    dst_xyz = np.argwhere(input_frontier_u8 > 0).astype(np.int32)

    if src_xyz.shape[0] == 0 or dst_xyz.shape[0] == 0:
        return None, None

    g = np.asarray(grow_direction_np, dtype=np.float32)
    gn = float(np.linalg.norm(g))
    if gn < 1e-6:
        g = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
    else:
        g = g / gn

    src_f = src_xyz.astype(np.float32)
    dst_f = dst_xyz.astype(np.float32)

    src_depth = src_f @ g
    dst_depth = dst_f @ g

    src_lat = src_f - src_depth[:, None] * g[None, :]
    dst_lat = dst_f - dst_depth[:, None] * g[None, :]

    lat_dist = np.linalg.norm(src_lat[:, None, :] - dst_lat[None, :, :], axis=-1)
    depth_dist = np.abs(src_depth[:, None] - dst_depth[None, :])

    # reverse bridge：偏向將 start seed 接到其後方 / 較淺處的 input frontier
    cost = lateral_weight * lat_dist + depth_weight * depth_dist
    reverse_bias = (src_depth[:, None] - dst_depth[None, :])   # 希望 dst 在 src 後方
    cost = cost + 0.10 * np.maximum(-reverse_bias, 0.0)

    ij = np.unravel_index(np.argmin(cost), cost.shape)
    src = src_xyz[int(ij[0])]
    dst = dst_xyz[int(ij[1])]
    return src.astype(np.int32), dst.astype(np.int32)


# ─────────────────────────────────────────────────────────────
# ▌ SECTION A — NUMPY (data-prep)                             ▌
# ─────────────────────────────────────────────────────────────
 
def _compute_lateral_bbox_np(mask_u8, grow_direction_np):
    """
    Compute the lateral bounding-box width and height of a binary mask
    measured in the plane perpendicular to `grow_direction_np`.
 
    Returns
    -------
    w : float  — larger lateral span
    h : float  — smaller lateral span
    area : float — w * h  (BBox area in the lateral plane)
    """
    pts = np.argwhere(mask_u8 > 0).astype(np.float32)
    if pts.shape[0] == 0:
        return 1.0, 1.0, 1.0
 
    g = np.asarray(grow_direction_np, dtype=np.float64)
    g = g / (np.linalg.norm(g) + 1e-8)
 
    # subtract the depth component → lateral coords
    depth_proj = (pts @ g)[:, None] * g[None, :]  # [N, 3]
    lateral = pts - depth_proj                      # [N, 3]
 
    lat_min = lateral.min(axis=0)
    lat_max = lateral.max(axis=0)
    spans = (lat_max - lat_min + 1.0)             # [3]
 
    # The grow-axis span is ~0; sort and take the two larger ones
    spans_sorted = np.sort(spans)[::-1]           # descending
    w = float(spans_sorted[0])
    h = float(spans_sorted[1])
    return w, h, w * h
 
 
def build_reverse_bridge_targets_np(
    initial_known_u8,
    input_frontier_u8,
    pattern_start_u8,
    grow_direction_np,
    step_dilate=0,
    tube_radius=3.0,    # kept for API compat (unused)
    reach_iters=6,      # kept for API compat (unused)
):
    """
    BBox-Interpolated Bridge Target  (replaces the old uniform-stamp sweep).
 
    Key change vs. the original:
      • We still sweep the pattern-end cross-section stamp along -grow_direction
        until it hits the existing building.
      • After sweeping, we TRIM each voxel of the accumulated tube to lie inside
        a cylindrical envelope whose radius interpolates between the two endpoint
        BBox sizes using a cosine schedule:
 
            r(t) = sqrt( ((1-t)*area_A + t*area_B) / π )
 
        where t=0 is the building-frontier end (A) and t=1 is the pattern end (B).
 
    This reduces the supervision of the rigid, constant-width stamp shape while
    still anchoring the model to the correct spatial corridor.  The remaining
    spatial-transition supervision is handed over to the PyTorch envelope losses
    below.
    """
    shape = initial_known_u8.shape
    empty_mask = (initial_known_u8 == 0).astype(np.uint8)
 
    # ── 1. Extract the pattern-side cross-section (endpoint B) ──────────────
    # _select_start_frontier_seed_band_np must already be defined in rewrite_47.py
    start_frontier_seed = _select_start_frontier_seed_band_np(   # noqa: F821
        pattern_start_u8=pattern_start_u8,
        grow_direction_np=grow_direction_np,
        q=RAG_START_BAND_Q,                                       # noqa: F821
    )
 
    src = _centroid_of_mask_np(start_frontier_seed)               # noqa: F821
    dst = _centroid_of_mask_np((input_frontier_u8 > 0).astype(np.uint8))  # noqa: F821
 
    if start_frontier_seed.sum() == 0:
        return np.zeros(shape, dtype=np.uint8), start_frontier_seed, src, dst
 
    # ── 2. Lateral BBox sizes at both endpoints ──────────────────────────────
    _, _, area_A = _compute_lateral_bbox_np(input_frontier_u8, grow_direction_np)
    _, _, area_B = _compute_lateral_bbox_np(start_frontier_seed,  grow_direction_np)
 
    # Corresponding lateral radii (circle-equivalent)
    r_A = max(float(np.sqrt(area_A / np.pi)), 1.0)
    r_B = max(float(np.sqrt(area_B / np.pi)), 1.0)
 
    # ── 3. Sweep direction: pattern → building  (-grow_direction) ───────────
    g = np.asarray(grow_direction_np, dtype=np.float64)
    g = g / (np.linalg.norm(g) + 1e-8)
    step_vec = -g
 
    dx, dy, dz = np.round(step_vec).astype(np.int32)
    if dx == 0 and dy == 0 and dz == 0:
        max_idx = int(np.argmax(np.abs(step_vec)))
        vec_int = [0, 0, 0]
        vec_int[max_idx] = 1 if step_vec[max_idx] > 0 else -1
        dx, dy, dz = vec_int
    dx, dy, dz = int(dx), int(dy), int(dz)
 
    # ── 4. Stamp sweep (same loop as original) ───────────────────────────────
    X, Y, Z = shape
    max_steps = int(X + Y + Z)
 
    stamp = start_frontier_seed.copy()
    bridge_accum = np.zeros(shape, dtype=np.uint8)
    slice_curr = stamp.copy()
 
    for _ in range(max_steps):
        valid_slice = (slice_curr & empty_mask).astype(np.uint8)
        bridge_accum = np.maximum(bridge_accum, valid_slice)
 
        # Stop when the leading edge touches the existing building
        leading_dilated = ndi.binary_dilation(
            slice_curr > 0,
            structure=ndi.generate_binary_structure(3, 1),
            iterations=1,
        ).astype(np.uint8)
 
        if (leading_dilated & (initial_known_u8 > 0)).any():
            break
 
        shifted = np.zeros(shape, dtype=np.uint8)
        sx0, sx1 = max(0, -dx), min(X, X - dx)
        sy0, sy1 = max(0, -dy), min(Y, Y - dy)
        sz0, sz1 = max(0, -dz), min(Z, Z - dz)
        tx0, tx1 = max(0,  dx), min(X, X + dx)
        ty0, ty1 = max(0,  dy), min(Y, Y + dy)
        tz0, tz1 = max(0,  dz), min(Z, Z + dz)
 
        if sx1 > sx0 and sy1 > sy0 and sz1 > sz0:
            shifted[tx0:tx1, ty0:ty1, tz0:tz1] = slice_curr[sx0:sx1, sy0:sy1, sz0:sz1]
 
        slice_curr = shifted
        if slice_curr.sum() == 0:
            break
 
    # ── 5. BBox-interpolated envelope trim ───────────────────────────────────
    # Project each accumulated voxel onto the grow axis → compute t ∈ [0, 1]
    bridge_vox = np.argwhere(bridge_accum > 0).astype(np.float64)
 
    if bridge_vox.shape[0] > 0 and src is not None and dst is not None:
        src_arr = np.array(src, dtype=np.float64)   # pattern end (t = 1, B)
        dst_arr = np.array(dst, dtype=np.float64)   # building end (t = 0, A)
 
        # signed depth from building end toward pattern end
        total_depth = float(np.dot(src_arr - dst_arr, g))
        if abs(total_depth) > 1.0:
            depth_from_A = (bridge_vox - dst_arr) @ g          # [N]
            t = np.clip(depth_from_A / total_depth, 0.0, 1.0)  # [N] in [0, 1]
 
            # Cosine-smoothed area interpolation (softer at the two ends)
            cos_blend = (1.0 - np.cos(np.pi * t)) / 2.0        # 0→0, 0.5→0.5, 1→1
            area_t = (1.0 - cos_blend) * area_A + cos_blend * area_B
            r_t = np.sqrt(area_t / np.pi).clip(min=1.0)        # [N]
 
            # Lateral distance of each voxel from the sweep axis
            # sweep axis passes through dst_arr in direction g
            rel = bridge_vox - dst_arr                          # [N, 3]
            proj_on_g = depth_from_A[:, None] * g[None, :]     # [N, 3]
            lateral = rel - proj_on_g                           # [N, 3]
            lateral_dist = np.linalg.norm(lateral, axis=1)     # [N]
 
            keep = lateral_dist <= r_t
 
            # Keep endpoints unconditionally (always include the anchor seeds)
            keep_pts = bridge_vox[keep].astype(np.int32)
            refined = np.zeros(shape, dtype=np.uint8)
            if keep_pts.shape[0] > 0:
                refined[keep_pts[:, 0], keep_pts[:, 1], keep_pts[:, 2]] = 1
 
            bridge_accum = refined.astype(np.uint8)
 
    # Always include the pattern-side seed (ensures the target is non-empty)
    bridge_accum = np.maximum(bridge_accum, start_frontier_seed).astype(np.uint8)
 
    # ── 6. Optional thickness dilation ──────────────────────────────────────
    if step_dilate > 0:
        bridge_accum = ndi.binary_dilation(
            bridge_accum > 0,
            structure=ndi.generate_binary_structure(3, 1),
            iterations=int(step_dilate),
        ).astype(np.uint8)
        bridge_accum = (bridge_accum * empty_mask).astype(np.uint8)
 
    return (
        bridge_accum.astype(np.uint8),
        start_frontier_seed.astype(np.uint8),
        src,
        dst,
    )


def _centroid_of_mask_np(mask_u8):
    """Returns (x, y, z) centroid of a binary mask, or None if empty."""
    pts = np.argwhere(mask_u8 > 0)
    if pts.shape[0] == 0:
        return None
    return tuple(pts.mean(axis=0).tolist())

def matching_start_frontier(start_frontier_seed_u8, reverse_bridge_seed_u8, grow_direction_np):
    """
    將 start_frontier_seed 限制在 reverse_bridge_seed 於生長方向上的投影寬度內。
    """
    seed_bin = (np.asarray(start_frontier_seed_u8) > 0)
    ref_bin = (np.asarray(reverse_bridge_seed_u8) > 0)

    # 防呆：如果 reference 或 seed 係空嘅，直接回傳原本嘅 seed
    if ref_bin.sum() == 0 or seed_bin.sum() == 0:
        return start_frontier_seed_u8.copy()

    # 1. 將生長方向單位化 (變成標準長度 1 嘅向量)
    g = np.asarray(grow_direction_np, dtype=np.float32)
    gn = float(np.linalg.norm(g))
    if gn < 1e-6:
        g = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
    else:
        g = g / gn

    # 2. 搵出 reverse_bridge_seed 喺生長方向上嘅「投影邊界」
    ref_xyz = np.argwhere(ref_bin).astype(np.float32)
    ref_proj = ref_xyz @ g  # 內積投影
    min_proj = float(np.min(ref_proj))
    max_proj = float(np.max(ref_proj))

    # 3. 對 start_frontier_seed 進行投影並裁剪
    seed_xyz = np.argwhere(seed_bin)
    seed_proj = seed_xyz.astype(np.float32) @ g

    # 加入 0.5 voxel 嘅容差，避免浮點數誤差導致漏咗邊緣嘅點
    keep = (seed_proj >= min_proj - 0.5) & (seed_proj <= max_proj + 0.5)

    kept_xyz = seed_xyz[keep]

    # 4. 畫返出嚟
    out = np.zeros_like(start_frontier_seed_u8, dtype=np.uint8)
    if kept_xyz.shape[0] > 0:
        out[kept_xyz[:, 0], kept_xyz[:, 1], kept_xyz[:, 2]] = 1

    return out.astype(np.uint8)

def build_reverse_bridge_field_targets_np(
    initial_known_u8,
    input_frontier_u8,
    pattern_start_u8,
    grow_direction_np,
    step_dilate=BRIDGE_STEP_DILATE,
    field_reach_iters=2,
):
    """
    field 版 target：
    不是完整 reverse bridge path，
    而是從 start_frontier_seed 附近可達的那一小段 reverse corridor。
    """
    shape = initial_known_u8.shape
    empty_mask = (initial_known_u8 == 0).astype(np.uint8)

    reverse_bridge_target_full, start_frontier_seed, src, dst = build_reverse_bridge_targets_np(
        initial_known_u8=initial_known_u8,
        input_frontier_u8=input_frontier_u8,
        pattern_start_u8=pattern_start_u8,
        grow_direction_np=grow_direction_np,
        step_dilate=step_dilate,
    )

    if reverse_bridge_target_full.sum() == 0 or start_frontier_seed.sum() == 0:
        return (
            np.zeros(shape, dtype=np.uint8),
            reverse_bridge_target_full.astype(np.uint8),
            start_frontier_seed.astype(np.uint8),
            src,
            dst,
        )

    reach = start_frontier_seed.astype(np.uint8)
    for _ in range(int(field_reach_iters)):
        reach = ndi.binary_dilation(
            reach,
            structure=ndi.generate_binary_structure(3, 1),
            iterations=1,
        ).astype(np.uint8)

    reach = (reach * empty_mask).astype(np.uint8)
    reverse_bridge_field_target = (reverse_bridge_target_full * reach).astype(np.uint8)

    if reverse_bridge_field_target.sum() == 0 and src is not None:
        tmp = np.zeros(shape, dtype=np.uint8)
        x, y, z = map(int, src)
        if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
            tmp[x, y, z] = 1
        reverse_bridge_field_target = ndi.binary_dilation(
            tmp,
            structure=ndi.generate_binary_structure(3, 1),
            iterations=1,
        ).astype(np.uint8)
        reverse_bridge_field_target = (reverse_bridge_field_target * empty_mask).astype(np.uint8)

    return (
        reverse_bridge_field_target.astype(np.uint8),
        reverse_bridge_target_full.astype(np.uint8),
        start_frontier_seed.astype(np.uint8),
        src,
        dst,
    )

def build_bridge_converge_targets_np(
    initial_known_u8,
    current_frontier_u8,
    pattern_start_u8,
    grow_direction_np,
    step_dilate=BRIDGE_STEP_DILATE,
):
    """
    新版：
    current_frontier
      -> selected_source_frontier_band
      -> attachment pair
      -> bridge path to pattern_start_frontier
    """
    shape = initial_known_u8.shape
    empty_mask = (initial_known_u8 == 0).astype(np.uint8)

    selected_source_frontier = _select_front_frontier_band_by_pattern_bbox_np(
        initial_known_u8=initial_known_u8,
        current_frontier_u8=current_frontier_u8,
        pattern_start_u8=pattern_start_u8,
        grow_direction_np=grow_direction_np,
        front_depth_q=0.90,
        bbox_margin=1.5,
    )

    src, dst = _choose_attachment_pair_np(
        selected_source_frontier_u8=selected_source_frontier,
        pattern_start_u8=pattern_start_u8,
        grow_direction_np=grow_direction_np,
        lateral_weight=1.5,
        depth_weight=0.35,
    )

    if src is None or dst is None:
        return (
            np.zeros(shape, dtype=np.uint8),
            selected_source_frontier.astype(np.uint8),
            None,
            None,
        )

    path = _draw_axis_path_np(shape, src, dst)

    if step_dilate > 0:
        path = ndi.binary_dilation(
            path,
            structure=ndi.generate_binary_structure(3, 1),
            iterations=int(step_dilate),
        ).astype(np.uint8)

    bridge_target = (path * empty_mask).astype(np.uint8)
    attach = ((pattern_start_u8 > 0).astype(np.uint8) * empty_mask).astype(np.uint8)
    bridge_target = np.maximum(bridge_target, attach).astype(np.uint8)

    return (
        bridge_target,
        selected_source_frontier.astype(np.uint8),
        src,
        dst,
    )

def build_bridge_field_targets_np(
    initial_known_u8,
    current_frontier_u8,
    pattern_start_u8,
    grow_direction_np,
    step_dilate=BRIDGE_STEP_DILATE,
    field_reach_iters=2,
):
    """
    給 single-step bridge potential field 用：
    不是完整 bridge path target，
    而是 bridge full target 中、當前 frontier 一步/兩步可達的那部分 corridor。
    """
    shape = initial_known_u8.shape
    empty_mask = (initial_known_u8 == 0).astype(np.uint8)

    bridge_target_full, selected_source_frontier, src, dst =  build_bridge_converge_targets_np(
        initial_known_u8=initial_known_u8,
        current_frontier_u8=current_frontier_u8,
        pattern_start_u8=pattern_start_u8,
        grow_direction_np=grow_direction_np,
        step_dilate=step_dilate,
    )

    if bridge_target_full.sum() == 0 or selected_source_frontier.sum() == 0:
        return (
            np.zeros(shape, dtype=np.uint8),
            bridge_target_full.astype(np.uint8),
            selected_source_frontier.astype(np.uint8),
            src,
            dst,
        )

    # 只保留 source frontier 附近 field head 真正可表達/可達的局部區域
    reach = selected_source_frontier.astype(np.uint8)
    for _ in range(int(field_reach_iters)):
        reach = ndi.binary_dilation(
            reach,
            structure=ndi.generate_binary_structure(3, 1),
            iterations=1,
        ).astype(np.uint8)

    reach = (reach * empty_mask).astype(np.uint8)
    bridge_field_target = (bridge_target_full * reach).astype(np.uint8)

    # fallback：如果局部 target 太細，至少保留 src 周圍一圈，避免全零
    if bridge_field_target.sum() == 0 and src is not None:
        tmp = np.zeros(shape, dtype=np.uint8)
        x, y, z = map(int, src)
        if 0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]:
            tmp[x, y, z] = 1
        bridge_field_target = ndi.binary_dilation(
            tmp,
            structure=ndi.generate_binary_structure(3, 1),
            iterations=1,
        ).astype(np.uint8)
        bridge_field_target = (bridge_field_target * empty_mask).astype(np.uint8)

    return (
        bridge_field_target.astype(np.uint8),
        bridge_target_full.astype(np.uint8),
        selected_source_frontier.astype(np.uint8),
        src,
        dst,
    )

In [11]:
# DATA loader helper
def _safe_centier_of_mass_u8(mask):
        coords = np.argwhere(mask > 0)
        if coords.shape[0] == 0:
                return np.zeros((3,),dtype=np.float32), False
        return coords.mean(axis=0).astype(np.float32), True

def _normalize_np_vec(vec, eps=1e-6,default=(1.0, 0.0, 0.0)):
        vec = np.asarray(vec, dtype=np.float32)
        n = float(np.linalg.norm(vec))
        if n < eps:
                return np.array(default, dtype=np.float32)
        return (vec/n).astype(np.float32)

def compute_relative_growth_direction(initial_known, preview_union):
    initial_u8 = (np.asarray(initial_known) > 0).astype(np.uint8)
    next_u8 = (np.asarray(preview_union) > 0).astype(np.uint8)
    delta_u8 = ((next_u8 > 0) & (initial_u8 == 0)).astype(np.uint8)

    c0, ok0 = _safe_centier_of_mass_u8(initial_u8)
    c1, ok1 = _safe_centier_of_mass_u8(delta_u8)

    if (not ok0) or (not ok1):
        return np.array([1.0, 0.0, 0.0], dtype=np.float32), np.float32(0.0)

    vec = c1 - c0
    n = float(np.linalg.norm(vec))
    if n < 1e-6:
        return np.array([1.0, 0.0, 0.0], dtype=np.float32), np.float32(0.0)

    return (vec / n).astype(np.float32), np.float32(1.0)

# %%
def _bbox_center_of_mask(mask_u8):
    xyz = np.argwhere(mask_u8 > 0).astype(np.int32)
    if xyz.shape[0] == 0:
        return np.zeros((3,), dtype=np.int32), False
    lo = xyz.min(axis=0)
    hi = xyz.max(axis=0)
    ctr = ((lo + hi) // 2).astype(np.int32)
    return ctr, True


def _shift_mask_by_int_offset(mask_u8, offset_xyz):
    dx, dy, dz = map(int, offset_xyz)
    return shift3d(mask_u8.astype(np.uint8), dx, dy, dz).astype(np.uint8)


def canonicalize_mask_to_center(mask_u8):
    """
    將 mask 平移到 bbox center 落在 volume 中心附近，消除絕對原點偏移。
    """
    mask_u8 = (np.asarray(mask_u8) > 0).astype(np.uint8)
    shape = np.asarray(mask_u8.shape, dtype=np.int32)
    ctr, ok = _bbox_center_of_mask(mask_u8)
    if not ok:
        return mask_u8.copy(), np.zeros((3,), dtype=np.int32)

    target_ctr = shape // 2
    offset = (target_ctr - ctr).astype(np.int32)
    out = _shift_mask_by_int_offset(mask_u8, offset)
    return out.astype(np.uint8), offset.astype(np.int32)


def build_pattern_descriptor_np(mask_local_u8, seed_local_u8, grow_direction_np):
    """
    輕量 descriptor，用來 top-k retrieval。
    """
    mask = (mask_local_u8 > 0).astype(np.uint8)
    seed = (seed_local_u8 > 0).astype(np.uint8)

    xyz = np.argwhere(mask > 0).astype(np.float32)
    if xyz.shape[0] == 0:
        return np.zeros((16,), dtype=np.float32)

    g = np.asarray(grow_direction_np, dtype=np.float32)
    gn = float(np.linalg.norm(g))
    if gn < 1e-6:
        g = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
    else:
        g = g / gn

    depth = xyz @ g
    lat = xyz - depth[:, None] * g[None, :]

    bbox = xyz.max(axis=0) - xyz.min(axis=0) + 1.0
    lat_std = lat.std(axis=0)
    seed_ratio = float(seed.sum()) / max(float(mask.sum()), 1.0)

    desc = np.array([
        float(mask.sum()),
        float(seed.sum()),
        seed_ratio,
        float(bbox[0]), float(bbox[1]), float(bbox[2]),
        float(depth.mean()), float(depth.std()),
        float(lat_std[0]), float(lat_std[1]), float(lat_std[2]),
        float(g[0]), float(g[1]), float(g[2]),
        float((depth > np.median(depth)).mean()),
        float((depth < np.median(depth)).mean()),
    ], dtype=np.float32)
    return desc

import numpy as np
from numba import njit

# =====================================================================
# 1. 核心底層算法：使用 Numba JIT 編譯為機器碼 (極速運行)
# =====================================================================
@njit(nopython=True)
def _evaluate_ray_depth_numba(grid, cx, cy, cz, ax, sign, L):
    """
    以純 C 語言級別速度執行 3D 射線探測。
    包含 Early Stop 機制：若 3x3 核心完全暢通，直接跳過 5x5 計算。
    """
    if L <= 0:
        return False, 0.0
        
    X, Y, Z = grid.shape
    
    # -------------------------------------------------------------
    # 階段 A：Early Stop 檢查 (探測 3x3 核心走廊)
    # -------------------------------------------------------------
    strict_window = 1
    is_strict = True
    
    if ax == 0:
        y0, y1 = max(0, cy - strict_window), min(Y - 1, cy + strict_window)
        z0, z1 = max(0, cz - strict_window), min(Z - 1, cz + strict_window)
        for y in range(y0, y1 + 1):
            for z in range(z0, z1 + 1):
                for step in range(1, L + 1):
                    if grid[cx + step * sign, y, z] > 0:
                        is_strict = False
                        break
                if not is_strict: break
            if not is_strict: break
            
    elif ax == 1:
        x0, x1 = max(0, cx - strict_window), min(X - 1, cx + strict_window)
        z0, z1 = max(0, cz - strict_window), min(Z - 1, cz + strict_window)
        for x in range(x0, x1 + 1):
            for z in range(z0, z1 + 1):
                for step in range(1, L + 1):
                    if grid[x, cy + step * sign, z] > 0:
                        is_strict = False
                        break
                if not is_strict: break
            if not is_strict: break
            
    else: # ax == 2
        x0, x1 = max(0, cx - strict_window), min(X - 1, cx + strict_window)
        y0, y1 = max(0, cy - strict_window), min(Y - 1, cy + strict_window)
        for x in range(x0, x1 + 1):
            for y in range(y0, y1 + 1):
                for step in range(1, L + 1):
                    if grid[x, y, cz + step * sign] > 0:
                        is_strict = False
                        break
                if not is_strict: break
            if not is_strict: break

    # 【輕量化核心】如果 3x3 完全無阻礙，直接給予滿分並提早結束！省下大量算力！
    if is_strict:
        return True, 1.0 
        
    # -------------------------------------------------------------
    # 階段 B：計算 5x5 走廊的平均穿透深度
    # -------------------------------------------------------------
    window = 2
    total_depth = 0
    total_rays = 0
    
    if ax == 0:
        y0, y1 = max(0, cy - window), min(Y - 1, cy + window)
        z0, z1 = max(0, cz - window), min(Z - 1, cz + window)
        total_rays = (y1 - y0 + 1) * (z1 - z0 + 1)
        for y in range(y0, y1 + 1):
            for z in range(z0, z1 + 1):
                depth = L
                for step in range(1, L + 1):
                    if grid[cx + step * sign, y, z] > 0:
                        depth = step - 1
                        break
                total_depth += depth
                
    elif ax == 1:
        x0, x1 = max(0, cx - window), min(X - 1, cx + window)
        z0, z1 = max(0, cz - window), min(Z - 1, cz + window)
        total_rays = (x1 - x0 + 1) * (z1 - z0 + 1)
        for x in range(x0, x1 + 1):
            for z in range(z0, z1 + 1):
                depth = L
                for step in range(1, L + 1):
                    if grid[x, cy + step * sign, z] > 0:
                        depth = step - 1
                        break
                total_depth += depth
                
    else: # ax == 2
        x0, x1 = max(0, cx - window), min(X - 1, cx + window)
        y0, y1 = max(0, cy - window), min(Y - 1, cy + window)
        total_rays = (x1 - x0 + 1) * (y1 - y0 + 1)
        for x in range(x0, x1 + 1):
            for y in range(y0, y1 + 1):
                depth = L
                for step in range(1, L + 1):
                    if grid[x, y, cz + step * sign] > 0:
                        depth = step - 1
                        break
                total_depth += depth
                
    ratio = total_depth / (total_rays * L) if total_rays > 0 else 0.0
    return False, ratio


# =====================================================================
# 2. 上層封裝函數：負責資料準備與邏輯評估 (供主程式呼叫)
# =====================================================================
def find_open_grow_directions(grid_u8, threshold=0.15):
    """
    高級深度探測版 (Numba 極速優化)
    尋找建築圖案中所有有效、無阻擋或斜向的開口生長方向。
    """
    xyz = np.argwhere(grid_u8 > 0)
    if len(xyz) == 0:
        return []

    # 取得 Bounding Box 邊界與中心點
    xmin, ymin, zmin = xyz.min(axis=0)
    xmax, ymax, zmax = xyz.max(axis=0)
    cx, cy, cz = np.round(xyz.mean(axis=0)).astype(int)

    directions_info = []

    # 6 個方向: (所屬軸 0=X/1=Y/2=Z, 正負向 1/-1, 向量)
    dirs = [
        (0,  1, np.array([ 1.,  0.,  0.], dtype=np.float32)),
        (0, -1, np.array([-1.,  0.,  0.], dtype=np.float32)),
        (1,  1, np.array([ 0.,  1.,  0.], dtype=np.float32)),
        (1, -1, np.array([ 0., -1.,  0.], dtype=np.float32)),
        (2,  1, np.array([ 0.,  0.,  1.], dtype=np.float32)),
        (2, -1, np.array([ 0.,  0., -1.], dtype=np.float32)),
    ]

    for ax, sign, vec in dirs:
        # 計算該方向到 Bounding Box 邊界的標準距離 L
        if ax == 0:
            L = max(0, xmax - cx) if sign == 1 else max(0, cx - xmin)
        elif ax == 1:
            L = max(0, ymax - cy) if sign == 1 else max(0, cy - ymin)
        else:
            L = max(0, zmax - cz) if sign == 1 else max(0, cz - zmin)

        if L <= 0:
            continue

        # 呼叫 Numba 加速函數進行探測
        is_strict, ratio = _evaluate_ray_depth_numba(grid_u8, cx, cy, cz, ax, sign, L)

        directions_info.append({
            "vec": vec,
            "is_strict": is_strict,
            "ratio": ratio
        })

    # ================= 評估階段 =================
    valid_directions = []
    
    # 找出所有「確定有缺口」的基準暢通率
    strict_ratios = [d["ratio"] for d in directions_info if d["is_strict"]]

    if len(strict_ratios) > 0:
        base_ratio = np.mean(strict_ratios)
        for d in directions_info:
            if d["is_strict"] or d["ratio"] >= (base_ratio - threshold):
                valid_directions.append(d["vec"])
    else:
        # 【防呆】如果沒有完全無阻擋的缺口，找出最暢通的方向作為基準
        best_ratio = max([d["ratio"] for d in directions_info] + [0])
        
        if best_ratio > 0.4: 
            for d in directions_info:
                if d["ratio"] >= (best_ratio - threshold):
                    valid_directions.append(d["vec"])
        else:
            # 終極防呆：退回 Bounding Box 最長軸
            spans = np.array([xmax-xmin, ymax-ymin, zmax-zmin])
            ax_max = int(np.argmax(spans))
            g = np.zeros((3,), dtype=np.float32)
            g[ax_max] = 1.0
            valid_directions.append(g)

    return valid_directions
    
def build_pattern_bank_from_seq_items(seq_items):
    """
    將每一個 frame 都視作 pattern bank item。
    如果一個 frame 有多個無阻礙的開口，將會被分裂成多個帶有不同 grow_direction 的 item。
    """
    bank = []

    for seq in seq_items:
        L = seq["length"]
        for i in range(L):
            uid = PATH2UID[os.path.join(seq["cache_dir"], f"{int(seq['chain_ids'][i]):06d}.npz")]
            grid, _ = load_voxel_grid_by_uid(uid)
            if grid is None:               # ← skip empty frames in chain
                continue
            grid = (grid > 0).astype(np.uint8)
            
            if grid.sum() == 0:
                continue

            # 1. 取得所有無阻礙的有效生長方向
            grow_dirs = find_open_grow_directions(grid)

            # 2. 為每一個有效的生長方向，分別切出 seed 並存入 bank
            for g in grow_dirs:
                # 以 reverse-direction band 作 seed
                seed = _select_start_frontier_seed_band_np(
                    pattern_start_u8=grid,
                    grow_direction_np=g,
                    q=RAG_START_BAND_Q,
                )

                mask_local, offset = canonicalize_mask_to_center(grid)
                seed_local = _shift_mask_by_int_offset(seed, offset)

                desc = build_pattern_descriptor_np(mask_local, seed_local, g)

                bank.append({
                    "uid": int(uid),
                    "seq_name": seq["source_name"],
                    "mask_local": mask_local.astype(np.uint8),
                    "seed_local": seed_local.astype(np.uint8),
                    "grow_direction": g.astype(np.float32),
                    "descriptor": desc.astype(np.float32),
                })

    print(f"[pattern bank] items = {len(bank)}")
    return bank
    
# pattern bank
PATTERN_BANK_TOPK = 3
PATTERN_BANK = build_pattern_bank_from_seq_items(train_seq_items)
PATTERN_BANK_INDEX = {
    item["seed_local"].tobytes(): idx
    for idx, item in enumerate(PATTERN_BANK)
}

@torch.no_grad()
def update_pattern_bank_embeddings(pattern_bank, encoder, device):
    """
    將整個 pattern_bank 裡面的圖案過一次 encoder (大幅減少 to(device) 的次數)
    """
    encoder.eval()
    embeddings = []
    
    batch_size = 32
    for i in range(0, len(pattern_bank), batch_size):
        batch_items = pattern_bank[i : i+batch_size]
        
        # 判斷大小是否一致，如果一致我們可以直接在 CPU 端組裝成一個大陣列
        # 如果大小不一致（你的前面報錯提過），我們只能用 List 送進 Encoder
        vols = []
        for item in batch_items:
            # 只轉 Tensor，先不搬到 GPU (把 to(device) 留到後面)
            vol_t = torch.from_numpy(item["seed_local"]).float()
            vols.append(vol_t)
            
        # 統一在這裡把整個 batch 的 Tensor 搬到 GPU
        vols_gpu = [v.to(device, non_blocking=True) for v in vols]
        
        emb = encoder(vols_gpu)
        embeddings.append(emb)
        
    bank_embeddings = torch.cat(embeddings, dim=0)
    return bank_embeddings

def build_query_descriptor_np(current_frontier_u8, grow_direction_np):
    frontier = (np.asarray(current_frontier_u8) > 0).astype(np.uint8)
    if frontier.sum() == 0:
        return np.zeros((16,), dtype=np.float32)

    frontier_local, _ = canonicalize_mask_to_center(frontier)
    seed_local = frontier_local.copy()  # query 無獨立 seed，就先用 frontier 本身
    return build_pattern_descriptor_np(frontier_local, seed_local, grow_direction_np)


def retrieve_topk_neural(
    current_frontier_u8, 
    grow_direction_np, 
    pattern_bank, 
    bank_embeddings_t, # [N_items, d_model] 這是第二區塊算出來的矩陣
    encoder, 
    device,
    top_k=8
):
    # 1. 將 Query (目前的建築缺口) 轉成 Embedding
    frontier_local, _ = canonicalize_mask_to_center(current_frontier_u8)
    query_vol = torch.from_numpy(frontier_local).float().unsqueeze(0).unsqueeze(0).to(device)
    
    encoder.eval()
    with torch.no_grad():
        query_emb = encoder(query_vol) # [1, d_model]
        
        # 2. 計算餘弦相似度 (Cosine Similarity)
        # 矩陣相乘，範圍是 [-1, 1]，越大代表越相似
        cosine_sims = torch.matmul(query_emb, bank_embeddings_t.T).squeeze(0) # [N_items]
        
    # 轉回 numpy 方便與原本邏輯結合
    cosine_sims_np = cosine_sims.cpu().numpy()
    
    # 3. 計算方向懲罰 (保留你原本的邏輯)
    q_xy_mag = np.linalg.norm(grow_direction_np[:2])
    q_z_mag = np.abs(grow_direction_np[2])

    scored = []
    for idx, item in enumerate(pattern_bank):
        i_dir = item["grow_direction"]
        i_xy_mag = np.linalg.norm(i_dir[:2])
        i_z_mag = np.abs(i_dir[2])
        
        dir_pen = np.abs(i_xy_mag - q_xy_mag) + np.abs(i_z_mag - q_z_mag)
        
        # 綜合評分：
        # Cosine Similarity 越大越好，所以我們用 1.0 - cosine_sims 把它變成「誤差」(越小越好)
        # 這樣就能跟你原本的邏輯 (分數越小越好) 完美相容！
        semantic_distance = 1.0 - cosine_sims_np[idx]
        
        # 分數 = 幾何語意距離 + 0.5 * 方向懲罰
        score = semantic_distance + 0.5 * dir_pen
        scored.append((score, item))

    # 取分數最低 (誤差最小) 的 Top-K
    scored.sort(key=lambda x: x[0])

    #Radom Test, Return random item in pattern bank, need to different everytime enquiry
    # import random
    # random_item = random.SystemRandom().choice(pattern_bank) 
    # return [random_item]

    #Normal Return
    return [it for _, it in scored[:max(1, int(top_k))]]

def get_real_grow_direction_for_chosen_item(chosen_item, seq_items):
    """
    透過 chosen_item 的 uid，回到原始 sequence 中尋找當前幀(t)與前後幀(t+1, t-1)，
    並計算出真實的生長方向。
    """
    uid = chosen_item["uid"]
    seq_name = chosen_item["seq_name"]

    # 1. 找出這個 item 屬於哪一個 sequence
    target_seq = None
    for seq in seq_items:
        if seq["source_name"] == seq_name:
            target_seq = seq
            break

    if target_seq is None:
        return None, False, None, False

    # 2. 透過 UID 找回它在 sequence chain_ids 中的位置 (t)
    target_path = UID2PATH[uid]
    t_idx = -1
    for i, fid in enumerate(target_seq["chain_ids"]):
        p = os.path.join(target_seq["cache_dir"], f"{int(fid):06d}.npz")
        if p == target_path:
            t_idx = i
            break

    if t_idx == -1:
        return None, False, None, False

    # 3. 準備載入當前網格
    current_fid = int(target_seq["chain_ids"][t_idx])
    current_uid = PATH2UID[os.path.join(target_seq["cache_dir"], f"{current_fid:06d}.npz")]
    grid_curr, _ = load_voxel_grid_by_uid(current_uid)

    # 初始化返回值
    forward_direction, is_valid_forward = None, False
    from_direction, is_valid_from = None, False

    # 4. 如果有「下一幀」(t+1)，計算 forward_direction
    if t_idx < target_seq["length"] - 1:
        next_fid = int(target_seq["chain_ids"][t_idx + 1])
        next_uid = PATH2UID[os.path.join(target_seq["cache_dir"], f"{next_fid:06d}.npz")]
        grid_next, _ = load_voxel_grid_by_uid(next_uid)
        forward_direction, is_valid_forward = compute_relative_growth_direction(grid_curr, grid_next)

    # 5. 如果有「上一幀」(t-1)，計算 from_direction
    if t_idx > 0:
        previous_fid = int(target_seq["chain_ids"][t_idx - 1])
        previous_uid = PATH2UID[os.path.join(target_seq["cache_dir"], f"{previous_fid:06d}.npz")]
        grid_previous, _ = load_voxel_grid_by_uid(previous_uid)
        from_direction, is_valid_from = compute_relative_growth_direction(grid_curr, grid_previous)

    return forward_direction, is_valid_forward, from_direction, is_valid_from


def align_mask_direction_3d(mask, src_dir, tgt_dir):
    """
    Rotate the mask in the XY plane (around Z axis) so that
    the XY projection of src_dir aligns with the XY projection of tgt_dir.
    Z dimension is left untouched.
    """
    src = np.asarray(src_dir, dtype=np.float64)
    tgt = np.asarray(tgt_dir, dtype=np.float64)

    # Project onto XY plane
    src_xy = src[:2]
    tgt_xy = tgt[:2]

    src_len = np.linalg.norm(src_xy)
    tgt_len = np.linalg.norm(tgt_xy)

    # Either direction has no XY component → nothing to rotate
    if src_len < 1e-6 or tgt_len < 1e-6:
        return mask.copy()

    src_xy = src_xy / src_len
    tgt_xy = tgt_xy / tgt_len

    # Signed angle from src_xy to tgt_xy (atan2 of the 2D cross product)
    sin_a = float(src_xy[0] * tgt_xy[1] - src_xy[1] * tgt_xy[0])  # 2D cross
    cos_a = float(np.dot(src_xy, tgt_xy))
    angle_deg = float(np.degrees(np.arctan2(sin_a, cos_a)))

    # Already aligned (within 2°)
    if abs(angle_deg) < 2.0:
        return mask.copy()

    # Rotate in the XY plane (axes 0,1), keep Z (axis 2) intact
    # scipy rotates in the plane of the two given axes
    mask_f = np.asarray(mask, dtype=np.float32)
    rotated = ndi.rotate(
        mask_f,
        angle=angle_deg,
        axes=(0, 1),        # rotate in XY plane around Z
        reshape=False,      # keep same grid shape
        order=0,            # nearest-neighbour → stays binary
        mode='constant',
        cval=0.0,
    )
    return (rotated > 0.5).astype(np.uint8)


def instantiate_pattern_from_bank_item(bank_item, ref_shape, grow_direction):
    """
    將 bank item 的 local canonical pattern 旋轉對齊後，
    放到 ref_shape 中央，之後再交俾 shift_n 推到 initial_known 前方。
    """
    mask_local = bank_item["mask_local"].astype(np.uint8)
    src_dir = bank_item["grow_direction"]
    
    # --- [新增] 將 mask_local 旋轉至與目標 grow_direction 一致 ---
    mask_local = align_mask_direction_3d(mask_local, src_dir, grow_direction)
    
    shape = tuple(ref_shape)
    out = np.zeros(shape, dtype=np.uint8)

    # !!! 重要細節：旋轉後 mask_local 的形狀 (sx, sy, sz) 可能會改變 (例如長寬互換)
    # 所以必須在旋轉「之後」重新讀取 shape
    sx, sy, sz = mask_local.shape
    X, Y, Z = shape

    x0 = max(0, (X - sx) // 2)
    y0 = max(0, (Y - sy) // 2)
    z0 = max(0, (Z - sz) // 2)

    x1 = min(X, x0 + sx)
    y1 = min(Y, y0 + sy)
    z1 = min(Z, z0 + sz)

    # 貼到中央
    out[x0:x1, y0:y1, z0:z1] = mask_local[:x1-x0, :y1-y0, :z1-z0]
    return out.astype(np.uint8)

def _build_sample_for_one_pattern(
    chosen_item,
    initial_known,
    current_frontier,
    grow_direction,
    grow_direction_valid,
    step_known,
    step_delta,
    final_union,
    final_delta,
    align_meta,
):
    """
    Given a pre-retrieved pattern bank item and the already-computed
    scene context, build one training sample dict.
    All the expensive grid loading / alignment is done once upstream.
    """
    pattern_start_base = instantiate_pattern_from_bank_item(
        chosen_item,
        ref_shape=initial_known.shape,
        grow_direction=grow_direction,
    ).astype(np.uint8)

    shift_n = compute_pattern_shift_along_direction(
        pattern_start_base,
        initial_known,
        grow_direction,
        clearance=RAG_PATTERN_CLEARANCE,
        overlap_thresh=RAG_PATTERN_OVERLAP_THRESH,
    )

    # temporary forward shift to locate pattern tail
    temp_shifted = shift_volume_combined(pattern_start_base, shift_n, grow_direction)
    temp_delta   = ((temp_shifted > 0) & (initial_known == 0)).astype(np.uint8)
    temp_seed    = _select_start_frontier_seed_band_np(
        pattern_start_u8=temp_delta,
        grow_direction_np=grow_direction,
        q=RAG_START_BAND_Q,
    )

    # lateral alignment
    curr_coords   = np.argwhere(current_frontier > 0)
    target_center = curr_coords.mean(axis=0) if curr_coords.shape[0] > 0 \
                    else np.array(initial_known.shape) / 2.0

    seed_coords   = np.argwhere(temp_seed > 0)
    source_center = seed_coords.mean(axis=0) if seed_coords.shape[0] > 0 \
                    else target_center

    lateral_offset = target_center - source_center
    g_unit = np.asarray(grow_direction, dtype=np.float64)
    g_unit = g_unit / (np.linalg.norm(g_unit) + 1e-8)
    lateral_offset = lateral_offset - np.dot(lateral_offset, g_unit) * g_unit

    # =========================================================
    # 🌟 新增：底面 (Z軸) 絕對對齊邏輯
    # 假設 Numpy Grid 中 Z 軸為索引 2 (X=0, Y=1, Z=2)
    # =========================================================
    # 1. 取得現有建築的最低 Z 座標
    known_pts = np.argwhere(initial_known > 0)
    min_z_known = known_pts[:, 2].min() if known_pts.shape[0] > 0 else 0

    # 2. 取得圖案目前的最低 Z 座標 (使用已沿著方向前進過的 temp_shifted)
    pattern_pts = np.argwhere(temp_shifted > 0)
    min_z_pattern = pattern_pts[:, 2].min() if pattern_pts.shape[0] > 0 else 0

    # 3. 如果主要是「水平生長」，則強制覆寫 Z 軸偏移量，讓底面對齊
    if abs(g_unit[2]) < 0.5: 
        lateral_offset[2] = min_z_known - min_z_pattern
    # =========================================================

    pattern_start_shifted = shift_volume_combined(
        pattern_start_base,
        shift_voxels=shift_n,
        grow_direction_np=grow_direction,
        lateral_offset_np=lateral_offset,
    )

    pattern_start_delta_shifted = (
        (pattern_start_shifted > 0) & (initial_known == 0)
    ).astype(np.uint8)

    start_frontier_seed = _select_start_frontier_seed_band_np(
        pattern_start_u8=pattern_start_delta_shifted,
        grow_direction_np=grow_direction,
        q=RAG_START_BAND_Q,
    )

    (
        reverse_bridge_field_target,
        reverse_bridge_target_full,
        reverse_bridge_seed,
        bridge_src_xyz,
        bridge_dst_xyz,
    ) = build_reverse_bridge_field_targets_np(
        initial_known_u8=initial_known,
        input_frontier_u8=current_frontier,
        pattern_start_u8=pattern_start_delta_shifted,
        grow_direction_np=grow_direction,
        step_dilate=BRIDGE_STEP_DILATE,
    )

    start_frontier_seed = matching_start_frontier(
        start_frontier_seed,
        reverse_bridge_seed,
        grow_direction,
    )

    return {
        "initial_known":    initial_known.astype(np.uint8),
        "next_known":       step_known.astype(np.uint8),
        "next_delta":       step_delta.astype(np.uint8),

        "preview_known":    pattern_start_shifted.astype(np.uint8),
        "preview_delta":    pattern_start_delta_shifted.astype(np.uint8),

        "final_union":      final_union.astype(np.uint8),
        "final_delta":      final_delta.astype(np.uint8),

        "grow_direction":       grow_direction.astype(np.float32),
        "grow_direction_valid": np.float32(grow_direction_valid),
        "n_shift":              np.float32(shift_n),

        "current_frontier":     current_frontier.astype(np.uint8),

        "start_frontier_seed":      start_frontier_seed.astype(np.uint8),
        "pattern_start_frontier":   start_frontier_seed.astype(np.uint8),
        "pattern_start_mask":       pattern_start_delta_shifted.astype(np.uint8),

        "selected_source_frontier": reverse_bridge_seed.astype(np.uint8),
        "bridge_target":            reverse_bridge_field_target.astype(np.uint8),
        "bridge_target_full":       reverse_bridge_target_full.astype(np.uint8),

        "start_completion_target":  pattern_start_delta_shifted.astype(np.uint8),

        "pattern_bank_uid": np.int32(chosen_item["uid"]),
        "bridge_src_xyz": (
            np.asarray(bridge_src_xyz, dtype=np.int32)
            if bridge_src_xyz is not None else np.asarray([-1, -1, -1], dtype=np.int32)
        ),
        "bridge_dst_xyz": (
            np.asarray(bridge_dst_xyz, dtype=np.int32)
            if bridge_dst_xyz is not None else np.asarray([-1, -1, -1], dtype=np.int32)
        ),
        "align_meta": {
            "grid_lo": align_meta["grid_lo"].astype(np.int32),
            "grid_hi": align_meta["grid_hi"].astype(np.int32),
        },
    }

def retrieve_topk_pattern_bank(pattern_bank, current_frontier_u8, grow_direction_np, top_k=8):# for training matching,reduce training calculation
    q = build_query_descriptor_np(current_frontier_u8, grow_direction_np)

    # --- [新增] 計算 Query 的 XY 水平維度強度 與 Z 垂直維度強度 ---
    # grow_direction_np 是一個 [x, y, z] 的單位向量
    q_xy_mag = np.linalg.norm(grow_direction_np[:2]) # sqrt(x^2 + y^2)
    q_z_mag = np.abs(grow_direction_np[2])           # abs(z)

    scored = []
    for item in pattern_bank:
        d = item["descriptor"]  # 恢復原本的 descriptor，不要加 abs，以免破壞幾何特徵
        
        # --- [新增] 計算 Item 的 XY 水平維度強度 與 Z 垂直維度強度 ---
        i_dir = item["grow_direction"]
        i_xy_mag = np.linalg.norm(i_dir[:2])
        i_z_mag = np.abs(i_dir[2])
        
        # --- [修改] 維度一致性懲罰 (Direction Penalty) ---
        # 比較 XY 強度差異與 Z 強度差異。
        # 如果兩者都在 XY 平面 (不管東南西北)，dir_pen 會趨近於 0
        # 如果兩者都在 Z 軸 (不管上下)，dir_pen 也會趨近於 0
        dir_pen = np.abs(i_xy_mag - q_xy_mag) + np.abs(i_z_mag - q_z_mag)
        
        # L1 幾何特徵誤差
        l1 = np.abs(d - q).mean()
        
        # 總分 (分數越低越好)。0.5 是權重，你可以視情況調大調小
        score = l1 + 0.5 * dir_pen
        scored.append((score, item))

    scored.sort(key=lambda x: x[0])
    return [it for _, it in scored[:max(1, int(top_k))]]


def build_directional_growth_sample(
    seq,
    start,
    ctx_len=CTX_LEN,
    next_offset=DIRECTION_NEXT_OFFSET,
    preview_horizon=PREVIEW_HORIZON,
    patterns_per_scene=PATTERN_BANK_TOPK,   # how many patterns to pair with each scene
):
    """
    Returns a LIST of sample dicts — one per retrieved pattern.
    The scene context (grid loading, alignment, frontier, grow_direction)
    is computed ONCE and reused across all patterns.
    """
    L = seq["length"]

    worker_ctx_ids = []
    for k in range(ctx_len):
        t = min(start + k, L - 1)
        fid = int(seq["chain_ids"][t])
        p = os.path.join(seq["cache_dir"], f"{fid:06d}.npz")
        worker_ctx_ids.append(PATH2UID[p])

    next_t = min(start + int(ctx_len) - 1 + int(next_offset), L - 1)
    future_ids = []
    for t in range(next_t, L):
        fid = int(seq["chain_ids"][t])
        p = os.path.join(seq["cache_dir"], f"{fid:06d}.npz")
        future_ids.append(PATH2UID[p])

    worker_ctx  = [load_voxel_grid_by_uid(uid) for uid in worker_ctx_ids]
    future_grids = [load_voxel_grid_by_uid(uid) for uid in future_ids]

    all_pack = worker_ctx + future_grids
    aligned_all, align_meta = align_many_voxel_grids(all_pack)

    n_w = len(worker_ctx)
    aligned_worker = aligned_all[:n_w]
    aligned_future = aligned_all[n_w:]

    if len(aligned_future) == 0:
        raise ValueError("aligned_future is empty in build_directional_growth_sample")

    # ── scene context (shared across all patterns) ──
    initial_known = np.zeros_like(aligned_future[0], dtype=np.uint8)
    for g in aligned_worker:
        initial_known = np.maximum(initial_known, (g > 0).astype(np.uint8))

    step_known  = (aligned_future[0] > 0).astype(np.uint8)
    step_delta  = ((step_known > 0) & (initial_known == 0)).astype(np.uint8)

    final_union = np.zeros_like(step_known, dtype=np.uint8)
    for g in aligned_future:
        final_union = np.maximum(final_union, (g > 0).astype(np.uint8))
    final_delta = ((final_union > 0) & (initial_known == 0)).astype(np.uint8)

    current_frontier = frontier_from_known_3d(initial_known)

    grow_direction, grow_direction_valid = compute_relative_growth_direction(
        initial_known, step_delta
    )

    # ── retrieve top-k patterns — use ALL of them, not just one ──
    topk_items = retrieve_topk_pattern_bank(
        PATTERN_BANK,
        current_frontier_u8=current_frontier,
        grow_direction_np=grow_direction,
        top_k=patterns_per_scene,
    )

    samples = []
    for chosen_item in topk_items:
        try:
            s = _build_sample_for_one_pattern(
                chosen_item=chosen_item,
                initial_known=initial_known,
                current_frontier=current_frontier,
                grow_direction=grow_direction,
                grow_direction_valid=grow_direction_valid,
                step_known=step_known,
                step_delta=step_delta,
                final_union=final_union,
                final_delta=final_delta,
                align_meta=align_meta,
            )
            samples.append(s)
        except Exception as e:
            # skip bad patterns silently rather than crashing the whole epoch
            print(f"[WARN] pattern uid={chosen_item['uid']} skipped: {e}")
            continue

    return samples  # list of dicts, length = len(topk_items)

k:\MasterEssay\Attempt_source_03\.venv\lib\site-packages\numba\core\decorators.py:248: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


[pattern bank] items = 1104


In [12]:
# trim helper
def crop_many_to_nonzero_union_3d(arrs, margin=2):
    if len(arrs) == 0:
        return [], np.zeros((3,), dtype=np.int32)

    arrs = [np.asarray(arr) for arr in arrs]
    ref_shape = arrs[0].shape
    union = np.zeros(ref_shape, dtype=np.uint8)

    for arr in arrs:
        if arr.shape != ref_shape:
            raise ValueError(
                f"shape mismatch in crop_many_to_nonzero_union_3d: {arr.shape} vs {ref_shape}"
            )
        union |= (arr > 0).astype(np.uint8)

    xs, ys, zs = np.where(union > 0)
    if len(xs) == 0:
        return [arr.copy() for arr in arrs], np.zeros((3,), dtype=np.int32)

    x0 = max(0, int(xs.min()) - margin)
    x1 = min(ref_shape[0], int(xs.max()) + margin + 1)
    y0 = max(0, int(ys.min()) - margin)
    y1 = min(ref_shape[1], int(ys.max()) + margin + 1)
    z0 = max(0, int(zs.min()) - margin)
    z1 = min(ref_shape[2], int(zs.max()) + margin + 1)

    cropped = [arr[x0:x1, y0:y1, z0:z1] for arr in arrs]
    crop_lo = np.asarray([x0, y0, z0], dtype=np.int32)
    return cropped, crop_lo


def crop_to_nonzero_union_3d(a, b, margin=2):
    cropped, _ = crop_many_to_nonzero_union_3d([a, b], margin=margin)
    return cropped[0], cropped[1]


In [13]:
class FrontierDirStateDataset3D(Dataset):
    def __init__(self, seq_items, ctx_len=CTX_LEN, seed=42,
                 use_cache=True, patterns_per_scene=PATTERN_BANK_TOPK):
        self.seq_items = seq_items
        self.ctx_len = ctx_len
        self.seed = seed
        self.use_cache = use_cache
        self.patterns_per_scene = patterns_per_scene
        self._sample_cache = {}
        self.samples = []   # each entry is (seq_idx, start, pattern_rank)

        for si, seq in enumerate(seq_items):
            L = seq["length"]
            for start in range(0, L - ctx_len):
                for rank in range(patterns_per_scene):
                    self.samples.append((si, start, rank))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.use_cache and idx in self._sample_cache:
            return self._sample_cache[idx]

        seq_idx, start, rank = self.samples[idx]
        seq = self.seq_items[seq_idx]

        # build all patterns for this (seq, start) — cache by (seq_idx, start)
        scene_key = (seq_idx, start)
        if self.use_cache and scene_key in self._sample_cache:
            scene_samples = self._sample_cache[scene_key]
        else:
            scene_samples = build_directional_growth_sample(
                seq, start=start, ctx_len=self.ctx_len,
                patterns_per_scene=self.patterns_per_scene,
            )
            if self.use_cache:
                self._sample_cache[scene_key] = scene_samples

        # pick the rank-th pattern; clamp in case fewer patterns returned
        sample = scene_samples[min(rank, len(scene_samples) - 1)]

        # crop
        arrs = [
            sample["initial_known"], sample["next_known"], sample["next_delta"],
            sample["preview_known"], sample["preview_delta"],
            sample["final_union"],   sample["final_delta"],
            sample["current_frontier"], sample["selected_source_frontier"],
            sample["start_frontier_seed"], sample["pattern_start_frontier"],
            sample["pattern_start_mask"], sample["bridge_target"],
            sample["bridge_target_full"], sample["start_completion_target"],
        ]
        cropped, crop_lo = crop_many_to_nonzero_union_3d(arrs, margin=CROP_MARGIN)

        out = {
            "initial_known":            cropped[0],
            "next_known":               cropped[1],
            "next_delta":               cropped[2],
            "preview_known":            cropped[3],
            "preview_delta":            cropped[4],
            "final_known":              cropped[5],
            "final_delta":              cropped[6],
            "current_frontier":         cropped[7],
            "selected_source_frontier": cropped[8],
            "start_frontier_seed":      cropped[9],
            "pattern_start_frontier":   cropped[10],
            "pattern_start_mask":       cropped[11],
            "bridge_target":            cropped[12],
            "bridge_target_full":       cropped[13],
            "start_completion_target":  cropped[14],
            "grow_direction":           sample["grow_direction"],
            "grow_direction_valid":     sample["grow_direction_valid"],
            "n_shift":                  np.float32(sample["n_shift"]),
            "crop_lo":                  crop_lo.astype(np.int32),
        }

        if self.use_cache:
            self._sample_cache[idx] = out
        return out

In [14]:
# collate batch helper
def frontier_collate_3d(batch):
    B = len(batch)
    if B == 0:
        raise ValueError("Empty batch")

    Xmax = max(x["initial_known"].shape[0] for x in batch)
    Ymax = max(x["initial_known"].shape[1] for x in batch)
    Zmax = max(x["initial_known"].shape[2] for x in batch)

    def alloc_vol():
        return torch.zeros((B, 1, Xmax, Ymax, Zmax), dtype=torch.float32)

    initial_known = alloc_vol()
    next_known = alloc_vol()
    next_delta = alloc_vol()
    preview_known = alloc_vol()
    preview_delta = alloc_vol()
    final_known = alloc_vol()
    final_delta = alloc_vol()
    current_frontier = alloc_vol()
    pattern_start_frontier = alloc_vol()
    pattern_start_mask = alloc_vol()
    bridge_target = alloc_vol()

    selected_source_frontier = alloc_vol()

    start_frontier_seed = alloc_vol()
    start_completion_target = alloc_vol()
    
    bridge_target_full = alloc_vol()

    grow_direction = torch.zeros((B, DIRECTION_DIM), dtype=torch.float32)
    grow_direction_valid = torch.zeros((B,), dtype=torch.float32)

    n_shift = torch.zeros((B,), dtype=torch.float32)

    def _pad_into(dst, arr, bidx):
        sx, sy, sz = arr.shape
        dst[bidx, 0, :sx, :sy, :sz] = torch.from_numpy(arr).float()

    for b, x in enumerate(batch):
        _pad_into(initial_known, x["initial_known"], b)
        _pad_into(next_known, x["next_known"], b)
        _pad_into(next_delta, x["next_delta"], b)
        _pad_into(preview_known, x["preview_known"], b)
        _pad_into(preview_delta, x["preview_delta"], b)
        _pad_into(final_known, x["final_known"], b)
        _pad_into(final_delta, x["final_delta"], b)
        _pad_into(current_frontier, x["current_frontier"], b)
        _pad_into(pattern_start_frontier, x["pattern_start_frontier"], b)
        _pad_into(pattern_start_mask, x["pattern_start_mask"], b)
        _pad_into(bridge_target, x["bridge_target"], b)
        grow_direction[b] = torch.from_numpy(x["grow_direction"]).float()
        grow_direction_valid[b] = float(x["grow_direction_valid"])
        n_shift[b] = float(x["n_shift"])
        _pad_into(selected_source_frontier, x["selected_source_frontier"], b)
        _pad_into(start_frontier_seed, x["start_frontier_seed"], b)
        _pad_into(start_completion_target, x["start_completion_target"], b)
        _pad_into(bridge_target_full, x["bridge_target_full"], b)

    return {
        "initial_known": initial_known,
        "next_known": next_known,
        "next_delta": next_delta,
        "preview_known": preview_known,
        "preview_delta": preview_delta,
        "final_known": final_known,
        "final_delta": final_delta,
        "grow_direction": grow_direction,
        "grow_direction_valid": grow_direction_valid,
        "current_frontier": current_frontier,
        "pattern_start_frontier": pattern_start_frontier,
        "pattern_start_mask": pattern_start_mask,
        "bridge_target": bridge_target,
        "n_shift": n_shift,
        "selected_source_frontier": selected_source_frontier,
        "start_frontier_seed": start_frontier_seed,
        "start_completion_target": start_completion_target,
        "bridge_target_full":bridge_target_full
    }

In [15]:
# load dataset

train_ds = FrontierDirStateDataset3D(train_seq_items, ctx_len=CTX_LEN, seed=42, use_cache=True)
val_ds = FrontierDirStateDataset3D(val_seq_items, ctx_len=CTX_LEN, seed=42, use_cache=True)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    collate_fn=frontier_collate_3d,
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    collate_fn=frontier_collate_3d,
)

print("train size:", len(train_ds))
print("val size:", len(val_ds))
print("NUM_WORKERS:", NUM_WORKERS)

train size: 1284
val size: 36
NUM_WORKERS: 0


Module Helper

In [16]:
def sample_global_features_at_nodes_3d(feat_map, node_pos, state_shape):
    """
    feat_map: [B, D, Xg, Yg, Zg]
    node_pos: [B, N, 3] in full-res voxel coords
    state_shape: (X, Y, Z) of full-res state grid
    return: [B, N, D]
    """
    B, D, Xg, Yg, Zg = feat_map.shape
    _, N, _ = node_pos.shape
    X, Y, Z = state_shape

    xx = torch.floor(node_pos[..., 0].float() * Xg / max(X, 1)).long().clamp(0, Xg - 1)
    yy = torch.floor(node_pos[..., 1].float() * Yg / max(Y, 1)).long().clamp(0, Yg - 1)
    zz = torch.floor(node_pos[..., 2].float() * Zg / max(Z, 1)).long().clamp(0, Zg - 1)

    feat_map_xyz = feat_map.permute(0, 2, 3, 4, 1)  # [B, Xg, Yg, Zg, D]
    b_idx = torch.arange(B, device=feat_map.device)[:, None].expand(B, N)

    out = feat_map_xyz[b_idx, xx, yy, zz]  # [B, N, D]
    return out

def build_3d_sincos_posenc(node_pos, d_model, state_shape=None):
    """
    node_pos: [B, N, 3]  (voxel coords)
    state_shape: optional (X, Y, Z), used to normalize coords to [-1, 1]
    return: [B, N, d_model]
    """
    device = node_pos.device
    B, N, _ = node_pos.shape

    if state_shape is not None:
        X, Y, Z = state_shape
        px = node_pos[..., 0].float() / max(X - 1, 1)
        py = node_pos[..., 1].float() / max(Y - 1, 1)
        pz = node_pos[..., 2].float() / max(Z - 1, 1)

        px = px * 2.0 - 1.0
        py = py * 2.0 - 1.0
        pz = pz * 2.0 - 1.0
    else:
        px = node_pos[..., 0].float()
        py = node_pos[..., 1].float()
        pz = node_pos[..., 2].float()

    part = d_model // 6
    rem = d_model - part * 6
    if part <= 0:
        raise ValueError(f"d_model too small: {d_model}")

    omega = torch.arange(part, device=device, dtype=torch.float32)
    omega = 1.0 / (10000 ** (omega / max(1, part - 1)))

    px = px.unsqueeze(-1) * omega
    py = py.unsqueeze(-1) * omega
    pz = pz.unsqueeze(-1) * omega

    pos = torch.cat([
        torch.sin(px), torch.cos(px),
        torch.sin(py), torch.cos(py),
        torch.sin(pz), torch.cos(pz),
    ], dim=-1)

    if rem > 0:
        pos = torch.cat([
            pos,
            torch.zeros(B, N, rem, device=device, dtype=pos.dtype)
        ], dim=-1)

    return pos

def _safe_normalize_direction_torch(direction, direction_valid=None, eps=1e-6):
    direction = direction.float()
    norm = direction.norm(dim=-1, keepdim=True)
    direction = direction / norm.clamp_min(eps)

    default_dir = torch.zeros_like(direction)
    default_dir[..., 0] = 1.0

    if direction_valid is None:
        use_dir = (norm.squeeze(-1) > eps).float().unsqueeze(-1)
    else:
        use_dir = (direction_valid > 0.5).float().unsqueeze(-1)

    return direction * use_dir + default_dir * (1.0 - use_dir)

In [17]:
def build_local_geom_descriptor(node_patches):
    """
    node_patches: [B, N, C, P, P, P]
    channels:
      0 known
      1 frontier
      2 exterior_empty
      3 interior_empty
      4 locked_solid
      5 locked_exterior
      6 locked_interior
      7 resolved_nodes

    return: [B, N, 16]
    """
    B, N, C, P, _, _ = node_patches.shape
    x = node_patches.float()

    known = x[:, :, 0]
    frontier = x[:, :, 1]
    exterior = x[:, :, 2]
    interior = x[:, :, 3]
    locked_solid = x[:, :, 4]
    locked_ext = x[:, :, 5]
    locked_int = x[:, :, 6]
    resolved = x[:, :, 7]

    known_ratio = known.mean(dim=(-1, -2, -3)).unsqueeze(-1)
    frontier_ratio = frontier.mean(dim=(-1, -2, -3)).unsqueeze(-1)
    exterior_ratio = exterior.mean(dim=(-1, -2, -3)).unsqueeze(-1)
    interior_ratio = interior.mean(dim=(-1, -2, -3)).unsqueeze(-1)
    locked_solid_ratio = locked_solid.mean(dim=(-1, -2, -3)).unsqueeze(-1)
    locked_ext_ratio = locked_ext.mean(dim=(-1, -2, -3)).unsqueeze(-1)
    locked_int_ratio = locked_int.mean(dim=(-1, -2, -3)).unsqueeze(-1)
    resolved_ratio = resolved.mean(dim=(-1, -2, -3)).unsqueeze(-1)

    c = P // 2
    center_known = known[:, :, c, c, c].unsqueeze(-1)
    center_frontier = frontier[:, :, c, c, c].unsqueeze(-1)

    grid = torch.linspace(-1.0, 1.0, P, device=x.device, dtype=x.dtype)
    gx, gy, gz = torch.meshgrid(grid, grid, grid, indexing="ij")
    gx = gx.view(1, 1, P, P, P)
    gy = gy.view(1, 1, P, P, P)
    gz = gz.view(1, 1, P, P, P)

    mass = known.sum(dim=(-1, -2, -3)).clamp_min(1e-6)
    cx = ((known * gx).sum(dim=(-1, -2, -3)) / mass).unsqueeze(-1)
    cy = ((known * gy).sum(dim=(-1, -2, -3)) / mass).unsqueeze(-1)
    cz = ((known * gz).sum(dim=(-1, -2, -3)) / mass).unsqueeze(-1)

    fmass = frontier.sum(dim=(-1, -2, -3)).clamp_min(1e-6)
    fcx = ((frontier * gx).sum(dim=(-1, -2, -3)) / fmass).unsqueeze(-1)
    fcy = ((frontier * gy).sum(dim=(-1, -2, -3)) / fmass).unsqueeze(-1)
    fcz = ((frontier * gz).sum(dim=(-1, -2, -3)) / fmass).unsqueeze(-1)

    desc = torch.cat([
        known_ratio,
        frontier_ratio,
        exterior_ratio,
        interior_ratio,
        locked_solid_ratio,
        locked_ext_ratio,
        locked_int_ratio,
        resolved_ratio,
        center_known,
        center_frontier,
        cx, cy, cz,
        fcx, fcy, fcz,
    ], dim=-1)

    return desc

Module

In [18]:
# small part of model
class Conv3dBlock(nn.Module):
    def __init__(self, c_in, c_out, k=3, s=1, p=1, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(c_in, c_out, kernel_size=k, stride=s, padding=p, bias=False),
            nn.GroupNorm(1, c_out),
            nn.GELU(),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)


class SpatialGlobalStateEncoder3D(nn.Module):
    def __init__(self, in_ch=8, d_model=D_MODEL):
        super().__init__()
        self.net = nn.Sequential(
            Conv3dBlock(in_ch, 16, k=3, s=2, p=1, dropout=DROPOUT),   # /2
            Conv3dBlock(16, 32, k=3, s=2, p=1, dropout=DROPOUT),      # /4
            Conv3dBlock(32, d_model, k=3, s=1, p=1, dropout=DROPOUT),
        )

    def forward(self, x):
        return self.net(x)
    
class LocalPatchEncoder3D(nn.Module):
    def __init__(self, in_ch=8, d_model=D_MODEL):
        super().__init__()
        self.net = nn.Sequential(
            Conv3dBlock(in_ch, 16, dropout=DROPOUT),
            Conv3dBlock(16, 32, dropout=DROPOUT),
            Conv3dBlock(32, d_model, dropout=DROPOUT),
            nn.AdaptiveAvgPool3d((1, 1, 1)),
        )

    def forward(self, x):
        return self.net(x).flatten(1)


class SharedTransformerBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + a
        x = x + self.ff(self.ln2(x))
        return x

In [19]:
class FrontierDirStateModel3D(nn.Module):
    def __init__(
        self,
        state_ch=10,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
    ):
        super().__init__()
        self.d_model = d_model

        self.global_encoder = SpatialGlobalStateEncoder3D(in_ch=state_ch, d_model=d_model)
        self.patch_encoder = LocalPatchEncoder3D(in_ch=state_ch, d_model=d_model)

        self.local_desc_proj = nn.Sequential(
            nn.Linear(16, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model),
        )
        self.fuse = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.input_ln = nn.LayerNorm(d_model)
        self.pos_scale = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))

        self.blocks = nn.ModuleList([
            SharedTransformerBlock(d_model=d_model, n_heads=n_heads)
            for _ in range(n_layers)
        ])
        self.shift_proj = nn.Sequential(
            nn.Linear(1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.bridge_fuse = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        self.bridge_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 6),
        )
        self.grow_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 6),
        )

        self.direction_proj = nn.Sequential(
            nn.Linear(3, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.phase_embedding = nn.Embedding(2,d_model)

    def _encode_inputs(self, state, node_patches, node_pos):
        B, N, C, P, _, _ = node_patches.shape
        patch_in = node_patches.reshape(B * N, C, P, P, P)
        patch_vec = self.patch_encoder(patch_in).reshape(B, N, -1)

        global_map = self.global_encoder(state)
        global_node = sample_global_features_at_nodes_3d(global_map, node_pos, state.shape[2:])
        local_desc = self.local_desc_proj(build_local_geom_descriptor(node_patches))
        pos = build_3d_sincos_posenc(node_pos, self.d_model, state_shape=state.shape[2:])

        fused = self.fuse(torch.cat([patch_vec, global_node, local_desc], dim=-1))
        x = fused + self.pos_scale * pos + 0.30 * patch_vec + 0.20 * global_node + 0.20 * local_desc
        x = self.input_ln(x)
        return x, global_map.mean(dim=(2, 3, 4))

    def forward(
        self,
        state,
        node_patches,
        node_pos,
        node_valid,
        grow_direction=None,
        n_shift=None,
        phase_mode = "grow",
    ):
        x, global_ctx = self._encode_inputs(state, node_patches, node_pos)

        # 2. 【新增】將 grow_direction 轉換並加到每個 Node 特徵上
        if grow_direction is not None:
            # grow_direction: [B, 3] -> dir_feat: [B, d_model]
            dir_feat = self.direction_proj(grow_direction)
            # 廣播到所有 Nodes: [B, N, d_model]
            x = x + dir_feat.unsqueeze(1)

        phase_id = 0 if phase_mode == "bridge" else 1 # phase status transform into num.

        B = x.shape[0]
        phase_tensor = torch.full((B,), phase_id, dtype=torch.long, device=x.device)
        
        # 3. 通過 Embedding 獲取特徵 [B, d_model]
        phase_feat = self.phase_embedding(phase_tensor)

        # 4. 加到節點特徵 x 上面 [B, N, d_model]
        x = x + phase_feat.unsqueeze(1)

        key_padding_mask = None
        if node_valid is not None:
            key_padding_mask = (node_valid <= 0)

        for blk in self.blocks:
            x = blk(x, key_padding_mask=key_padding_mask)

        grow_logits = self.grow_head(x)

        if n_shift is None:
            shift_feat = torch.zeros(
                (x.shape[0], x.shape[-1]),
                device=x.device,
                dtype=x.dtype,
            )
        else:
            if n_shift.ndim == 1:
                n_shift = n_shift.unsqueeze(-1)  # [B,1]
            shift_feat = self.shift_proj(n_shift.float())  # [B,D]

        shift_feat = shift_feat.unsqueeze(1).expand(-1, x.shape[1], -1)  # [B,N,D]
        bridge_x = self.bridge_fuse(torch.cat([x, shift_feat], dim=-1))
        bridge_logits = self.bridge_head(bridge_x)

        return {
            "node_feat": x,
            "grow_logits": grow_logits,
            "bridge_logits": bridge_logits,
            "global_ctx": global_ctx,
        }

In [20]:
model = FrontierDirStateModel3D(
    state_ch=10,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print("num params:", num_params / 1e6, "M")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

num params: 3.901333 M


RAG & Embeding

In [21]:
class SparsePatternEncoder3D(nn.Module):
    def __init__(self, d_model=128, n_heads=4, n_layers=3, max_nodes=512):
        """
        max_nodes: 為了防止 OOM，如果一個 pattern 體素太多，我們隨機採樣上限。
                   這跟你腳本中的 MAX_TRAIN_NODES 概念完全一致。
        """
        super().__init__()
        self.d_model = d_model
        self.max_nodes = max_nodes
        
        # 實體方塊的初始特徵 (我們可以用一個可學習的 Embedding 來代表 "這是一個建築方塊")
        self.voxel_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # 空間座標映射 (將 x, y, z 轉成高維向量)
        self.coord_proj = nn.Sequential(
            nn.Linear(3, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model)
        )
        
        # Transformer 編碼器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 最終輸出映射
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, d_model)

    def forward(self, pattern_vol):
        from torch.nn.utils.rnn import pad_sequence

        if isinstance(pattern_vol, list):
            B = len(pattern_vol)
            device = pattern_vol[0].device
            is_list = True
        else:
            B = pattern_vol.shape[0]
            device = pattern_vol.device
            is_list = False
            
        batch_coords = []
        lengths = []

        # 1. 提取座標 (這裡還是需要迴圈，但只是簡單的資料準備)
        for b in range(B):
            vol = pattern_vol[b] if is_list else pattern_vol[b, 0]
            coords = torch.nonzero(vol > 0.5, as_tuple=False).float()
            if coords.shape[-1] > 3:
                coords = coords[:, -3:]
                
            N = coords.shape[0]
            if N > self.max_nodes:
                perm = torch.randperm(N, device=device)[:self.max_nodes]
                coords = coords[perm]
            elif N == 0:
                # 處理全空的極端情況，塞一個虛擬點
                coords = torch.zeros((1, 3), device=device)
                
            centroid = coords.mean(dim=0, keepdim=True)
            coords = coords - centroid
            
            batch_coords.append(coords)
            lengths.append(coords.shape[0])

        # 2. 將不同長度的座標 Padding 成同一個大 Tensor [B, max_N, 3]
        padded_coords = pad_sequence(batch_coords, batch_first=True, padding_value=0.0)
        max_len = padded_coords.shape[1]

        # 3. 建立 Padding Mask，告訴 Transformer 哪些是補出來的 0，不需要算 Attention
        lengths_t = torch.tensor(lengths, device=device)
        # mask shape: [B, max_N]，True 代表是補零的位置 (需忽略)
        padding_mask = torch.arange(max_len, device=device).expand(B, max_len) >= lengths_t.unsqueeze(1)

        # 4. 「一口氣」進行特徵轉換 (GPU 平行加速！)
        pos_enc = self.coord_proj(padded_coords) # [B, max_N, d_model]
        x_tokens = self.voxel_token.view(1, 1, -1).expand(B, max_len, -1)
        x = x_tokens + pos_enc

        # 5. 「一口氣」通過 Transformer
        # 傳入 src_key_padding_mask 避免計算 Padding 區域
        x = self.transformer(x, src_key_padding_mask=padding_mask)

        # 6. Global Average Pooling (只針對有效點進行平均)
        valid_mask = (~padding_mask).float().unsqueeze(-1) # [B, max_N, 1]
        x_sum = (x * valid_mask).sum(dim=1)                # [B, d_model]
        x_count = valid_mask.sum(dim=1).clamp(min=1e-9)    # 避免除以 0
        x_pooled = x_sum / x_count

        out = self.head(self.norm(x_pooled))
        return F.normalize(out, p=2, dim=1)

In [22]:
pattern_encoder = SparsePatternEncoder3D().to(device)
num_params = sum(p.numel() for p in model.parameters())
print("num params:", num_params / 1e6, "M")

RAG_optimizer = torch.optim.AdamW(pattern_encoder.parameters(), lr=RAG_LR, weight_decay=WD)
RAG_scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

num params: 3.901333 M


k:\MasterEssay\Attempt_source_03\.venv\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loss Setting

In [23]:
# ==========================================
# 新增: Grow 階段專用進階 Loss 函數
# ==========================================

def masked_soft_dice_loss(logits, target, node_valid, smooth=1e-5):
    """1. Soft Dice Loss: 直接優化預測形狀與 Ground Truth 的重疊率"""
    valid = node_valid.unsqueeze(-1).expand_as(target)
    if valid.sum() == 0:
        return logits.sum() * 0.0
    
    prob = torch.sigmoid(logits) * valid
    tgt = target.float() * valid
    
    intersection = (prob * tgt).sum(dim=(1, 2))
    cardinality = prob.sum(dim=(1, 2)) + tgt.sum(dim=(1, 2))
    dice = (2.0 * intersection + smooth) / (cardinality + smooth)
    return (1.0 - dice).mean()

def bbox_leakage_penalty(prob_vol, target_vol, empty_mask, xy_margin=5):
    """
    2. Leakage Penalty: 嚴厲懲罰在 Target Bounding Box 外的任何預測 (防溢出)
    加入 xy_margin 參數，允許在 XY 平面上向外擴張一定的容許範圍。
    """
    B, _, X, Y, Z = target_vol.shape
    loss = prob_vol.new_tensor(0.0)
    valid_batches = 0
    
    for b in range(B):
        coords = torch.where(target_vol[b, 0] > 0.5)
        if coords[0].numel() == 0:
            continue
            
        # 1. 取得原始 BBox 邊界
        x_min, x_max = coords[0].min(), coords[0].max()
        y_min, y_max = coords[1].min(), coords[1].max()
        z_min, z_max = coords[2].min(), coords[2].max()
        
        # 2. 在 XY 平面向外擴大範圍，並確保不超出 Tensor 的最大與最小邊界
        x_min_exp = max(0, x_min - xy_margin)
        x_max_exp = min(X - 1, x_max + xy_margin)
        y_min_exp = max(0, y_min - xy_margin)
        y_max_exp = min(Y - 1, y_max + xy_margin)
        
        # 3. 建立 BBox 外部的 Mask，只懲罰原本是空地且在「擴張後 BBox」之外的地方
        outside_mask = torch.ones_like(target_vol[b, 0])
        # 將擴張後的內部區域設為 0.0 (不懲罰)
        outside_mask[x_min_exp:x_max_exp+1, y_min_exp:y_max_exp+1, z_min:z_max+1] = 0.0
        outside_mask = outside_mask * empty_mask[b, 0]
        
        leak_mass = (prob_vol[b, 0] * outside_mask).sum()
        
        # 4. 按照「擴張後的 BBox 體積」做 Normalize
        box_vol = (x_max_exp - x_min_exp + 1) * (y_max_exp - y_min_exp + 1) * (z_max - z_min + 1)
        
        loss += leak_mass / float(box_vol)
        valid_batches += 1
        
    return loss / max(valid_batches, 1)

def differentiable_void_ratio_loss(prob_vol, target_vol):
    """3. 可微 Void Ratio Loss: 確保生成結果的空隙率與 Target 吻合"""
    B = target_vol.shape[0]
    loss = prob_vol.new_tensor(0.0)
    valid_batches = 0
    
    for b in range(B):
        coords = torch.where(target_vol[b, 0] > 0.5)
        if coords[0].numel() == 0:
            continue
            
        x_min, x_max = coords[0].min(), coords[0].max()
        y_min, y_max = coords[1].min(), coords[1].max()
        z_min, z_max = coords[2].min(), coords[2].max()
        
        box_vol = float((x_max - x_min + 1) * (y_max - y_min + 1) * (z_max - z_min + 1))
        
        bbox_mask = torch.zeros_like(target_vol[b, 0])
        bbox_mask[x_min:x_max+1, y_min:y_max+1, z_min:z_max+1] = 1.0
        
        pred_mass = (prob_vol[b, 0] * bbox_mask).sum()
        target_mass = target_vol[b, 0].sum()
        
        pred_void_ratio = 1.0 - (pred_mass / box_vol)
        target_void_ratio = 1.0 - (target_mass / box_vol)
        
        loss += F.mse_loss(pred_void_ratio, target_void_ratio)
        valid_batches += 1
        
    return loss / max(valid_batches, 1)

def differentiable_spatial_spread_loss(prob_vol, target_vol):
    """4. 空間特徵對齊 (代替 Isovist): 計算預測與目標體素在 X,Y,Z 上的分佈變異數與質心差距"""
    B, _, X, Y, Z = target_vol.shape
    device = target_vol.device
    
    xs = torch.arange(X, device=device).float().view(1, 1, X, 1, 1)
    ys = torch.arange(Y, device=device).float().view(1, 1, 1, Y, 1)
    zs = torch.arange(Z, device=device).float().view(1, 1, 1, 1, Z)
    
    def get_moments(vol):
        mass = vol.sum(dim=(2, 3, 4), keepdim=True).clamp_min(1e-6)
        cx = (vol * xs).sum(dim=(2, 3, 4), keepdim=True) / mass
        cy = (vol * ys).sum(dim=(2, 3, 4), keepdim=True) / mass
        cz = (vol * zs).sum(dim=(2, 3, 4), keepdim=True) / mass
        
        vx = (vol * (xs - cx)**2).sum(dim=(2, 3, 4)) / mass.squeeze(-1).squeeze(-1).squeeze(-1)
        vy = (vol * (ys - cy)**2).sum(dim=(2, 3, 4)) / mass.squeeze(-1).squeeze(-1).squeeze(-1)
        vz = (vol * (zs - cz)**2).sum(dim=(2, 3, 4)) / mass.squeeze(-1).squeeze(-1).squeeze(-1)
        
        return torch.cat([cx.flatten(1), cy.flatten(1), cz.flatten(1)], dim=1), \
               torch.cat([vx, vy, vz], dim=1)
    
    pred_c, pred_v = get_moments(prob_vol)
    tgt_c, tgt_v = get_moments(target_vol.float())
    
    valid = (target_vol.sum(dim=(2, 3, 4)) > 0).float()
    if valid.sum() == 0:
        return prob_vol.new_tensor(0.0)
    
    loss_c = F.mse_loss(pred_c * valid, tgt_c * valid, reduction='none').sum(dim=1)
    loss_v = F.mse_loss(pred_v * valid, tgt_v * valid, reduction='none').sum(dim=1)
    
    norm_c = (X**2 + Y**2 + Z**2) ** 0.5
    norm_v = float(X**2 + Y**2 + Z**2)
    
    return ((loss_c / norm_c + loss_v / norm_v) * valid.squeeze(-1)).mean()

Train Helper

In [24]:

# ===============================================
# RAG CONTRASTIVE LOSS FUNCTION
# Add after the other loss functions (around line 3064)
# ===============================================
def rag_contrastive_loss(query_emb, positive_emb, negative_embs, temperature=RAG_TEMPERATURE):
    """
    InfoNCE contrastive loss for training the RAG encoder.
    
    Args:
        query_emb: [B, d_model] - Current frontier embeddings
        positive_emb: [B, d_model] - Ground truth pattern embeddings  
        negative_embs: [B, K, d_model] - Negative pattern embeddings
        temperature: Temperature parameter for softmax
        
    Returns:
        Contrastive loss scalar
    """
    B = query_emb.shape[0]
    
    # Normalize embeddings
    query_emb = F.normalize(query_emb, p=2, dim=1)
    positive_emb = F.normalize(positive_emb, p=2, dim=1)
    
    # Positive similarity: [B]
    pos_sim = torch.sum(query_emb * positive_emb, dim=1) / temperature
    
    # Negative similarities
    if negative_embs is not None and negative_embs.numel() > 0:
        negative_embs = F.normalize(negative_embs, p=2, dim=-1)  # [B, K, d_model]
        neg_sim = torch.matmul(
            query_emb.unsqueeze(1),  # [B, 1, d_model]
            negative_embs.transpose(-2, -1)  # [B, d_model, K]
        ).squeeze(1) / temperature  # [B, K]
        
        # Concatenate positive and negative similarities
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)  # [B, 1+K]
    else:
        logits = pos_sim.unsqueeze(1)  # [B, 1]
    
    # Labels: positive is always at index 0
    labels = torch.zeros(B, dtype=torch.long, device=query_emb.device)
    
    # Cross-entropy loss
    loss = F.cross_entropy(logits, labels)
    
    return loss
def get_negative_pattern_samples(pattern_bank, positive_indices, num_negatives=8, device='cuda'):
    bank_size = len(pattern_bank)
    B = len(positive_indices)
    
    # 建立所有可能的索引矩陣 [B, bank_size]
    # 我們在 CPU/GPU 端用 PyTorch 直接做矩陣運算，放棄緩慢的 python 迴圈 + list.remove
    all_indices = torch.arange(bank_size, device=device).unsqueeze(0).expand(B, -1)
    pos_idx_tensor = torch.tensor(positive_indices, device=device).unsqueeze(1)
    
    # 標記出不能選的 Positive Index
    valid_mask = all_indices != pos_idx_tensor
    
    # 產生隨機分數，並把不能選的地方設為 -inf，這樣 topk 就不會選到它
    rand_scores = torch.rand(B, bank_size, device=device)
    rand_scores[~valid_mask] = -float('inf')
    
    # 直接用 topk 挑選最高分的 num_negatives 個索引 (等同於不重複隨機採樣)
    # 如果 bank_size - 1 < num_negatives，允許重複採樣 (fallback)
    if bank_size - 1 >= num_negatives:
        _, neg_indices = torch.topk(rand_scores, num_negatives, dim=1)
    else:
        # 允許重複採樣的極端情況
        neg_indices = torch.randint(0, bank_size, (B, num_negatives), device=device)
    
    neg_indices_cpu = neg_indices.cpu().numpy()
    
    negative_samples = []
    for b in range(B):
        neg_vols = []
        for neg_idx in neg_indices_cpu[b]:
            # 從 Bank 讀取 Numpy，轉 Tensor 後上 GPU
            vol = torch.from_numpy(pattern_bank[neg_idx]["seed_local"]).float().to(device, non_blocking=True)
            neg_vols.append(vol)
        negative_samples.append(neg_vols)  
    
    return negative_samples

In [25]:
def neighbor_count_26_torch(x):
    device_ = x.device
    kernel = torch.ones((1, 1, 3, 3, 3), device=device_, dtype=torch.float32)
    kernel[:, :, 1, 1, 1] = 0.0
    y = F.conv3d(x.float(), kernel, padding=1)
    return y

def surface_boundary_from_known_torch(known_t, threshold=SURFACE_BOUNDARY_NEIGHBOR_THRESHOLD):
    surface = (known_t > 0.5).float()
    nb = neighbor_count_26_torch(surface)
    boundary = ((surface > 0.5) & (nb < float(threshold))).float()
    return boundary

@torch.no_grad()
def compute_exterior_interior_empty_torch_batch(known):
    """
    known: [B,1,X,Y,Z]
    返回:
      exterior_empty, interior_empty, outer_boundary, inner_boundary
    全部 shape = [B,1,X,Y,Z]
    """
    occ = (known > 0.5).float()
    B, _, X, Y, Z = occ.shape

    ext = torch.zeros_like(occ)
    ext[:, :, 0, :, :] = 1.0
    ext[:, :, -1, :, :] = 1.0
    ext[:, :, :, 0, :] = 1.0
    ext[:, :, :, -1, :] = 1.0
    ext[:, :, :, :, 0] = 1.0
    ext[:, :, :, :, -1] = 1.0
    ext = ext * (1.0 - occ)

    for _ in range(12):
        ext = F.max_pool3d(ext, kernel_size=3, stride=1, padding=1) * (1.0 - occ)

    exterior_empty = ext
    interior_empty = (1.0 - occ - exterior_empty).clamp_min(0.0)

    outer_boundary = occ * (F.max_pool3d(exterior_empty, 3, 1, 1) > 0.5).float()
    inner_boundary = occ * (F.max_pool3d(interior_empty, 3, 1, 1) > 0.5).float()

    return exterior_empty, interior_empty, outer_boundary, inner_boundary

def build_state_volume_3d_light_torch(
    known,
    frontier,
    exterior_empty,
    interior_empty,
    resolved_mask=None,
    pattern_start_frontier=None,
    phase_mode="grow",   # 新增
):
    """
    return: [B, C, X, Y, Z]
    原本 8 channels，現在 +2:
      ch[-2] = is_bridge
      ch[-1] = is_grow
    """
    known = (known > 0.5).float()
    frontier = (frontier > 0.5).float()
    exterior_empty = (exterior_empty > 0.5).float()
    interior_empty = (interior_empty > 0.5).float()

    if resolved_mask is None:
        resolved_mask = torch.zeros_like(known)
    else:
        resolved_mask = (resolved_mask > 0.5).float()

    if pattern_start_frontier is None:
        pattern_start_frontier = torch.zeros_like(known)
    else:
        pattern_start_frontier = (pattern_start_frontier > 0.5).float()

    unknown = (1.0 - known).clamp(0.0, 1.0)
    frontier_empty = frontier * unknown
    pattern_empty = pattern_start_frontier * unknown

    # phase one-hot
    if phase_mode == "bridge":
        phase_bridge = torch.ones_like(known)
        phase_grow = torch.zeros_like(known)
    elif phase_mode == "grow":
        phase_bridge = torch.zeros_like(known)
        phase_grow = torch.ones_like(known)
    else:
        raise ValueError(f"unknown phase_mode: {phase_mode}")

    state = torch.cat([
        known,                    # 0
        frontier,                 # 1
        exterior_empty,           # 2
        interior_empty,           # 3
        resolved_mask,            # 4
        pattern_start_frontier,   # 5
        frontier_empty,           # 6
        pattern_empty,            # 7
        phase_bridge,             # 8
        phase_grow,               # 9
    ], dim=1)

    return state

In [26]:
def select_frontier_nodes_torch(frontier_t, max_nodes=MAX_TRAIN_NODES):
    """
    frontier_t: [B,1,X,Y,Z]
    return:
      node_pos [B,N,3], node_valid [B,N]
    """
    B = frontier_t.shape[0]
    device_ = frontier_t.device
    node_pos_list = []
    node_valid_list = []

    for b in range(B):
        xyz = torch.nonzero(frontier_t[b, 0] > 0.5, as_tuple=False)
        if xyz.shape[0] == 0:
            pos = torch.zeros((1, 3), device=device_, dtype=torch.long)
            valid = torch.zeros((1,), device=device_, dtype=torch.float32)
        else:
            if xyz.shape[0] > max_nodes:
                perm = torch.randperm(xyz.shape[0], device=device_)[:max_nodes]
                xyz = xyz[perm]
            pos = xyz.long()
            valid = torch.ones((xyz.shape[0],), device=device_, dtype=torch.float32)

        node_pos_list.append(pos)
        node_valid_list.append(valid)

    Nmax = max(x.shape[0] for x in node_pos_list)
    node_pos = torch.zeros((B, Nmax, 3), device=device_, dtype=torch.long)
    node_valid = torch.zeros((B, Nmax), device=device_, dtype=torch.float32)

    for b in range(B):
        n = node_pos_list[b].shape[0]
        node_pos[b, :n] = node_pos_list[b]
        node_valid[b, :n] = node_valid_list[b]

    return node_pos, node_valid

def extract_local_patches_from_state_torch(state_b, node_pos_b, radius=PATCH_RADIUS):
    C, X, Y, Z = state_b.shape
    P = 2 * radius + 1

    if node_pos_b.shape[0] == 0:
        return state_b.new_zeros((0, C, P, P, P))

    state_pad = F.pad(
        state_b.unsqueeze(0),
        (radius, radius, radius, radius, radius, radius),
        mode="constant",
        value=0.0,
    )[0]

    patches_all = (
        state_pad.unfold(1, P, 1)
                 .unfold(2, P, 1)
                 .unfold(3, P, 1)
    )

    x = node_pos_b[:, 0].long()
    y = node_pos_b[:, 1].long()
    z = node_pos_b[:, 2].long()

    patches = patches_all[:, x, y, z]
    patches = patches.permute(1, 0, 2, 3, 4).contiguous()
    return patches

def self_state_from_topology_law_torch(node_pos_b, known_b, ext_empty_b, int_empty_b):
    if node_pos_b.shape[0] == 0:
        return node_pos_b.new_zeros((0,), dtype=torch.long)

    X, Y, Z = known_b.shape[-3:]
    pos = node_pos_b.long()
    N = pos.shape[0]

    touch_ext = torch.zeros(N, device=pos.device, dtype=torch.bool)
    touch_int = torch.zeros(N, device=pos.device, dtype=torch.bool)

    for dx, dy, dz in NEIGHBOR_6:
        nei = pos + torch.tensor([dx, dy, dz], device=pos.device)
        valid = (
            (nei[:, 0] >= 0) & (nei[:, 0] < X) &
            (nei[:, 1] >= 0) & (nei[:, 1] < Y) &
            (nei[:, 2] >= 0) & (nei[:, 2] < Z)
        )

        touch_ext |= ~valid

        if valid.any():
            vnei = nei[valid]
            touch_ext[valid] |= (ext_empty_b[vnei[:, 0], vnei[:, 1], vnei[:, 2]] > 0.5)
            touch_int[valid] |= (int_empty_b[vnei[:, 0], vnei[:, 1], vnei[:, 2]] > 0.5)

    out = torch.full((N,), SELF_TERMINAL, device=pos.device, dtype=torch.long)
    out[touch_ext] = SELF_OUTER
    out[touch_int] = SELF_INNER
    out[touch_ext & touch_int] = SELF_MIXED
    return out

def lock_state_from_topology_law_torch(node_pos_b, known_b, ext_empty_b, int_empty_b, self_b):
    if node_pos_b.shape[0] == 0:
        return node_pos_b.new_zeros((0, 6), dtype=torch.long)

    X, Y, Z = known_b.shape[-3:]
    pos = node_pos_b.long()
    N = pos.shape[0]
    out = torch.full((N, 6), LOCK_UNKNOWN, device=pos.device, dtype=torch.long)

    dir_offsets = torch.tensor(NEIGHBOR_6, device=pos.device, dtype=torch.long)

    for dir_id in range(6):
        nei = pos + dir_offsets[dir_id]
        valid = (
            (nei[:, 0] >= 0) & (nei[:, 0] < X) &
            (nei[:, 1] >= 0) & (nei[:, 1] < Y) &
            (nei[:, 2] >= 0) & (nei[:, 2] < Z)
        )

        out[~valid, dir_id] = LOCK_BLOCKED
        if not valid.any():
            continue

        vnei = nei[valid]
        vk = known_b[vnei[:, 0], vnei[:, 1], vnei[:, 2]] > 0.5
        ve = ext_empty_b[vnei[:, 0], vnei[:, 1], vnei[:, 2]] > 0.5
        vi = int_empty_b[vnei[:, 0], vnei[:, 1], vnei[:, 2]] > 0.5
        vs = self_b[valid]

        out_valid = out[valid, dir_id]
        out_valid[vk] = LOCK_SOLID

        m_ext = (~vk) & ve
        out_valid[m_ext] = torch.where(
            (vs[m_ext] == SELF_OUTER) | (vs[m_ext] == SELF_MIXED),
            torch.full_like(vs[m_ext], LOCK_SOLID),
            torch.full_like(vs[m_ext], LOCK_EXTERIOR),
        )

        m_int = (~vk) & (~ve) & vi
        out_valid[m_int] = torch.where(
            (vs[m_int] == SELF_INNER) | (vs[m_int] == SELF_MIXED),
            torch.full_like(vs[m_int], LOCK_SOLID),
            torch.full_like(vs[m_int], LOCK_INTERIOR),
        )

        out[valid, dir_id] = out_valid

    return out

@torch.no_grad()
def build_openended_episode_batch_torch(initial_known_t):
    """
    100% 純 GPU 的 episode 構建器：
    消滅了所有的 .item() CPU/GPU 同步，並修復了 self_tgt 與 lock_tgt 的零標籤 Bug。
    """
    known_t = (initial_known_t > 0.5).float()
    frontier_t = surface_boundary_from_known_torch(known_t)
    
    # 1. 取得空間標記 (純 GPU)
    exterior_empty, interior_empty, _, _ = compute_exterior_interior_empty_torch_batch(known_t)
    
    # 2. 建立 State Volume
    state_vol = build_state_volume_3d_light_torch(
        known_t, frontier_t, exterior_empty, interior_empty
    )
    
    # 3. 採樣 Frontier 節點
    node_pos, node_valid = select_frontier_nodes_torch(frontier_t, max_nodes=MAX_TRAIN_NODES)
    
    B, N, _ = node_pos.shape
    C, X, Y, Z = state_vol.shape[1:]
    P = 2 * PATCH_RADIUS + 1
    
    # 初始化 Tensor
    node_patches = torch.zeros((B, N, C, P, P, P), device=known_t.device, dtype=state_vol.dtype)
    self_tgt = torch.zeros((B, N), device=known_t.device, dtype=torch.long)
    lock_tgt = torch.zeros((B, N, 6), device=known_t.device, dtype=torch.long)
    
    # 4. 向量化提取與標籤計算 (消除 N 迴圈，只保留 B 迴圈，極度提速)
    for b in range(B):
        valid_idx = torch.where(node_valid[b] > 0.5)[0]
        if valid_idx.numel() == 0:
            continue
            
        pos_b = node_pos[b, valid_idx]
        
        # 使用極速 unfold 切片法，徹底消滅 x.item(), y.item() 迴圈
        patches_b = extract_local_patches_from_state_torch(state_vol[b], pos_b, radius=PATCH_RADIUS)
        node_patches[b, valid_idx] = patches_b
        
        # 正確計算 Topology Law Targets (修復原先的 zeros Bug)
        self_b = self_state_from_topology_law_torch(
            pos_b, known_t[b, 0], exterior_empty[b, 0], interior_empty[b, 0]
        )
        lock_b = lock_state_from_topology_law_torch(
            pos_b, known_t[b, 0], exterior_empty[b, 0], interior_empty[b, 0], self_b
        )
        
        self_tgt[b, valid_idx] = self_b
        lock_tgt[b, valid_idx] = lock_b

    return {
        "state_volume": state_vol,
        "known": known_t,
        "frontier": frontier_t,
        "outer_boundary": frontier_t,
        "inner_boundary": torch.zeros_like(frontier_t),
        "exterior_empty": exterior_empty,
        "interior_empty": interior_empty,
        "node_pos": node_pos,
        "node_patches": node_patches,
        "node_valid": node_valid,
        "self_tgt": self_tgt,   # <--- 真實計算的輔助標籤
        "lock_tgt": lock_tgt,   # <--- 真實計算的輔助標籤
    }


In [27]:
def shift3d_torch(x, dx, dy, dz):
    B, C, X, Y, Z = x.shape
    out = torch.zeros_like(x)

    sx0 = max(0, -dx)
    sx1 = min(X, X - dx)
    sy0 = max(0, -dy)
    sy1 = min(Y, Y - dy)
    sz0 = max(0, -dz)
    sz1 = min(Z, Z - dz)

    tx0 = max(0, dx)
    tx1 = min(X, X + dx)
    ty0 = max(0, dy)
    ty1 = min(Y, Y + dy)
    tz0 = max(0, dz)
    tz1 = min(Z, Z + dz)

    out[:, :, tx0:tx1, ty0:ty1, tz0:tz1] = x[:, :, sx0:sx1, sy0:sy1, sz0:sz1]
    return out

In [28]:
def random_shift_3d_batch_torch(batch_dict, pad=6):
    """
    Shift all spatial volume fields in the batch by the same random offset.

    This now includes the new attachment-related fields:
      - current_frontier
      - pattern_start_frontier
      - pattern_start_mask
      - bridge_target
    """
    spatial_keys = [
        "initial_known",
        "next_known",
        "next_delta",
        "preview_known",
        "preview_delta",
        "final_known",
        "final_delta",
        "current_frontier",
        "pattern_start_frontier",
        "pattern_start_mask",
        "bridge_target",
    ]

    ref = batch_dict["initial_known"]
    B, C, X, Y, Z = ref.shape
    device_ = ref.device

    sx = random.randint(-pad, pad)
    sy = random.randint(-pad, pad)
    sz = random.randint(-pad, pad)

    out = dict(batch_dict)
    for k in spatial_keys:
        if k in batch_dict and torch.is_tensor(batch_dict[k]) and batch_dict[k].ndim == 5:
            out[k] = shift3d_torch(batch_dict[k], sx, sy, sz)

    return out, (sx, sy, sz)

In [29]:
@torch.no_grad()
def prepare_growth_batch_from_bridge_torch(
    known_after_bridge,
    bridge_frontier_seed,
    grow_direction,
    pattern_start_frontier,
    n_shift,
    frontier_threshold=0.5, 
    phase_mode = "grow"
):
    known = (known_after_bridge > 0.5).float()
    frontier = (bridge_frontier_seed > frontier_threshold).float()

    exterior_empty, interior_empty, _, _ = compute_exterior_interior_empty_torch_batch(known)
    state = build_state_volume_3d_light_torch(
        known_after_bridge,
        bridge_frontier_seed,
        exterior_empty=exterior_empty,
        interior_empty=interior_empty,
        resolved_mask=None,
        pattern_start_frontier=pattern_start_frontier,
        phase_mode=phase_mode,   # 新增
    )

    node_pos, node_valid = select_frontier_nodes_torch(frontier, max_nodes=MAX_TRAIN_NODES)

    B, N, _ = node_pos.shape
    C = state.shape[1]
    P = 2 * PATCH_RADIUS + 1
    node_patches = torch.zeros((B, N, C, P, P, P), device=state.device, dtype=state.dtype)

    for b in range(B):
        valid_idx = torch.where(node_valid[b] > 0.5)[0]
        if valid_idx.numel() == 0:
            continue
        pos_b = node_pos[b, valid_idx]
        patches_b = extract_local_patches_from_state_torch(state[b], pos_b, radius=PATCH_RADIUS)
        node_patches[b, valid_idx] = patches_b

    _, _, X, Y, Z = known.shape
    grid_diag = float((X ** 2 + Y ** 2 + Z ** 2) ** 0.5)
    grid_diag = max(grid_diag, 1.0)
    n_shift_norm = (n_shift.float() / grid_diag).clamp(0.0, 1.0)

    return {
        "state": state,
        "known": known,
        "frontier": frontier,
        "exterior_empty": exterior_empty,
        "interior_empty": interior_empty,
        "node_pos": node_pos,
        "node_valid": node_valid,
        "node_patches": node_patches,
        "grow_direction": grow_direction,
        "n_shift": n_shift,
        "n_shift_norm": n_shift_norm,
        "pattern_start_frontier": pattern_start_frontier,
        "phase_mode":phase_mode,
    }

@torch.no_grad()
def prepare_training_batch_on_device(raw_batch):
    batch = {}
    for k, v in raw_batch.items():
        if torch.is_tensor(v):
            batch[k] = v.to(device=device, non_blocking=True)
        else:
            batch[k] = v

    known = (batch["initial_known"] > 0.5).float()

    # 新核心：以 start frontier seed 作為中樞 seed
    start_frontier_seed = (batch["start_frontier_seed"] > 0.5).float()

    exterior_empty, interior_empty, _, _ = compute_exterior_interior_empty_torch_batch(known)

    state = build_state_volume_3d_light_torch(
        known,
        start_frontier_seed,
        exterior_empty=exterior_empty,
        interior_empty=interior_empty,
        resolved_mask=None,
        pattern_start_frontier=batch["pattern_start_frontier"],
        phase_mode="bridge",   # 新增
    )

    node_pos, node_valid = select_frontier_nodes_torch(
        start_frontier_seed,
        max_nodes=MAX_TRAIN_NODES
    )

    B, N, _ = node_pos.shape
    C = state.shape[1]
    P = 2 * PATCH_RADIUS + 1
    node_patches = torch.zeros((B, N, C, P, P, P), device=state.device, dtype=state.dtype)

    for b in range(B):
        valid_idx = torch.where(node_valid[b] > 0.5)[0]
        if valid_idx.numel() == 0:
            continue
        pos_b = node_pos[b, valid_idx]
        patches_b = extract_local_patches_from_state_torch(
            state[b],
            pos_b,
            radius=PATCH_RADIUS
        )
        node_patches[b, valid_idx] = patches_b

    _, _, X, Y, Z = known.shape
    grid_diag = float((X ** 2 + Y ** 2 + Z ** 2) ** 0.5)
    grid_diag = max(grid_diag, 1.0)
    n_shift_norm = (batch["n_shift"].float() / grid_diag).clamp(0.0, 1.0)

    return {
        "state": state,
        "known": known,
        "frontier": start_frontier_seed,   # 這裡 frontier 已經等同於 start seed
        "start_frontier_seed": start_frontier_seed,

        "exterior_empty": exterior_empty,
        "interior_empty": interior_empty,

        "preview_known": batch["preview_known"],

        "node_pos": node_pos,
        "node_valid": node_valid,
        "node_patches": node_patches,

        "grow_direction": batch["grow_direction"],
        "grow_direction_valid": batch["grow_direction_valid"],

        "n_shift": batch["n_shift"],
        "n_shift_norm": n_shift_norm,

        # supervision
        "next_delta": batch["next_delta"],
        "bridge_target": batch["bridge_target"],
        "bridge_target_full":batch["bridge_target_full"],
        "pattern_start_frontier": batch["pattern_start_frontier"],
        "start_completion_target": batch["start_completion_target"],

        # debug / compatibility
        "current_frontier": batch["current_frontier"],
        "selected_source_frontier": batch["selected_source_frontier"],
    }

In [30]:
@torch.no_grad()
def build_action_targets_from_step_target_torch(batch, step_target):
    step_target = step_target.detach()
    node_pos = batch["node_pos"].detach()
    node_valid = batch["node_valid"].detach()

    B, N, _ = node_pos.shape
    X, Y, Z = step_target.shape[2:]
    out = torch.zeros((B, N, 6), device=step_target.device, dtype=torch.float32)

    for dir_id, (dx, dy, dz) in enumerate(NEIGHBOR_6):
        pos = node_pos.clone()
        pos[..., 0] += dx
        pos[..., 1] += dy
        pos[..., 2] += dz

        valid = (
            (node_valid > 0.5)
            & (pos[..., 0] >= 0) & (pos[..., 0] < X)
            & (pos[..., 1] >= 0) & (pos[..., 1] < Y)
            & (pos[..., 2] >= 0) & (pos[..., 2] < Z)
        )

        b_idx, n_idx = torch.where(valid)
        xx = pos[b_idx, n_idx, 0].long()
        yy = pos[b_idx, n_idx, 1].long()
        zz = pos[b_idx, n_idx, 2].long()

        out[b_idx, n_idx, dir_id] = step_target[b_idx, 0, xx, yy, zz]

    return out


def masked_bce_loss(logits, target, node_valid):
    valid = node_valid.unsqueeze(-1).expand_as(target)
    if valid.sum() == 0:
        return logits.sum() * 0.0   # zero but keeps grad_fn alive

    loss = F.binary_cross_entropy_with_logits(
        logits,
        target.float(),
        reduction="none",
    )
    return loss[valid].mean()

In [31]:
def grow_smooth_loss(prob_vol, empty_mask):
    """
    3D Total Variation smoothness loss on grow_prob_vol.
    Penalizes sharp probability jumps between adjacent voxels,
    discouraging scattered speckle predictions.
    Only measures transitions entirely inside empty space where grow can occur.
    prob_vol, empty_mask: [B, 1, X, Y, Z]
    """
    # 1. 先不要乘上 empty_mask，直接對原始預測值截斷
    p = prob_vol.clamp(0.0, 1.0)

    # 2. 計算所有相鄰體素的絕對差異
    dx = (p[:, :, 1:, :, :] - p[:, :, :-1, :, :]).abs()
    dy = (p[:, :, :, 1:, :] - p[:, :, :, :-1, :]).abs()
    dz = (p[:, :, :, :, 1:] - p[:, :, :, :, :-1]).abs()

    # 3. 建立位移後的 mask，確保「相鄰的兩個體素都必須是 empty_mask」才進行平滑懲罰
    mask_x = empty_mask[:, :, 1:, :, :] * empty_mask[:, :, :-1, :, :]
    mask_y = empty_mask[:, :, :, 1:, :] * empty_mask[:, :, :, :-1, :]
    mask_z = empty_mask[:, :, :, :, 1:] * empty_mask[:, :, :, :, :-1]

    # 4. 只針對真正空地內部的差異進行懲罰，消除牆壁邊緣的假性懲罰
    dx = dx * mask_x
    dy = dy * mask_y
    dz = dz * mask_z

    # 5. (優化建議) 根據實際計算的空地體積來 Normalize，而不是整個 Grid
    # 避免因為 Grid 大小改變導致 Loss 權重漂移
    n = max(float(empty_mask.sum()), 1.0)
    
    tv = (dx.sum(dim=(1, 2, 3, 4)) + dy.sum(dim=(1, 2, 3, 4)) + dz.sum(dim=(1, 2, 3, 4))) / n
    return tv.mean()


def grow_continuity_loss(prob_vol, empty_mask, eps=1e-6):
    """
    Continuity loss: penalizes predicted voxels that lack neighbourhood support.
    For each voxel, its 3x3x3 local average measures how densely its
    surroundings are also predicted. A high-prob voxel with a low local average
    is an isolated discontinuity — this loss penalises it.
    Encourages solid, gap-free grow regions rather than scattered speckles.
    prob_vol, empty_mask: [B, 1, X, Y, Z]
    """
    p = (prob_vol * empty_mask).clamp(0.0, 1.0)

    # local neighbourhood density — 3x3x3 average pool (includes self)
    local_avg = F.avg_pool3d(p, kernel_size=3, stride=1, padding=1)

    # isolation term: high prediction, low neighbourhood support
    isolation = p * (1.0 - local_avg)

    pred_mass = p.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    loss = isolation.sum(dim=(1, 2, 3, 4)) / pred_mass
    return loss.mean()

In [32]:
# %% 
def masked_balanced_focal_bce_loss(
    logits,
    target,
    node_valid,
    pos_weight=6.0,
    neg_weight=1.0,
    focal_gamma=2.0,
    label_smoothing=0.0,
):
    """
    logits, target: [B, N, 6]
    node_valid: [B, N] bool/float

    核心目的：
    1) 正樣本加權，避免被大量 0 淹沒
    2) focal，降低 easy negative 影響
    3) 仍然保持 BCE with logits 的穩定性
    """
    valid = node_valid.unsqueeze(-1).expand_as(target)
    if valid.sum() == 0:
        return logits.sum() * 0.0   # zero but keeps grad_fn alive

    target = target.float()

    if label_smoothing > 0.0:
        target = target * (1.0 - label_smoothing) + 0.5 * label_smoothing

    bce = F.binary_cross_entropy_with_logits(
        logits,
        target,
        reduction="none",
    )  # [B,N,6]

    prob = torch.sigmoid(logits)
    pt = torch.where(target > 0.5, prob, 1.0 - prob)   # 正類就睇 p，負類就睇 1-p
    focal = (1.0 - pt).clamp_min(1e-6).pow(focal_gamma)

    cls_w = torch.where(target > 0.5,
                        torch.full_like(target, float(pos_weight)),
                        torch.full_like(target, float(neg_weight)))

    loss = bce * focal * cls_w
    return loss[valid].mean()


def masked_positive_coverage_loss(
    logits,
    target,
    node_valid,
    eps=1e-6,
):
    """
    防止模型學成全空。
    對每個 sample，直接比較：
      預測正樣本總量  vs  target 正樣本總量
    只在 valid node × dir 上計。
    """
    valid = node_valid.unsqueeze(-1).expand_as(target).float()
    if valid.sum() == 0:
        return logits.sum() * 0.0   # zero but keeps grad_fn alive

    prob = torch.sigmoid(logits) * valid
    tgt = target.float() * valid

    pred_mass = prob.sum(dim=(1, 2))
    tgt_mass = tgt.sum(dim=(1, 2))

    has_pos = (tgt_mass > 0).float()
    if has_pos.sum() == 0:
        return logits.new_tensor(0.0)

    ratio = pred_mass / tgt_mass.clamp_min(eps)

    # 只懲罰「預測太少」，唔懲罰比 target 多少少
    miss = (1.0 - ratio).clamp_min(0.0)

    return (miss * has_pos).sum() / has_pos.sum().clamp_min(1.0)


def masked_positive_recall_loss(
    logits,
    target,
    node_valid,
    eps=1e-6,
):
    """
    進一步強調正樣本 recall：
    只在 target=1 的位上看 sigmoid(logit)，希望佢哋高。
    """
    valid = node_valid.unsqueeze(-1).expand_as(target)
    pos_mask = (target > 0.5) & valid

    if pos_mask.sum() == 0:
        return logits.new_tensor(0.0)

    prob = torch.sigmoid(logits)
    pos_prob = prob[pos_mask]

    return (1.0 - pos_prob).mean()

In [33]:
# %% [markdown]
# # GPU corridor mask (replace numpy/scipy corridor mask)
def _shift_mask_by_direction_torch(mask_t: torch.Tensor,
                                   direction_t: torch.Tensor,
                                   steps: int = 1) -> torch.Tensor:
    """
    Translate a [B,1,X,Y,Z] binary mask by `steps` voxels along direction_t [B,3].
    direction is unit-rounded to the nearest integer grid step each call.
    """
    B, _, X, Y, Z = mask_t.shape
    out = torch.zeros_like(mask_t)
    for b in range(B):
        d = direction_t[b].float()
        d = d / d.norm().clamp_min(1e-6)
        dx, dy, dz = [int(v) for v in (d * steps).round().long().tolist()]

        arr = mask_t[b, 0]
        sx0, sx1 = max(0, -dx), min(X, X - dx)
        sy0, sy1 = max(0, -dy), min(Y, Y - dy)
        sz0, sz1 = max(0, -dz), min(Z, Z - dz)
        tx0, tx1 = max(0,  dx), min(X, X + dx)
        ty0, ty1 = max(0,  dy), min(Y, Y + dy)
        tz0, tz1 = max(0,  dz), min(Z, Z + dz)

        if sx1 > sx0 and sy1 > sy0 and sz1 > sz0:
            out[b, 0, tx0:tx1, ty0:ty1, tz0:tz1] = arr[sx0:sx1, sy0:sy1, sz0:sz1]
    return out

def _dilate3d_binary_torch(x, iters=1):
    """
    x: [B,1,X,Y,Z] in {0,1}
    """
    x = (x > 0.5).float()
    for _ in range(int(iters)):
        x = F.max_pool3d(x, kernel_size=3, stride=1, padding=1)
    return (x > 0.5).float()


def _center_of_mass_from_mask_torch(mask):
    """
    mask: [B,1,X,Y,Z]
    return: [B,3]
    """
    B, _, X, Y, Z = mask.shape
    device = mask.device
    dtype = mask.dtype

    xs = torch.arange(X, device=device, dtype=dtype).view(1, 1, X, 1, 1)
    ys = torch.arange(Y, device=device, dtype=dtype).view(1, 1, 1, Y, 1)
    zs = torch.arange(Z, device=device, dtype=dtype).view(1, 1, 1, 1, Z)

    mass = mask.sum(dim=(1, 2, 3, 4)).clamp_min(1.0)  # [B]
    cx = (mask * xs).sum(dim=(1, 2, 3, 4)) / mass
    cy = (mask * ys).sum(dim=(1, 2, 3, 4)) / mass
    cz = (mask * zs).sum(dim=(1, 2, 3, 4)) / mass

    ctr = torch.stack([cx, cy, cz], dim=-1)  # [B,3]
    return ctr


def _make_axis_tube_mask_torch(start_seed_t, current_frontier_t, grow_direction_t, radius=3.0):
    """
    根據 start/frontier 中心與 reverse grow direction，生成一條粗 tube。
    start_seed_t: [B,1,X,Y,Z]
    current_frontier_t: [B,1,X,Y,Z]
    grow_direction_t: [B,3]
    return: [B,1,X,Y,Z]
    """
    B, _, X, Y, Z = start_seed_t.shape
    device = start_seed_t.device
    dtype = start_seed_t.dtype

    seed_ctr = _center_of_mass_from_mask_torch(start_seed_t)         # [B,3]
    front_ctr = _center_of_mass_from_mask_torch(current_frontier_t)  # [B,3]

    # bridge 是 reverse，所以軸向取 frontier -> seed 或 -grow_direction
    axis = seed_ctr - front_ctr  # [B,3]
    axis_norm = torch.norm(axis, dim=-1, keepdim=True).clamp_min(1e-6)
    axis = axis / axis_norm

    # 若中心重合或太近，fallback 用 -grow_direction
    g = grow_direction_t.float()
    g_norm = torch.norm(g, dim=-1, keepdim=True).clamp_min(1e-6)
    rg = -(g / g_norm)

    use_rg = (axis_norm.squeeze(-1) < 1.0).float().unsqueeze(-1)
    axis = axis * (1.0 - use_rg) + rg * use_rg
    axis = axis / torch.norm(axis, dim=-1, keepdim=True).clamp_min(1e-6)

    xs = torch.arange(X, device=device, dtype=dtype)
    ys = torch.arange(Y, device=device, dtype=dtype)
    zs = torch.arange(Z, device=device, dtype=dtype)
    gx, gy, gz = torch.meshgrid(xs, ys, zs, indexing="ij")
    grid = torch.stack([gx, gy, gz], dim=-1).unsqueeze(0).expand(B, -1, -1, -1, -1)  # [B,X,Y,Z,3]

    mid = 0.5 * (seed_ctr + front_ctr)  # [B,3]
    rel = grid - mid[:, None, None, None, :]  # [B,X,Y,Z,3]

    depth = (rel * axis[:, None, None, None, :]).sum(dim=-1, keepdim=True)  # [B,X,Y,Z,1]
    proj = depth * axis[:, None, None, None, :]                              # [B,X,Y,Z,3]
    lateral = rel - proj
    lateral_norm = torch.norm(lateral, dim=-1)                               # [B,X,Y,Z]

    tube = (lateral_norm <= float(radius)).float().unsqueeze(1)              # [B,1,X,Y,Z]
    return tube

import math

def _make_morphing_bbox_frustum_mask_torch(start_seed_t, current_frontier_t, min_radius=2.0, padding=1.0):
    """
    動態漸變矩形錐體 (Morphing BBox Frustum)
    自動計算兩端的 BBox 大小，並沿著生長軸線進行 Cosine 平滑漸變。
    """
    B, C, D, H, W = start_seed_t.shape
    device = start_seed_t.device

    # 1. 建立全局 3D 網格座標
    z = torch.arange(D, device=device)
    y = torch.arange(H, device=device)
    x = torch.arange(W, device=device)
    zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")
    grid = torch.stack([xx, yy, zz], dim=-1).float() # [D, H, W, 3]
    grid = grid.view(1, D, H, W, 3).expand(B, -1, -1, -1, -1)
    
    frustum_mask = torch.zeros_like(start_seed_t)
    
    for b in range(B):
        seed = start_seed_t[b, 0]
        target = current_frontier_t[b, 0]
        
        # 找出兩端的有效體素座標
        seed_pts = grid[b][seed > 0.5]
        target_pts = grid[b][target > 0.5]
        
        if len(seed_pts) == 0 or len(target_pts) == 0:
            continue
            
        # 計算質心
        c_seed = seed_pts.mean(dim=0)
        c_target = target_pts.mean(dim=0)
        
        # 生長軸線 (Axis)
        vec = c_target - c_seed
        length = torch.norm(vec)
        if length < 1e-3:
            frustum_mask[b, 0] = 1.0 # 退化為全滿
            continue
            
        axis = vec / length
        
        # 2. 建立正交基底 (U, V) 代表橫截面的「寬」與「高」
        # 找一個不與 axis 完全平行的參考向量 (預設找 Y 軸)
        ref = torch.tensor([0.0, 1.0, 0.0], device=device)
        if torch.abs(torch.dot(axis, ref)) > 0.9: 
            ref = torch.tensor([1.0, 0.0, 0.0], device=device) # 如果往 Y 走，就改用 X 軸
            
        u = torch.cross(axis, ref)
        u = u / torch.norm(u)
        v = torch.cross(axis, u) # u, v, axis 形成完美的 3D 垂直座標系
        
        # 3. 測量兩端真實的 BBox 半寬與半高
        seed_rel = seed_pts - c_seed
        w_seed = torch.abs(torch.matmul(seed_rel, u)).max().clamp_min(min_radius)
        h_seed = torch.abs(torch.matmul(seed_rel, v)).max().clamp_min(min_radius)
        
        target_rel = target_pts - c_target
        w_target = torch.abs(torch.matmul(target_rel, u)).max().clamp_min(min_radius)
        h_target = torch.abs(torch.matmul(target_rel, v)).max().clamp_min(min_radius)
        
        # 4. 對空間中所有點進行投影計算
        rel_all = grid[b] - c_seed
        
        # 投影到生長軸上的距離 (0 到 length)
        t_proj = (rel_all * axis).sum(dim=-1)
        t_ratio = (t_proj / length).clamp(0.0, 1.0) # 正規化進度 0~1
        
        # 投影到橫截面的 U, V 軸上的距離
        dist_u = torch.abs((rel_all * u).sum(dim=-1))
        dist_v = torch.abs((rel_all * v).sum(dim=-1))
        
        # 5. Cosine 漸變計算 (Morphing)
        blend = (1.0 - torch.cos(math.pi * t_ratio)) / 2.0 
        
        # 動態計算該深度(t)應該要有的長寬限制
        w_limit = w_seed + blend * (w_target - w_seed)
        h_limit = h_seed + blend * (h_target - h_seed)
        
        # 6. 遮罩判斷
        in_bbox = (dist_u <= (w_limit + padding)) & (dist_v <= (h_limit + padding))
        # 允許前後軸向稍微凸出一點 padding，避免切得太死
        in_axis = (t_proj >= -padding) & (t_proj <= length + padding) 
        
        frustum_mask[b, 0] = (in_bbox & in_axis).float()
        
    return frustum_mask

def corridor_mask_from_seed_to_frontier_torch(
    start_seed_t,
    current_frontier_t,
    empty_mask_t,
    grow_direction_t,
    start_dilate_iters=6,
    frontier_dilate_iters=6,
    tube_radius=3.0,
    mix_keep_max=True,
    eat_in_iters=6,  # 🌟 控制吃進「現有建築」的深度
):
    # ... 前面轉換 float 邏輯不變 ...

    # 🌟 修正點：我們要吃的是「現有的 Frontier (建築面)」
    # 將 current_frontier 往「建築內部」膨脹
    eat_in_zone = _dilate3d_binary_torch(current_frontier_t, iters=eat_in_iters)
    
    # 將原本嚴格的 empty_mask (只准在空氣長) 加上這個 zone (准許在牆壁邊緣長)
    # 這樣 Bridge 就能穿透牆壁表面
    relaxed_empty_mask = torch.maximum(empty_mask_t, eat_in_zone)

    # 1. 雙端膨脹 (保持原樣，這負責 corridor 的粗細)
    seed_band = _dilate3d_binary_torch(start_seed_t, iters=start_dilate_iters)
    frontier_band = _dilate3d_binary_torch(current_frontier_t, iters=frontier_dilate_iters)

    # 2. Morphing BBox (確保 Frustum 覆蓋範圍夠深)
    frustum = _make_morphing_bbox_frustum_mask_torch(
        start_seed_t=start_seed_t,
        current_frontier_t=current_frontier_t,
        min_radius=tube_radius,
        # 💡 重要：padding 必須覆蓋 eat_in 的深度，否則 frustum 會把嵌入部分切掉
        padding=max(3.0, float(eat_in_iters) + 2.0) 
    )

    # 3. 最終計算
    if mix_keep_max:
        band = torch.maximum(seed_band, frontier_band)
    else:
        band = seed_band * frontier_band

    corridor = band * frustum
    
    # 🌟 使用針對 Frontier 放寬的遮罩
    corridor = corridor * relaxed_empty_mask
    return corridor.clamp(0.0, 1.0)

In [34]:
# ─────────────────────────────────────────────────────────────
# ▌ SECTION B — PYTORCH (training-side)                       ▌
# ─────────────────────────────────────────────────────────────
 
def _compute_lateral_bbox_torch(mask_t, grow_direction_t, eps=1e-8):
    """
    Compute BBox lateral area for each sample in the batch.
 
    Parameters
    ----------
    mask_t : Tensor [B, 1, X, Y, Z]  — binary frontier/pattern mask
    grow_direction_t : Tensor [B, 3]
 
    Returns
    -------
    area : Tensor [B]  — lateral BBox area (in voxels²)
    """
    B = mask_t.shape[0]
    device = mask_t.device
    areas = torch.zeros(B, device=device, dtype=torch.float32)
 
    for b in range(B):
        pts = torch.nonzero(mask_t[b, 0] > 0.5, as_tuple=False).float()  # [N, 3]
        if pts.shape[0] == 0:
            areas[b] = 4.0
            continue
 
        g = grow_direction_t[b].float()
        g = g / g.norm().clamp_min(eps)
 
        depth = (pts @ g).unsqueeze(-1) * g.unsqueeze(0)  # [N, 3]
        lat = pts - depth                                  # [N, 3]
 
        lat_min = lat.min(dim=0).values
        lat_max = lat.max(dim=0).values
        spans = (lat_max - lat_min + 1.0)                 # [3]
        spans_sorted, _ = spans.sort(descending=True)
        areas[b] = spans_sorted[0] * spans_sorted[1]
 
    return areas
 
 
def _compute_lateral_void_ratio_torch(mask_t, area):
    """
    Approximate void ratio at an endpoint:  1 − fill_density.
    fill_density = (number of active voxels) / (lateral BBox area).
 
    Parameters
    ----------
    mask_t : Tensor [B, 1, X, Y, Z]
    area   : Tensor [B]
 
    Returns
    -------
    vr : Tensor [B] in [0, 1]
    """
    mass = (mask_t > 0.5).float().sum(dim=(1, 2, 3, 4))   # [B]
    vr = 1.0 - (mass / area.clamp_min(1.0)).clamp(0.0, 1.0)
    return vr
 
 
def _compute_endpoint_isovist_np_torch(mask_t, known_t, grow_direction_t,
                                       n_lateral_dirs=4, max_depth=20, eps=1e-8):
    """
    Compute the HARD (non-differentiable) isovist depth targets at one endpoint.
 
    The isovist is measured by casting `n_lateral_dirs` rays from the centroid
    of `mask_t`, perpendicular to `grow_direction_t`, through the EMPTY space
    (where known_t == 0).  We stop when we hit a solid voxel.
 
    Used ONLY to build the target; gradients are not needed here.
 
    Parameters
    ----------
    mask_t           : Tensor [B, 1, X, Y, Z]  — endpoint mask (frontier or seed)
    known_t          : Tensor [B, 1, X, Y, Z]  — current solid occupancy
    grow_direction_t : Tensor [B, 3]
    n_lateral_dirs   : int  — 4 (±u, ±v) by default
    max_depth        : int
 
    Returns
    -------
    iso_targets : Tensor [B, n_lateral_dirs]  — depth per ray per sample
    lateral_dirs : Tensor [B, n_lateral_dirs, 3]  — ray directions used
    """
    B, _, X, Y, Z = mask_t.shape
    device = mask_t.device
 
    iso_targets = torch.zeros((B, n_lateral_dirs), device=device, dtype=torch.float32)
    lat_dirs_out = torch.zeros((B, n_lateral_dirs, 3), device=device, dtype=torch.float32)
 
    for b in range(B):
        g = grow_direction_t[b].float()
        g = g / g.norm().clamp_min(eps)
 
        # Build 2 orthogonal lateral unit vectors (u, v)
        if g[0].abs() < 0.9:
            u = torch.tensor([1.0, 0.0, 0.0], device=device)
        else:
            u = torch.tensor([0.0, 1.0, 0.0], device=device)
        u = u - torch.dot(u, g) * g
        u = u / u.norm().clamp_min(eps)
        v = torch.linalg.cross(g, u)
        v = v / v.norm().clamp_min(eps)
 
        dirs = [u, -u, v, -v]               # 4 lateral rays
        lat_dirs_out[b] = torch.stack(dirs)  # [4, 3]
 
        # Centroid of the endpoint mask
        pts = torch.nonzero(mask_t[b, 0] > 0.5, as_tuple=False).float()
        if pts.shape[0] == 0:
            continue
        ctr = pts.mean(dim=0)               # [3]
 
        solid = (known_t[b, 0] > 0.5)      # [X, Y, Z]
 
        with torch.no_grad():
            for di, d in enumerate(dirs):
                depth = float(max_depth)
                for step in range(1, max_depth + 1):
                    pos = ctr + d * step
                    xi = int(pos[0].clamp(0, X - 1).item())
                    yi = int(pos[1].clamp(0, Y - 1).item())
                    zi = int(pos[2].clamp(0, Z - 1).item())
 
                    # Out-of-bounds → ray exits the grid
                    if (pos[0] < 0 or pos[0] >= X or
                            pos[1] < 0 or pos[1] >= Y or
                            pos[2] < 0 or pos[2] >= Z):
                        depth = float(step - 1)
                        break
 
                    if solid[xi, yi, zi]:
                        depth = float(step - 1)
                        break
 
                iso_targets[b, di] = depth
 
    return iso_targets, lat_dirs_out        # [B, 4],  [B, 4, 3]
 
 
def _differentiable_ray_depth_torch(field_b, centroid, direction, max_depth=16, eps=1e-6):
    """
    Differentiable isovist ray depth via soft alpha-compositing.
 
    Gradients flow back through `field_b` (the predicted probability volume).
 
    Parameters
    ----------
    field_b   : Tensor [X, Y, Z]  — predicted probability field (0..1)
    centroid  : Tensor [3]        — ray start (float, grid coords)
    direction : Tensor [3]        — unit ray direction (lateral)
    max_depth : int
 
    Returns
    -------
    expected_depth : scalar Tensor  (differentiable w.r.t. field_b)
 
    Note on compositing:
        At each step s, alpha_s = field_b[pos_s] (probability of solid).
        expected_depth += transmittance * alpha_s * s
        transmittance  *= (1 − alpha_s)
    This is the classical NeRF-style volume rendering depth estimator.
    """
    X, Y, Z = field_b.shape
    device = field_b.device
 
    ray_depth = field_b.new_zeros(())
    transmittance = field_b.new_ones(())
 
    for step in range(1, max_depth + 1):
        pos = centroid + direction * step
 
        # Boundary check (scalar, no gradient needed)
        if (pos[0].item() < 0 or pos[0].item() >= X or
                pos[1].item() < 0 or pos[1].item() >= Y or
                pos[2].item() < 0 or pos[2].item() >= Z):
            break
 
        xi = pos[0].long().clamp(0, X - 1)
        yi = pos[1].long().clamp(0, Y - 1)
        zi = pos[2].long().clamp(0, Z - 1)
 
        # Differentiable gather: grad flows through field_b[xi, yi, zi]
        alpha = field_b[xi, yi, zi].clamp(0.0, 1.0)
 
        ray_depth = ray_depth + transmittance * alpha * float(step)
        transmittance = transmittance * (1.0 - alpha)
 
        if transmittance.item() < 0.01:
            break
 
    return ray_depth
 
 
def bridge_dynamic_envelope_loss(
    bridge_prob_vol,           # [B, 1, X, Y, Z]  可導
    pattern_vol,               # [B, 1, X, Y, Z]  目標 Pattern
    input_vol,                 # [B, 1, X, Y, Z]  現有建築
    current_frontier_t,        # [B, 1, X, Y, Z]  A端起點
    pattern_start_frontier_t,  # [B, 1, X, Y, Z]  B端終點
    known_t,                   
    empty_mask_t,              # [B, 1, X, Y, Z]
    grow_direction_t,          # [B, 3]
    n_bins=8,
    eps=1e-6,
):
    B, _, X, Y, Z = bridge_prob_vol.shape
    device = bridge_prob_vol.device

    # ── 1. 基礎幾何特徵提取 ───────────────────────────────────────
    area_A = _compute_lateral_bbox_torch(input_vol,       grow_direction_t)
    area_B = _compute_lateral_bbox_torch(pattern_vol,     grow_direction_t)
    vr_A = _compute_lateral_void_ratio_torch(input_vol,   area_A)
    vr_B = _compute_lateral_void_ratio_torch(pattern_vol, area_B)

    # ── 2. 準備座標網格與 Moment 函數 ─────────────────────────────────────
    xs = torch.arange(X, device=device).float().view(1, 1, X, 1, 1)
    ys = torch.arange(Y, device=device).float().view(1, 1, 1, Y, 1)
    zs = torch.arange(Z, device=device).float().view(1, 1, 1, 1, Z)

    def get_moments(vol):
        # 提取 3D 體積的質心 (Centroid) 與 方差 (Variance)
        mass = vol.sum(dim=(2, 3, 4), keepdim=True).clamp_min(1e-6)
        cx = (vol * xs).sum(dim=(2, 3, 4), keepdim=True) / mass
        cy = (vol * ys).sum(dim=(2, 3, 4), keepdim=True) / mass
        cz = (vol * zs).sum(dim=(2, 3, 4), keepdim=True) / mass
        
        vx = (vol * (xs - cx)**2).sum(dim=(2, 3, 4)) / mass.squeeze(-1).squeeze(-1).squeeze(-1)
        vy = (vol * (ys - cy)**2).sum(dim=(2, 3, 4)) / mass.squeeze(-1).squeeze(-1).squeeze(-1)
        vz = (vol * (zs - cz)**2).sum(dim=(2, 3, 4)) / mass.squeeze(-1).squeeze(-1).squeeze(-1)
        
        return torch.cat([cx.flatten(1), cy.flatten(1), cz.flatten(1)], dim=1), \
               torch.cat([vx, vy, vz], dim=1).clamp_min(0.0)

    # ── 3. 提取全局目標特徵 (A端與B端基準) ──────────────────────────────
    with torch.no_grad():
        # 位置基準 (Frontier 決定路徑)
        cA, _ = get_moments(current_frontier_t.float())
        cB, _ = get_moments(pattern_start_frontier_t.float())

        # 肥瘦基準 (Volume 決定粗細)
        _, vA = get_moments(input_vol.float())
        _, vB = get_moments(pattern_vol.float())

        # 將 Variance 與 Area 轉化為長度尺度以進行平滑插值
        std_A = torch.sqrt(vA.clamp_min(1e-6))
        std_B = torch.sqrt(vB.clamp_min(1e-6))
        radius_A = torch.sqrt(area_A.clamp_min(1e-6))
        radius_B = torch.sqrt(area_B.clamp_min(1e-6))

    # ── 4. 準備 Bin 座標系與全局場 ──────────────────────────────────────────
    # 生成 [X, Y, Z, 3] 網格
    gx, gy, gz = torch.meshgrid(torch.arange(X, device=device), 
                                torch.arange(Y, device=device), 
                                torch.arange(Z, device=device), indexing="ij")
    grid = torch.stack([gx.float(), gy.float(), gz.float()], dim=-1)

    # 全局預測場 (套用遮罩)
    field_all = (bridge_prob_vol * empty_mask_t).clamp(0.0, 1.0)
    
    loss_void_accum = bridge_prob_vol.new_zeros(())
    loss_iso_accum  = bridge_prob_vol.new_zeros(()) # 這裡只存質心偏離
    loss_v_global   = bridge_prob_vol.new_zeros(()) # 全局方差對齊
    valid_count = 0

    for b in range(B):
        # 方向向量歸一化
        g = grow_direction_t[b].float()
        g = g / g.norm().clamp_min(eps)
        
        # 橫截面遮罩 (Lateral Mask): 1 - 生長軸貢獻
        lateral_mask = (1.0 - g.pow(2)).detach() 

        # 找出 A、B 兩端質心建立局部座標系
        pts_A = torch.nonzero(current_frontier_t[b, 0] > 0.5).float()
        pts_B = torch.nonzero(pattern_start_frontier_t[b, 0] > 0.5).float()
        if pts_A.shape[0] == 0 or pts_B.shape[0] == 0: continue
        
        ctr_A, ctr_B = pts_A.mean(dim=0), pts_B.mean(dim=0)
        total_depth = torch.dot(ctr_B - ctr_A, g).clamp_min(1.0)
        
        # 計算每個體素在生長軸上的進度 t [0, 1]
        depth_proj = ((grid - ctr_A.view(1, 1, 1, 3)) * g.view(1, 1, 1, 3)).sum(-1)
        t_vol = (depth_proj / total_depth).clamp(0.0, 1.0)
        
        field = field_all[b, 0]
        step_depth = total_depth / float(n_bins)

        # ── 5. 全局方差計算 (Global Variance Alignment) ───────────────────
        # 提取整條橋的預測矩
        pred_c_global, pred_v_global = get_moments(field.unsqueeze(0).unsqueeze(0))
        
        # 目標方差取 A/B 端中值 (或可根據需求調整)
        target_v_global = ((std_A[b] + std_B[b]) / 2.0).pow(2)
        
        # 全局方差 Loss (只看橫截面，避開長度干擾)
        loss_v_global = loss_v_global + torch.nn.functional.smooth_l1_loss(
            pred_v_global[0] * lateral_mask, 
            target_v_global.detach() * lateral_mask
        )

        # ── 6. Bin 循環 (處理局部質量與質心路徑) ─────────────────────────────
        bin_void_loss = bridge_prob_vol.new_zeros(())
        bin_path_loss = bridge_prob_vol.new_zeros(())

        for i in range(n_bins):
            t_i = (i + 0.5) / n_bins
            bin_mask = ((t_vol >= i/n_bins) & (t_vol < (i+1)/n_bins)).float()
            cos_blend = (1.0 - np.cos(np.pi * t_i)) / 2.0

            # 插值目標
            target_radius = (1.0 - cos_blend) * radius_A[b] + cos_blend * radius_B[b]
            target_vr     = (1.0 - cos_blend) * vr_A[b]     + cos_blend * vr_B[b]
            target_c      = (1.0 - cos_blend) * cA[b]        + cos_blend * cB[b]
            
            # A. 質量目標 (考慮 Empty Mask 比例)
            with torch.no_grad():
                effective_em_ratio = (empty_mask_t[b, 0] * bin_mask).sum() / (bin_mask.sum() + 1e-6)
                mass_target = (target_vr * target_radius.pow(2) * step_depth * effective_em_ratio).clamp_min(eps)
            
            mass_pred = (field * bin_mask).sum()
            bin_void_loss = bin_void_loss + torch.nn.functional.smooth_l1_loss(mass_pred, mass_target.detach())

            # B. 路徑目標 (質心對齊)
            bin_field = (field * bin_mask).unsqueeze(0).unsqueeze(0)
            if bin_field.sum() > eps:
                pred_c_bin, _ = get_moments(bin_field)
                bin_path_loss = bin_path_loss + torch.nn.functional.mse_loss(pred_c_bin[0], target_c.detach())

        loss_void_accum = loss_void_accum + bin_void_loss / n_bins
        loss_iso_accum  = loss_iso_accum  + bin_path_loss / n_bins
        valid_count += 1

    if valid_count > 0:
        # 最終 Iso Loss = 局部路徑偏差 + 全局肥瘦偏差
        # 權重建議：位置(2.0) > 肥瘦(1.0)
        final_iso = (loss_iso_accum / valid_count) * 2.0 + (loss_v_global / valid_count) * 1.0
        return (loss_void_accum / valid_count), final_iso

    return loss_void_accum, loss_iso_accum

In [35]:
#@torch.no_grad()
def decode_node_probs_to_volume_torch(
    probs,
    node_pos,
    node_valid,
    vol_shape,
):
    """
    高度優化版：使用 scatter_reduce_ 消除 for 迴圈帶來的巨量 VRAM 計算圖
    """
    B, N, _ = probs.shape
    X, Y, Z = vol_shape
    out = torch.zeros((B, 1, X, Y, Z), device=probs.device, dtype=probs.dtype)

    # 把 6 個方向的偏移量轉成 Tensor: [6, 3]
    dir_offsets = torch.tensor([
        [-1, 0, 0], [1, 0, 0],
        [0, -1, 0], [0, 1, 0],
        [0, 0, -1], [0, 0, 1]
    ], device=probs.device, dtype=torch.long)

    for b in range(B):
        valid_mask = node_valid[b] > 0.5
        if not valid_mask.any():
            continue

        pos_b = node_pos[b, valid_mask].long() # [V, 3]
        prob_b = probs[b, valid_mask]          # [V, 6]

        # 1. 利用廣播機制，一次算完所有 Node 往 6 個方向的目標座標
        # pos_b: [V, 1, 3] + dir_offsets: [1, 6, 3] -> target_pos: [V, 6, 3]
        target_pos = pos_b.unsqueeze(1) + dir_offsets.unsqueeze(0)

        tx = target_pos[..., 0] # [V, 6]
        ty = target_pos[..., 1]
        tz = target_pos[..., 2]

        # 2. 邊界檢查
        in_bounds = (
            (tx >= 0) & (tx < X) &
            (ty >= 0) & (ty < Y) &
            (tz >= 0) & (tz < Z)
        )

        # 3. 過濾出合法的座標與對應的機率
        tx = tx[in_bounds]
        ty = ty[in_bounds]
        tz = tz[in_bounds]
        p = prob_b[in_bounds]

        # 4. 將 3D 座標轉為 1D 索引，供 scatter_reduce 使用
        flat_idx = tx * Y * Z + ty * Z + tz

        # 5. 建立一個 1D 的空白畫布
        out_b_flat = torch.zeros(X * Y * Z, device=probs.device, dtype=probs.dtype)

        # 6. 使用 PyTorch 內建的高效聚合函數 (取最大值 amax)
        # 這一步直接在底層完成碰撞覆蓋，完全不產生迴圈計算圖！
        out_b_flat.scatter_reduce_(0, flat_idx, p, reduce="amax", include_self=False)

        # 7. 變回原本的 3D 形狀放回 output
        out[b, 0] = out_b_flat.view(X, Y, Z)

    return out

# %%
def soft_shell_from_frontier_torch(frontier_t, empty_mask, dilate_iters=1):
    """
    從當前 frontier 出發，只允許 proposal 落在 frontier 可達 shell。
    保留 rule：bridge 只能由 frontier 向外長，不可在內部或遠處跳生。
    frontier_t: [B,1,X,Y,Z]
    empty_mask: [B,1,X,Y,Z]
    """
    frontier_t = (frontier_t > 0.5).float()
    shell = frontier_t.clone()

    if dilate_iters > 0:
        for _ in range(int(dilate_iters)):
            shell = F.max_pool3d(shell, kernel_size=3, stride=1, padding=1)

    shell = shell * empty_mask
    return shell.clamp(0.0, 1.0)


def soft_union_torch(a, b):
    """
    differentiable soft union, 避免 hard OR。
    a,b in [0,1]
    """
    return 1.0 - (1.0 - a) * (1.0 - b)


def soft_frontier_from_field_torch(field_t, known_soft, threshold=0.10):
    """
    用 soft 場近似下一個 frontier。
    這裡不做 hard boundary extraction，只取新增 field 與空白交界的高值區。
    """
    field_t = field_t.clamp(0.0, 1.0)
    known_soft = known_soft.clamp(0.0, 1.0)

    empty_soft = (1.0 - known_soft).clamp(0.0, 1.0)
    active = field_t * empty_soft

    # 鄰域支持，避免零散單點
    neigh = F.avg_pool3d(active, kernel_size=3, stride=1, padding=1)
    frontier_soft = active * (neigh > threshold).float()
    return frontier_soft.clamp(0.0, 1.0)


def bridge_potential_alignment_loss(
    bridge_field,
    bridge_target,
    empty_mask,
    shell_mask,
    eps=1e-6,
):
    """
    sparse bridge 專用：
    1) recall_loss: target 位要亮
    2) precision_loss: 唔好喺 shell 內鋪太散
    3) leak_loss: shell 外非法生長（主要作 monitor）

    返回:
        recall_loss, precision_loss, leak_loss
    """
    bridge_field = bridge_field.clamp(0.0, 1.0)
    tgt = ((bridge_target > 0.5).float() * empty_mask).clamp(0.0, 1.0)
    shell_mask = (shell_mask.clamp(0.0, 1.0) * empty_mask).clamp(0.0, 1.0)

    # ========= 1) recall on target =========
    # target 上平均 activation 越高越好
    tgt_mass = tgt.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    tgt_hit = (bridge_field * tgt).sum(dim=(1, 2, 3, 4)) / tgt_mass
    recall_loss = 1.0 - tgt_hit

    # ========= 2) precision inside valid shell =========
    # 對 sparse 任務，更重要係唔好喺 shell 內鋪太多無關 mass
    pred_mass = bridge_field.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    true_pos = (bridge_field * tgt).sum(dim=(1, 2, 3, 4))
    precision = true_pos / pred_mass
    precision_loss = 1.0 - precision

    # ========= 3) leakage outside shell =========
    outside = ((1.0 - shell_mask).clamp(0.0, 1.0) * empty_mask).clamp(0.0, 1.0)
    outside_denom = outside.sum(dim=(1, 2, 3, 4)).clamp_min(1.0)
    leak_loss = (bridge_field * outside).sum(dim=(1, 2, 3, 4)) / outside_denom

    return recall_loss.mean(), precision_loss.mean(), leak_loss.mean()

def bridge_connectivity_pull_loss(
    bridge_field,
    bridge_target,
    shell_mask,
    empty_mask=None,
    dilate_iters=2,
    decay=0.70,
    eps=1e-6,
):
    """
    local-field 一致版 connectivity pull

    唔再要求一步去到 current_frontier。
    而係要求：
      1) mass 落在 local bridge_target 周邊
      2) 唔好平均鋪滿整個 corridor shell
      3) 讓 field 更集中貼近 local target band

    bridge_field: [B,1,X,Y,Z] in [0,1]
    bridge_target: [B,1,X,Y,Z] local field target
    shell_mask:   [B,1,X,Y,Z] valid corridor region
    """
    field = bridge_field.clamp(0.0, 1.0)
    tgt = (bridge_target > 0.5).float()
    shell = shell_mask.clamp(0.0, 1.0)

    if empty_mask is not None:
        tgt = tgt * empty_mask.float()
        shell = shell * empty_mask.float()

    # local target attraction band
    attract = tgt.clone()
    band = tgt.clone()
    weight = 1.0

    for _ in range(int(dilate_iters)):
        band = F.max_pool3d(band, kernel_size=3, stride=1, padding=1)
        weight *= decay
        attract = torch.maximum(attract, band * weight)

    attract = (attract * shell).clamp(0.0, 1.0)

    pred_mass = field.sum(dim=(1, 2, 3, 4)).clamp_min(eps)

    # 1) 預測質量是否主要落在 local target/band
    attracted_mass = (field * attract).sum(dim=(1, 2, 3, 4)) / pred_mass
    attract_loss = 1.0 - attracted_mass

    # 2) 避免 field 平均鋪滿整個 shell
    shell_mass = shell.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    spread = (field * shell).sum(dim=(1, 2, 3, 4)) / shell_mass

    # 3) 與 target 重心貼近：直接鼓勵 local overlap
    tgt_mass = tgt.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    local_hit = (field * tgt).sum(dim=(1, 2, 3, 4)) / tgt_mass
    local_hit_loss = 1.0 - local_hit

    return (0.70 * attract_loss + 0.25 * local_hit_loss + 0.05 * spread).mean()

def compute_bridge_confidence_on_target(
    bridge_logits,
    bridge_tgt,
    node_valid,
    eps=1e-6,
):
    valid = node_valid.unsqueeze(-1).expand_as(bridge_tgt)
    pos_mask = (bridge_tgt > 0.5) & valid

    prob = torch.sigmoid(bridge_logits)
    conf = torch.zeros((bridge_logits.shape[0],), device=bridge_logits.device, dtype=prob.dtype)

    for b in range(bridge_logits.shape[0]):
        mask_b = pos_mask[b]
        if mask_b.any():
            conf[b] = prob[b][mask_b].mean()
        else:
            conf[b] = 0.0
    return conf.clamp(eps, 1.0)

def bridge_touch_frontier_loss(
    bridge_field,
    current_frontier,
    empty_mask=None,
    eps=1e-6,
):
    field = bridge_field.clamp(0.0, 1.0)
    frontier = (current_frontier > 0.5).float()

    if empty_mask is not None:
        frontier = frontier * empty_mask.float()

    frontier_band = F.max_pool3d(frontier, kernel_size=3, stride=1, padding=1)

    pred_mass = field.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    touch_score = (field * frontier_band).sum(dim=(1, 2, 3, 4)) / pred_mass

    return (1.0 - touch_score).mean()

def bridge_final_outcome_loss(
    bridge_final,
    bridge_target_full,
    current_frontier,
    empty_mask=None,
    eps=1e-6,
):
    """
    外層 whole-bridge loss:
    對 rollout 累積完成後的最終 bridge 結果計 loss。
    注意：
    - pred / tgt 可以乘 empty_mask
    - frontier 唔可以乘 empty_mask，因為 frontier 本身係 known 邊界實體側
    """
    pred = bridge_final.clamp(0.0, 1.0)
    tgt = (bridge_target_full > 0.5).float()
    frontier = (current_frontier > 0.5).float()

    if empty_mask is not None:
        empty_mask = empty_mask.float()
        pred = pred * empty_mask
        tgt = tgt * empty_mask
        # frontier 唔好乘 empty_mask !!!

    # final recall
    tgt_mass = tgt.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    final_recall = (pred * tgt).sum(dim=(1, 2, 3, 4)) / tgt_mass
    final_recall_loss = 1.0 - final_recall

    # final precision
    pred_mass = pred.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
    final_precision = (pred * tgt).sum(dim=(1, 2, 3, 4)) / pred_mass
    final_precision_loss = 1.0 - final_precision

    # final touch to input frontier
    # frontier 先膨脹，之後 pred 落喺 frontier 鄰近空間就算 touch 到
    frontier_band = F.max_pool3d(frontier, kernel_size=3, stride=1, padding=1)
    final_touch = (pred * frontier_band).sum(dim=(1, 2, 3, 4)) / pred_mass
    final_touch_loss = 1.0 - final_touch

    loss = (
        1.00 * final_recall_loss
        + 0.20 * final_precision_loss
        + 0.40 * final_touch_loss
    ).mean()

    stats = {
        "final_recall": final_recall.mean(),
        "final_precision": final_precision.mean(),
        "final_touch": final_touch.mean(),
    }
    return loss, stats

def grow_direction_cone_loss(prob_vol, seed_mask, grow_dir, threshold_cos=-0.25):
    B, _, X, Y, Z = prob_vol.shape
    device = prob_vol.device
    
    # 預先準備 grid (最好搬出去做類成員變量，唔好每次 loop 都重新建構)
    xs = torch.arange(X, device=device).float()
    ys = torch.arange(Y, device=device).float()
    zs = torch.arange(Z, device=device).float()
    grid_x, grid_y, grid_z = torch.meshgrid(xs, ys, zs, indexing='ij')
    grid = torch.stack([grid_x, grid_y, grid_z], dim=-1) # [X, Y, Z, 3]

    total_penalty = 0.0
    for b in range(B):
        # 將幾何遮罩嘅計算從梯度圖分離
        with torch.no_grad():
            pts = torch.nonzero(seed_mask[b, 0] > 0.5)
            if pts.shape[0] == 0: continue
            centroid = pts.float().mean(dim=0)

            rel_vec = grid - centroid.view(1, 1, 1, 3)
            dist = torch.norm(rel_vec, dim=-1, keepdim=True).clamp_min(1e-6)
            # 直接計點積，減少中間變量
            d = grow_dir[b].view(1, 1, 1, 3)
            cos_theta = (rel_vec * d).sum(dim=-1) / dist.squeeze(-1)
            penalty_mask = (cos_theta < threshold_cos).float()

        # 只有呢一步參與 BP，極大節省內存
        penalty = (prob_vol[b, 0] * penalty_mask).sum() / (penalty_mask.sum() + 1e-6)
        total_penalty += penalty

    return total_penalty / B

In [36]:
def bridge_rollout_train_phase(
    model,
    batch,
    gt_bridge_mix=0.0,
    max_bridge_steps=3,
):
    """
    nested bridge:
      inner = step-local bridge loss
      outer = final whole-bridge outcome loss
    """

    known_init = batch["known"].float()
    frontier_init   = (batch["frontier"] > 0.5).float()
    target_frontier = batch["current_frontier"].float()
    pattern_start_frontier = (batch["pattern_start_frontier"] > 0.5).float()
    grow_direction = batch["grow_direction"]
    n_shift = batch["n_shift"]

    bridge_direction = grow_direction
    empty_mask_init = (known_init <= 0.5).float()

    # rollout states (detached truncated rollout state)
    known_curr = known_init.detach().clone()
    frontier_curr = frontier_init.detach().clone()
    bridge_accum = torch.zeros_like(known_init).detach()

    # ── logging / accumulation ────────────────────────────────────────────────
    step_loss_sum           = known_init.new_zeros(())
    step_main_sum           = known_init.new_zeros(())
    step_cov_sum            = known_init.new_zeros(())
    step_recall_sum         = known_init.new_zeros(())

    step_field_recall_sum   = known_init.new_zeros(())
    step_field_precision_sum = known_init.new_zeros(())
    step_field_leak_sum     = known_init.new_zeros(())
    step_field_conn_sum     = known_init.new_zeros(())
    step_touch_sum          = known_init.new_zeros(())

    # FIX 1 ── new envelope-loss accumulators (declare BEFORE the loop)
    step_void_trans_sum     = known_init.new_zeros(())
    step_iso_trans_sum      = known_init.new_zeros(())

    final_loss_sum          = known_init.new_zeros(())
    final_recall_sum        = known_init.new_zeros(())
    final_precision_sum     = known_init.new_zeros(())
    final_touch_sum         = known_init.new_zeros(())

    steps_used = 0

    last_bridge_conf    = known_init.new_zeros((known_init.shape[0],), dtype=torch.float32)
    last_corridor_mask  = torch.zeros_like(known_init)

    frontier_band_for_touch = F.max_pool3d(frontier_init, kernel_size=3, stride=1, padding=1)

    for bt in range(int(max_bridge_steps)):
        bridge_batch = prepare_growth_batch_from_bridge_torch(
            known_after_bridge=known_curr,
            bridge_frontier_seed=frontier_curr,
            grow_direction=-bridge_direction,
            pattern_start_frontier=pattern_start_frontier,
            n_shift=n_shift,
            frontier_threshold=0.05,
            phase_mode="bridge",
        )

        out_bridge = model(
            state=bridge_batch["state"],
            node_patches=bridge_batch["node_patches"],
            node_pos=bridge_batch["node_pos"],
            node_valid=bridge_batch["node_valid"],
            grow_direction=bridge_direction,
            n_shift=bridge_batch["n_shift_norm"],
            phase_mode=bridge_batch["phase_mode"],
        )

        node_valid_bridge = bridge_batch["node_valid"] > 0.5

        # ── inner step target ────────────────────────────────────────────────
        active_bridge_target = (
            batch["bridge_target_full"].float() * (known_curr <= 0.5).float()
        ).clamp(0.0, 1.0)

        bridge_tgt = build_action_targets_from_step_target_torch(
            bridge_batch, active_bridge_target
        )

        # node-level step supervision
        bridge_main = masked_balanced_focal_bce_loss(
            out_bridge["bridge_logits"],
            bridge_tgt,
            node_valid_bridge,
            pos_weight=10.0,
            neg_weight=1.0,
            focal_gamma=2.0,
        )
        bridge_cov = masked_positive_coverage_loss(
            out_bridge["bridge_logits"], bridge_tgt, node_valid_bridge
        )
        bridge_recall = masked_positive_recall_loss(
            out_bridge["bridge_logits"], bridge_tgt, node_valid_bridge
        )

        bridge_prob_vol = decode_node_probs_to_volume_torch(
            torch.sigmoid(out_bridge["bridge_logits"]),
            bridge_batch["node_pos"],
            bridge_batch["node_valid"],
            vol_shape=known_init.shape[2:],
        )

        empty_mask_curr = (known_curr <= 0.5).float()

        # corridor is a rule mask only; no gradient needed
        with torch.no_grad():
            corridor_mask = corridor_mask_from_seed_to_frontier_torch(
                start_seed_t=frontier_curr,
                current_frontier_t=target_frontier,
                empty_mask_t=empty_mask_curr,
                grow_direction_t=bridge_direction,
                start_dilate_iters=6,
                frontier_dilate_iters=6,
                tube_radius=5.0,
                mix_keep_max=True,
            ).clamp(0.0, 1.0)

        last_corridor_mask = corridor_mask.detach()

        if corridor_mask.sum().item() <= 0:
            break

        # light attraction toward pattern side
        pattern_pull = (
            F.max_pool3d(pattern_start_frontier.float(), kernel_size=5, stride=1, padding=2)
            * empty_mask_curr
        )

        raw_bridge_field = bridge_prob_vol * (0.65 + 0.35 * pattern_pull)
        bridge_field = (raw_bridge_field * corridor_mask).clamp(0.0, 1.0)

        # ── inner step loss ──────────────────────────────────────────────────
        # FIX 2 ── correct indentation (8 spaces) + correct variable alignment
        bridge_void_trans, bridge_iso_trans = bridge_dynamic_envelope_loss(
            bridge_prob_vol=bridge_field,
            pattern_vol = batch["start_completion_target"],
            input_vol = batch["known"],
            current_frontier_t=target_frontier,
            pattern_start_frontier_t=pattern_start_frontier,
            known_t=known_curr,
            empty_mask_t=empty_mask_curr,
            grow_direction_t=grow_direction,
            n_bins=8,
        )
        step_bridge_loss = (
            bridge_main
            + 0.50 * bridge_recall
            + 0.30 * bridge_cov
            + 0.30 * bridge_void_trans
            + 0.20 * bridge_iso_trans
        )

        step_loss_sum   = step_loss_sum   + step_bridge_loss
        step_main_sum   = step_main_sum   + bridge_main
        step_cov_sum    = step_cov_sum    + bridge_cov
        step_recall_sum = step_recall_sum + bridge_recall

        # FIX 3 ── accumulate the new envelope-loss values (inside the loop)
        step_void_trans_sum = step_void_trans_sum + bridge_void_trans.detach()
        step_iso_trans_sum  = step_iso_trans_sum  + bridge_iso_trans.detach()

        # ── step field metrics (for logging) ─────────────────────────────────
        eps = 1e-6
        tgt_mass  = active_bridge_target.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
        pred_mass = bridge_field.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
        hit_mass  = (bridge_field * active_bridge_target).sum(dim=(1, 2, 3, 4))

        field_recall    = hit_mass / tgt_mass
        field_precision = hit_mass / pred_mass

        raw_mass     = raw_bridge_field.sum(dim=(1, 2, 3, 4)).clamp_min(eps)
        outside_mask = ((1.0 - corridor_mask) * empty_mask_curr).clamp(0.0, 1.0)
        field_leak   = (raw_bridge_field * outside_mask).sum(dim=(1, 2, 3, 4)) / raw_mass

        conn_loss = bridge_connectivity_pull_loss(
            bridge_field=bridge_field,
            bridge_target=active_bridge_target,
            shell_mask=corridor_mask,
            empty_mask=empty_mask_curr,
        )
        field_conn_metric = (1.0 - conn_loss.detach()).clamp(0.0, 1.0)

        touch_mass  = (bridge_field * frontier_band_for_touch).sum(dim=(1, 2, 3, 4))
        field_touch = touch_mass / pred_mass

        step_field_recall_sum    = step_field_recall_sum    + field_recall.mean()
        step_field_precision_sum = step_field_precision_sum + field_precision.mean()
        step_field_leak_sum      = step_field_leak_sum      + field_leak.mean()
        step_field_conn_sum      = step_field_conn_sum      + field_conn_metric.mean()
        step_touch_sum           = step_touch_sum           + field_touch.mean()

        # ── outer whole-bridge loss on current accumulated result ─────────────
        bridge_step = bridge_field * empty_mask_curr
        bridge_accum_for_loss = soft_union_torch(bridge_accum, bridge_step).clamp(0.0, 1.0)

        curr_final_loss, curr_final_stats = bridge_final_outcome_loss(
            bridge_final=bridge_accum_for_loss,
            bridge_target_full=batch["bridge_target_full"].float(),
            current_frontier=target_frontier,
            empty_mask=empty_mask_init,
        )

        final_loss_sum      = final_loss_sum      + curr_final_loss
        final_recall_sum    = final_recall_sum    + curr_final_stats["final_recall"]
        final_precision_sum = final_precision_sum + curr_final_stats["final_precision"]
        final_touch_sum     = final_touch_sum     + curr_final_stats["final_touch"]

        steps_used += 1

        # ── rollout state update (DETACHED) ───────────────────────────────────
        with torch.no_grad():
            bridge_step_det = bridge_step.detach()

            bridge_accum = soft_union_torch(bridge_accum, bridge_step_det).clamp(0.0, 1.0).detach()
            known_curr   = soft_union_torch(known_curr,   bridge_step_det).clamp(0.0, 1.0).detach()

            dilated = _dilate3d_binary_torch((bridge_step_det > 0.10).float(), iters=2)
            new_frontier = (dilated * (known_curr <= 0.5).float()).clamp(0.0, 1.0).detach()

            has_active = (new_frontier > 0.5).float().flatten(1).sum(dim=1)
            for b in range(new_frontier.shape[0]):
                if has_active[b] < 1.0:
                    new_frontier[b] = frontier_curr[b]

            frontier_curr = new_frontier

        last_bridge_conf = compute_bridge_confidence_on_target(
            out_bridge["bridge_logits"],
            bridge_tgt,
            node_valid_bridge,
        )

    # ── finalize ──────────────────────────────────────────────────────────────
    if steps_used == 0:
        bridge_loss = known_init.new_zeros((), requires_grad=True)
        metric_final_loss, metric_final_stats = bridge_final_outcome_loss(
            bridge_final=bridge_accum,
            bridge_target_full=batch["bridge_target_full"].float(),
            current_frontier=frontier_init,
            empty_mask=empty_mask_init,
        )

        zero = known_init.new_zeros(())
        # FIX 4a ── add the two new keys to the early-exit return dict
        return {
            "known_after_bridge":    known_curr.detach(),
            "bridge_frontier_seed":  frontier_curr.detach(),
            "bridge_field":          bridge_accum.detach(),
            "bridge_shell_mask":     last_corridor_mask.detach(),
            "bridge_loss":           bridge_loss,

            "bridge_main":           zero,
            "bridge_cov":            zero,
            "bridge_recall":         zero,
            "bridge_field_recall":   zero,
            "bridge_field_precision": zero,
            "bridge_field_leak":     zero,
            "bridge_field_conn":     zero,
            "bridge_touch_frontier": zero,

            "bridge_void_trans":     zero,   # ← FIX 4a
            "bridge_iso_trans":      zero,   # ← FIX 4a

            "bridge_final_loss":      metric_final_loss.detach(),
            "bridge_final_recall":    metric_final_stats["final_recall"].detach(),
            "bridge_final_precision": metric_final_stats["final_precision"].detach(),
            "bridge_final_touch":     metric_final_stats["final_touch"].detach(),

            "bridge_conf": last_bridge_conf,
        }

    bridge_loss = (
        0.60 * (step_loss_sum / steps_used)
        + 1.00 * (final_loss_sum / steps_used)
    )

    # metric-only final readout
    with torch.no_grad():
        metric_final_loss, metric_final_stats = bridge_final_outcome_loss(
            bridge_final=bridge_accum,
            bridge_target_full=batch["bridge_target_full"].float(),
            current_frontier=target_frontier,
            empty_mask=empty_mask_init,
        )

    # FIX 4b ── add the two new keys to the normal return dict
    return {
        "known_after_bridge":    known_curr.detach(),
        "bridge_frontier_seed":  frontier_curr.detach(),
        "bridge_field":          bridge_accum.detach(),
        "bridge_shell_mask":     last_corridor_mask.detach(),
        "bridge_loss":           bridge_loss,

        "bridge_main":           (step_main_sum   / steps_used).detach(),
        "bridge_cov":            (step_cov_sum     / steps_used).detach(),
        "bridge_recall":         (step_recall_sum  / steps_used).detach(),
        "bridge_field_recall":   (step_field_recall_sum    / steps_used).detach(),
        "bridge_field_precision": (step_field_precision_sum / steps_used).detach(),
        "bridge_field_leak":     (step_field_leak_sum  / steps_used).detach(),
        "bridge_field_conn":     (step_field_conn_sum  / steps_used).detach(),
        "bridge_touch_frontier": (step_touch_sum        / steps_used).detach(),

        "bridge_void_trans":     (step_void_trans_sum / steps_used).detach(),  # ← FIX 4b
        "bridge_iso_trans":      (step_iso_trans_sum  / steps_used).detach(),  # ← FIX 4b

        "bridge_final_loss":      metric_final_loss.detach(),
        "bridge_final_recall":    metric_final_stats["final_recall"].detach(),
        "bridge_final_precision": metric_final_stats["final_precision"].detach(),
        "bridge_final_touch":     metric_final_stats["final_touch"].detach(),

        "bridge_conf": last_bridge_conf,
    }

Training

In [37]:
# %%
def run_one_epoch(
    model,
    dataloader,
    optimizer=None,
    train=True,
    epoch=1,
    max_steps=None,
    scaler=None,
    pattern_encoder=None,  # Add RAG encoder
    rag_optimizer=None,     # Add RAG optimizer
    rag_scaler=None,        # Add RAG scaler
):
    if train:
        model.train()
        if pattern_encoder is not None and RAG_ENABLE_TRAINING:
            pattern_encoder.train()
    else:
        model.eval()
        if pattern_encoder is not None:
            pattern_encoder.eval()
    logs_sum = None
    n_steps = 0
    gt_bridge_mix = 0.7 if train else 1.0
    for step, raw_batch in enumerate(dataloader):
        if max_steps is not None and step >= max_steps:
            break
        batch = prepare_training_batch_on_device(raw_batch)
        if train:
            optimizer.zero_grad(set_to_none=True)
            if rag_optimizer is not None and RAG_ENABLE_TRAINING:
                rag_optimizer.zero_grad(set_to_none=True)
        # =========================================
        # RAG: Update pattern bank embeddings
        # =========================================
        # Add this constant near your other config
        BANK_UPDATE_INTERVAL = 100  # update every N steps
        bank_index = PATTERN_BANK_INDEX
        
        # Then replace the per-step update with:
        global BANK_EMBEDDINGS
        if pattern_encoder is not None and (BANK_EMBEDDINGS is None or step % BANK_UPDATE_INTERVAL == 0):
            BANK_EMBEDDINGS = update_pattern_bank_embeddings(
                PATTERN_BANK, pattern_encoder, device
            )
        with torch.set_grad_enabled(train):
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                
                # =========================================
                # RAG: Retrieve top-k patterns
                # =========================================
                rag_loss = torch.tensor(0.0, device=device)

                if train and RAG_ENABLE_TRAINING and pattern_encoder is not None:
                    # ── 1. All CPU extractions done ONCE, outside any loop ──────────────
                    B = batch["start_frontier_seed"].shape[0]
                    frontier_seeds_np = batch["start_frontier_seed"][:, 0].cpu().numpy().astype(np.uint8)  # [B,X,Y,Z]
                    preview_knowns_np = batch["preview_known"][:, 0].cpu().numpy().astype(np.uint8)         # [B,X,Y,Z]
                    grow_dir_np       = batch["grow_direction"][0].cpu().numpy()
                
                    # ── 2. Retrieval (only when actually training) ───────────────────────
                    topk_items = retrieve_topk_neural(
                        current_frontier_u8=frontier_seeds_np[0],
                        grow_direction_np=grow_dir_np,
                        pattern_bank=PATTERN_BANK,
                        bank_embeddings_t=BANK_EMBEDDINGS,
                        encoder=pattern_encoder,
                        device=device,
                        top_k=PATTERNS_PER_SCENE,
                    )
                
                    # ── 3. Canonicalize all volumes in one pass ──────────────────────────
                    query_vols    = []
                    positive_vols = []
                    positive_indices = []
                
                    # Precomputed O(1) lookup: bytes → bank index
                
                    for b in range(B):
                        # Query
                        q_local, _ = canonicalize_mask_to_center(frontier_seeds_np[b])
                        query_vols.append(torch.from_numpy(q_local).float().to(device))
                
                        # Positive
                        p_local, _ = canonicalize_mask_to_center(preview_knowns_np[b])
                        positive_vols.append(torch.from_numpy(p_local).float().to(device))
                
                        # O(1) positive index lookup
                        best_idx = 0
                        if len(topk_items) > 0:
                            key = topk_items[0]["seed_local"].tobytes()
                            best_idx = bank_index.get(key, 0)
                        positive_indices.append(best_idx)
                
                    # ── 4. Negatives ─────────────────────────────────────────────────────
                    negative_vols = get_negative_pattern_samples(
                        PATTERN_BANK, positive_indices, num_negatives=8, device=device
                    )
                    negative_vols_flat = [vol for sublist in negative_vols for vol in sublist]
                    K_neg = len(negative_vols[0]) if len(negative_vols) > 0 else 0
                
                    # ── 5. Single encoder forward pass for all three sets ────────────────
                    all_vols = query_vols + positive_vols + negative_vols_flat
                    all_emb  = pattern_encoder(all_vols)  # [B + B + B*K_neg, d_model]
                
                    query_emb        = all_emb[:B]
                    positive_emb     = all_emb[B : 2 * B]
                    negative_emb_flat = all_emb[2 * B:]
                    negative_emb     = negative_emb_flat.view(B, K_neg, -1)
                
                    # ── 6. Contrastive loss ───────────────────────────────────────────────
                    rag_loss = rag_contrastive_loss(
                        query_emb, positive_emb, negative_emb, temperature=RAG_TEMPERATURE
                    )
                # =========================
                # Phase A: bridge rollout first
                # =========================
                max_bridge_steps = min(int(batch["n_shift"].max().item()) + 5, 5)
                bridge_info = bridge_rollout_train_phase(
                    model=model,
                    batch=batch,
                    gt_bridge_mix=0.0,
                    max_bridge_steps=max_bridge_steps,
                )
                known_after_bridge = bridge_info["known_after_bridge"]
                bridge_frontier_seed = bridge_info["bridge_frontier_seed"]
                
                # =========================
                # Phase B: grow on pattern start frontier
                # =========================
                grow_batch = prepare_growth_batch_from_bridge_torch(
                    known_after_bridge=known_after_bridge,
                    bridge_frontier_seed=bridge_frontier_seed,
                    grow_direction=batch["grow_direction"],
                    pattern_start_frontier=batch["pattern_start_frontier"],
                    n_shift=batch["n_shift"],
                    frontier_threshold=0.05, 
                    phase_mode = "grow"
                )
                out_grow = model(
                    state=grow_batch["state"],
                    node_patches=grow_batch["node_patches"],
                    node_pos=grow_batch["node_pos"],
                    node_valid=grow_batch["node_valid"],
                    grow_direction=grow_batch["grow_direction"],
                    n_shift=grow_batch["n_shift_norm"],
                    phase_mode = grow_batch["phase_mode"]
                )
                grow_tgt = build_action_targets_from_step_target_torch(
                    grow_batch, batch["preview_known"]
                )
                node_valid_grow = grow_batch["node_valid"] > 0.5
                grow_main = masked_balanced_focal_bce_loss(
                    out_grow["grow_logits"], grow_tgt, node_valid_grow,
                    pos_weight=8.0, neg_weight=1.0, focal_gamma=2.0,
                )
                grow_cov = masked_positive_coverage_loss(out_grow["grow_logits"], grow_tgt, node_valid_grow)
                grow_recall = masked_positive_recall_loss(out_grow["grow_logits"], grow_tgt, node_valid_grow)
                # ----------------------------------------------------
                # Advanced grow losses
                # ----------------------------------------------------
                
                grow_dice = masked_soft_dice_loss(out_grow["grow_logits"], grow_tgt, node_valid_grow)
                grow_prob_vol = decode_node_probs_to_volume_torch(
                    torch.sigmoid(out_grow["grow_logits"]),
                    grow_batch["node_pos"],
                    grow_batch["node_valid"],
                    vol_shape=batch["known"].shape[2:],
                )
                empty_mask_grow = (known_after_bridge <= 0.5).float()
                grow_target_vol =  batch["preview_known"].float()
                grow_leakage      = bbox_leakage_penalty(grow_prob_vol, grow_target_vol, empty_mask_grow)
                grow_void_ratio   = differentiable_void_ratio_loss(grow_prob_vol, grow_target_vol)
                grow_spatial_spread = differentiable_spatial_spread_loss(grow_prob_vol, grow_target_vol)
                grow_smooth = grow_smooth_loss(grow_prob_vol, empty_mask_grow)
                grow_continuity = grow_continuity_loss(grow_prob_vol, empty_mask_grow)
                grow_dir_penalty = grow_direction_cone_loss(
                    grow_prob_vol, 
                    grow_batch["frontier"],
                    grow_batch["grow_direction"]
                )
                # Combined grow loss
                grow_loss_raw = (
                    1.00 * grow_main +
                    0.50 * grow_dice +
                    0.25 * grow_recall +
                    0.50 * grow_leakage +
                    0.40 * grow_void_ratio +
                    0.40 * grow_spatial_spread +
                    0.20 * grow_smooth +
                    0.30 * grow_continuity +
                    0.10 * grow_dir_penalty
                )
                bridge_conf = bridge_info["bridge_conf"]
                grow_weight = (1.0 + 1.5 * (1.0 - bridge_conf)).detach()
                grow_loss = grow_loss_raw * grow_weight.mean()
                bridge_loss = bridge_info["bridge_loss"]
                
                # =========================================
                # TOTAL LOSS WITH RAG
                # =========================================
                # Weight RAG loss appropriately
                rag_weight = 0.1 if (train and RAG_ENABLE_TRAINING) else 0.0
                total_loss = bridge_loss + grow_loss + rag_weight * rag_loss
            if train:
                # Backward pass for main model
                if scaler is not None:
                    # 1. 使用主 scaler 放大總 loss 並計算所有梯度
                    scaler.scale(total_loss).backward()
                    
                    # 2. 使用主 scaler 更新主模型的 optimizer
                    scaler.step(optimizer)
                    
                    # 3. 使用「同一個主 scaler」更新 RAG 的 optimizer
                    if rag_optimizer is not None and RAG_ENABLE_TRAINING and rag_loss > 0:
                        scaler.step(rag_optimizer)
                        
                    # 4. 等所有 optimizer 都 step 完之後，統一更新一次 scaler
                    scaler.update()
                else:
                    # 如果沒有使用 AMP 的備用方案
                    total_loss.backward()
                    optimizer.step()
                    if rag_optimizer is not None and RAG_ENABLE_TRAINING and rag_loss > 0:
                        rag_optimizer.step()
        logs = {
            "loss":               float(total_loss.detach().item()),
            "rag_loss":           float(rag_loss.detach().item()) if isinstance(rag_loss, torch.Tensor) else 0.0,
            "grow_loss":          float(grow_loss.detach().item()),
            "bridge_loss":        float(bridge_loss.detach().item()),
            "grow_main":          float(grow_main.detach().item()),
            "grow_dice":          float(grow_dice.detach().item()),
            "grow_leakage":       float(grow_leakage.detach().item()),
            "grow_void":          float(grow_void_ratio.detach().item()),
            "grow_spread":        float(grow_spatial_spread.detach().item()),
            "grow_smooth":        float(grow_smooth.detach().item()),
            "grow_continuity":    float(grow_continuity.detach().item()),
            "grow_recall":        float(grow_recall.detach().item()),
            "grow_dir_penal":     float(grow_dir_penalty),
            "bridge_main":        float(bridge_info["bridge_main"].detach().item()),
            "bridge_cov":         float(bridge_info["bridge_cov"].detach().item()),
            "bridge_recall":      float(bridge_info["bridge_recall"].detach().item()),
            "bridge_void_trans":  float(bridge_info["bridge_void_trans"].detach().item()),
            "bridge_iso_trans":   float(bridge_info["bridge_iso_trans"].detach().item()),
            "bridge_conf":        float(bridge_conf.mean().detach().item()),
            "grow_weight":        float(grow_weight.mean().detach().item()),
            "valid_nodes_bridge": float(batch["node_valid"].sum().item()),
            "valid_nodes_grow":   float(grow_batch["node_valid"].sum().item()),
            "bridge_final_loss":       float(bridge_info["bridge_final_loss"].detach().item()),
            "bridge_final_recall":     float(bridge_info["bridge_final_recall"].detach().item()),
            "bridge_final_precision":  float(bridge_info["bridge_final_precision"].detach().item()),
            "bridge_final_touch":      float(bridge_info["bridge_final_touch"].detach().item()),
        }
        if logs_sum is None:
            logs_sum = {k: 0.0 for k in logs.keys()}
        for k, v in logs.items():
            logs_sum[k] += float(v)
        n_steps += 1
    if n_steps == 0:
        return {
            "loss": 0.0,
            "rag_loss": 0.0,
            "grow_loss": 0.0,
            "bridge_loss": 0.0,
            "grow_main": 0.0,
            "grow_cov": 0.0,
            "grow_recall": 0.0,
            "grow_smooth": 0.0,
            "grow_connectivity": 0.0,
            "bridge_main": 0.0,
            "bridge_cov": 0.0,
            "bridge_recall": 0.0,
            "bridge_conf": 0.0,
            "grow_weight": 0.0,
            "valid_nodes_bridge": 0.0,
            "valid_nodes_grow": 0.0,
        }
    return {k: v / n_steps for k, v in logs_sum.items()}

In [38]:
def _filter_state_dict_for_model(model, loaded_state):
    model_state = model.state_dict()
    matched = {}
    skipped = {}

    for k, v in loaded_state.items():
        if k not in model_state:
            skipped[k] = "unexpected_key"
            continue
        if tuple(v.shape) != tuple(model_state[k].shape):
            skipped[k] = f"shape_mismatch loaded={tuple(v.shape)} current={tuple(model_state[k].shape)}"
            continue
        matched[k] = v

    missing = [k for k in model_state.keys() if k not in matched]
    return matched, skipped, missing

def load_ckpt(path, model, optimizer=None, map_location=device, strict_arch=False, verbose=True):
    ckpt = torch.load(path, map_location=map_location)
    loaded_state = ckpt["model"]

    matched_state, skipped, missing = _filter_state_dict_for_model(model, loaded_state)

    current_state = model.state_dict()
    current_state.update(matched_state)
    model.load_state_dict(current_state, strict=False)

    arch_compatible = (len(skipped) == 0 and len(missing) == 0)

    optimizer_loaded = False
    if optimizer is not None and ckpt.get("optimizer") is not None and arch_compatible:
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
            optimizer_loaded = True
        except Exception:
            optimizer_loaded = False

    if strict_arch and (not arch_compatible):
        raise RuntimeError(
            f"Incompatible checkpoint: {path} | skipped={len(skipped)} | missing={len(missing)}"
        )

    if verbose:
        print(f"[load_ckpt] path={path}")
        print(f"[load_ckpt] matched params: {len(matched_state)} / {len(current_state)}")
        print(f"[load_ckpt] skipped params: {len(skipped)}")
        print(f"[load_ckpt] missing params: {len(missing)}")
        print(f"[load_ckpt] optimizer_loaded: {optimizer_loaded}")

    ckpt["_arch_compatible"] = arch_compatible
    ckpt["_optimizer_loaded"] = optimizer_loaded
    ckpt["_matched_param_count"] = len(matched_state)
    ckpt["_skipped_param_count"] = len(skipped)
    ckpt["_missing_param_count"] = len(missing)
    return ckpt


In [39]:
# ===== checkpoint helpers =====
OUT_DIR = os.path.join(ROOT_DIR, "FRONTIER_EDGE_STATE_RUN")
CKPT_DIR = os.path.join(OUT_DIR, "ckpt")
os.makedirs(CKPT_DIR, exist_ok=True)

print("OUT_DIR:", OUT_DIR)
print("CKPT_DIR:", CKPT_DIR)

# ===== resume if exists =====
RESUME_PATH = os.path.join(CKPT_DIR, "last.pt")

if os.path.isfile(RESUME_PATH):
    ckpt = load_ckpt(
        RESUME_PATH,
        model=model,
        map_location=device,
        strict_arch=False,
        verbose=True,
    )

    if ckpt.get("_arch_compatible", False):
        start_epoch = int(ckpt["epoch"]) + 1
        history = ckpt.get("history", [])
        best_val = ckpt.get("best_val")
        if best_val is None:
            best_val = float("inf")
        print("resumed full state from:", RESUME_PATH)
        print("start_epoch:", start_epoch)
        print("best_val:", best_val)
    else:
        # 架构变了：只当作 partial pretrained init
        start_epoch = 1
        history = []
        best_val = ckpt.get("best_val")
        if best_val is None:
            best_val = float("inf")
        print("partial weights loaded from incompatible checkpoint:", RESUME_PATH)
        print("optimizer/state/history reset for new architecture")
else:
    start_epoch = 1
    history = []
    best_val = float("inf")
    print("no checkpoint found, start from scratch")

OUT_DIR: .\FRONTIER_EDGE_STATE_RUN
CKPT_DIR: .\FRONTIER_EDGE_STATE_RUN\ckpt


C:\Users\Coder\AppData\Local\Temp\ipykernel_19136\1975790592.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=map_location)


[load_ckpt] path=.\FRONTIER_EDGE_STATE_RUN\ckpt\last.pt
[load_ckpt] matched params: 98 / 98
[load_ckpt] skipped params: 0
[load_ckpt] missing params: 0
[load_ckpt] optimizer_loaded: False
resumed full state from: .\FRONTIER_EDGE_STATE_RUN\ckpt\last.pt
start_epoch: 121
best_val: 1553.415137688319


In [ ]:
import time
import datetime
SAVE_EVERY = 1
EVAL_EVERY = 1
os.makedirs(OUT_DIR, exist_ok=True)
LOG_TXT_PATH = os.path.join(OUT_DIR, "train_log_01.txt")

steps_per_epoch = len(train_dl)
total_train_steps = EPOCHS * steps_per_epoch
global_step = (start_epoch - 1) * steps_per_epoch
start_time = time.time()

for epoch in range(start_epoch, EPOCHS + 1):
    train_logs = run_one_epoch(
        model,
        train_dl,
        optimizer=optimizer,
        train=True,
        epoch=epoch,
        scaler=scaler,
        pattern_encoder=pattern_encoder,    # Add this
        rag_optimizer=RAG_optimizer,         # Add this
        rag_scaler=RAG_scaler,              # Add this
    )
    val_logs = run_one_epoch(
        model,
        val_dl,
        optimizer=None,
        train=False,
        epoch=epoch,
        scaler=None,
        pattern_encoder=pattern_encoder,    # Add this
        rag_optimizer=None,
        rag_scaler=None,
    )
    # ----------------------------------------------------
    # 🌟 新增：Save Best Model 邏輯
    # ----------------------------------------------------
    current_val_metric = val_logs['bridge_loss']  # 也可以改成監控 val_logs['bridge_loss']
    
    is_best = current_val_metric < best_val
    if is_best:
        best_val = current_val_metric
        print(f"🏆 發現最佳模型！Validation 降至: {best_val:.4f}，正在儲存 best_model.pt ...")
        
        best_ckpt = {
            "epoch": epoch,
            "model": model.state_dict(),
            "best_val": best_val,
        }
        if pattern_encoder is not None:
            best_ckpt["pattern_encoder"] = pattern_encoder.state_dict()
            
        torch.save(best_ckpt, os.path.join(CKPT_DIR, "best_model.pt"))
    
    # 原本的 last.pt 還是每回合存一下，確保隨時可以中斷恢復
    last_ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer else None,
        "best_val": best_val,
    }
    if pattern_encoder is not None:
        last_ckpt["pattern_encoder"] = pattern_encoder.state_dict()
    torch.save(last_ckpt, os.path.join(CKPT_DIR, "last.pt"))
    # ----------------------------------------------------


    print(
        f"[{datetime.datetime.now()}][Epoch {epoch:03d}] "
        f"train loss={train_logs['loss']:.4f} "
        f"(grow={train_logs['grow_loss']:.4f}, bridge={train_logs['bridge_loss']:.4f};dir_penal = {train_logs["grow_dir_penal"]:.4f} "        f"nodes(b={train_logs['valid_nodes_bridge']:.1f}, g={train_logs['valid_nodes_grow']:.1f}) | "
        f"\nGrow (grow_dice = {train_logs["grow_dice"]}, grow_leakage = {train_logs["grow_leakage"]}, grow_void = {train_logs["grow_void"]}, grow_spread = {train_logs["grow_spread"]}，grow_smooth = {train_logs["grow_smooth"]}，grow_continuity = {train_logs["grow_continuity"]})"
        f"\nBridge_final = {train_logs["bridge_final_loss"]}(bridge_final_recall={train_logs["bridge_final_recall"]:.4f},bridge_final_precision={train_logs["bridge_final_precision"]:.4f},bridge_final_touch={train_logs["bridge_final_touch"]:.4f})|"
        f"\n Bridge_Space (bridge_void_trans ={train_logs["bridge_void_trans"]} , bridge_iso_trans ={train_logs["bridge_iso_trans"]})"
        f"\n RAG_Loss = {train_logs["rag_loss"]}"
        f"\nval loss={val_logs['loss']:.4f} "
        f"(grow={val_logs['grow_loss']:.4f}, bridge={val_logs['bridge_loss']:.4f}; "
        f"\nGrow (grow_dice = {val_logs["grow_dice"]}, grow_leakage = {val_logs["grow_leakage"]}, grow_void = {val_logs["grow_void"]}, grow_spread = {val_logs["grow_spread"]}grow_smooth = {val_logs["grow_smooth"]}，grow_continuity = {val_logs["grow_continuity"]})"
        f"bridge_main={val_logs['bridge_main']:.4f}, bridge_cov={val_logs['bridge_cov']:.4f}, bridge_recall={val_logs['bridge_recall']:.4f}; "
        f"bridge_conf={val_logs['bridge_conf']:.4f}, grow_weight={val_logs['grow_weight']:.4f}) | "
        f"nodes(b={val_logs['valid_nodes_bridge']:.1f}, g={val_logs['valid_nodes_grow']:.1f}) | "
        f"\nBridge_final = {val_logs["bridge_final_loss"]}(bridge_final_recall={val_logs["bridge_final_recall"]:.4f},bridge_final_precision={val_logs["bridge_final_precision"]:.4f},bridge_final_touch={val_logs["bridge_final_touch"]:.4f})|"
        f"\n RAG_Loss = {val_logs["rag_loss"]}"
    )

    log_entry = (
        f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}][Epoch {epoch:03d}]\n"
        f"TRAIN | loss: {train_logs['loss']:.4f} (grow: {train_logs['grow_loss']:.4f}, bridge: {train_logs['bridge_loss']:.4f}, dir_penal: {train_logs['grow_dir_penal']:.4f})\n"
        f"      | Grow: dice={train_logs['grow_dice']:.4f}, leak={train_logs['grow_leakage']:.4f}, void={train_logs['grow_void']:.4f}, spread={train_logs['grow_spread']:.4f}, smooth={train_logs['grow_smooth']:.4f}, cont={train_logs['grow_continuity']:.4f}\n"
        f"      | Space: void_tr={train_logs['bridge_void_trans']:.4f}, iso_tr={train_logs['bridge_iso_trans']:.4f}\n"
        f"      | Final: loss={train_logs['bridge_final_loss']:.4f} (R:{train_logs['bridge_final_recall']:.4f}, P:{train_logs['bridge_final_precision']:.4f}, T:{train_logs['bridge_final_touch']:.4f})\n"
        f"VAL   | loss: {val_logs['loss']:.4f} (grow: {val_logs['grow_loss']:.4f}, bridge: {val_logs['bridge_loss']:.4f})\n"
        f"      | Grow: dice={val_logs['grow_dice']:.4f}, leak={val_logs['grow_leakage']:.4f}, void={val_logs['grow_void']:.4f}, spread={val_logs['grow_spread']:.4f}\n"
        f"      | Bridge: main={val_logs['bridge_main']:.4f}, cov={val_logs['bridge_cov']:.4f}, recall={val_logs['bridge_recall']:.4f}, conf={val_logs['bridge_conf']:.4f}\n"
        f"      | Nodes: Bridge={val_logs['valid_nodes_bridge']:.1f}, Grow={val_logs['valid_nodes_grow']:.1f}\n"
        f"      | Nodes: RAG_Loss_Train = {train_logs["rag_loss"]}RAG_Loss_VAl = {val_logs["rag_loss"]}"
        f"{'-'*80}\n"
    )
    
    # 使用 'a' (append) 模式開啟，並指定 utf-8 編碼避免亂碼
    with open(LOG_TXT_PATH, "a", encoding="utf-8") as f:
        f.write(log_entry)
    
    

/tmp/ipykernel_318336/3649165310.py:157: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at ../aten/src/ATen/native/Cross.cpp:62.)
  u = torch.cross(axis, ref)


🏆 發現最佳模型！Validation 降至: 61.7174，正在儲存 best_model.pt ...
[2026-04-02 05:49:52.904218][Epoch 001] train loss=14.6426 (grow=2.6611, bridge=11.8224;dir_penal = 0.0007 nodes(b=216.0, g=1437.3) | 
Grow (grow_dice = 0.9262704966224242, grow_leakage = 0.17517358148209403, grow_void = 0.0238210712080804, grow_spread = 0.1496599318631533，grow_smooth = 0.000529159955981328，grow_continuity = 0.8050951493956219)
Bridge_final = 1.0295637493657175(bridge_final_recall=0.5112,bridge_final_precision=0.2375,bridge_final_touch=0.0294)|
 Bridge_Space (bridge_void_trans =43.657823734751375 , bridge_iso_trans =18.168876588901625)
 RAG_Loss = 1.591259283161609
val loss=64.7616 (grow=3.0442, bridge=61.7174; 
Grow (grow_dice = 0.8701183870434761, grow_leakage = 0.027686556180318195, grow_void = 0.004714369773864746, grow_spread = 0.6099907544751962grow_smooth = 0.0017805421327163156，grow_continuity = 0.7735989515980085)bridge_main=1.4265, bridge_cov=0.1423, bridge_recall=0.5984; bridge_conf=0.0915, grow_weight=2

In [40]:
# %% [markdown]
# # Rewritten Inference Helpers

# %%
import os
import numpy as np
import torch
import torch.nn.functional as F
import scipy.ndimage as ndi


def _vol_np_to_torch(arr, device=device):
    arr = np.asarray(arr)
    return torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(0).to(device)


def _vec_np_to_torch(arr, device=device):
    arr = np.asarray(arr, dtype=np.float32)
    return torch.from_numpy(arr).float().unsqueeze(0).to(device)


def _to_np_u8(x):
    if torch.is_tensor(x):
        x = x.detach().float().cpu().numpy()
    x = np.asarray(x)
    if x.ndim == 5:
        x = x[0, 0]
    elif x.ndim == 4:
        x = x[0]
    return (x > 0.5).astype(np.uint8)


@torch.no_grad()
def prepare_infer_batch_from_sample(sample, device=device):
    pattern_start_mask = sample["pattern_start_mask"] if "pattern_start_mask" in sample else sample["start_completion_target"]

    out = {
        "known": _vol_np_to_torch(sample["initial_known"], device),
        "current_frontier": _vol_np_to_torch(sample["current_frontier"], device),
        "start_frontier_seed": _vol_np_to_torch(sample["start_frontier_seed"], device),
        "pattern_start_frontier": _vol_np_to_torch(sample["pattern_start_frontier"], device),
        "preview_known": _vol_np_to_torch(sample["preview_known"], device),
        "pattern_start_mask": _vol_np_to_torch(pattern_start_mask, device),
        "start_completion_target": _vol_np_to_torch(sample["start_completion_target"], device),
        "bridge_target": _vol_np_to_torch(sample["bridge_target"], device),
        "bridge_target_full": _vol_np_to_torch(sample["bridge_target_full"], device),
        "grow_direction": _vec_np_to_torch(sample["grow_direction"], device),
        "grow_direction_valid": torch.tensor(
            [float(sample.get("grow_direction_valid", 1.0))],
            dtype=torch.float32,
            device=device,
        ),
        "n_shift": torch.tensor(
            [float(sample["n_shift"])],
            dtype=torch.float32,
            device=device,
        ),
        "raw_sample": sample,
    }
    return out


def binary_metrics(pred_u8, gt_u8):
    pred = (np.asarray(pred_u8) > 0).astype(np.uint8)
    gt = (np.asarray(gt_u8) > 0).astype(np.uint8)

    tp = int(((pred > 0) & (gt > 0)).sum())
    fp = int(((pred > 0) & (gt == 0)).sum())
    fn = int(((pred == 0) & (gt > 0)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    dice = 2 * tp / max(2 * tp + fp + fn, 1)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "iou": iou,
        "dice": dice,
    }


def connectivity_to_target(pred_u8, target_u8):
    pred = (np.asarray(pred_u8) > 0).astype(np.uint8)
    tgt = (np.asarray(target_u8) > 0).astype(np.uint8)
    inter = ((pred > 0) & (tgt > 0)).sum()
    tgt_mass = max(int(tgt.sum()), 1)
    return float(inter) / float(tgt_mass)


def largest_component(mask_u8):
    mask = (np.asarray(mask_u8) > 0).astype(np.uint8)
    if mask.sum() == 0:
        return mask, 0

    lab, n = ndi.label(mask, structure=ndi.generate_binary_structure(3, 1))
    if n == 0:
        return mask, 0

    counts = np.bincount(lab.ravel())
    counts[0] = 0
    idx = int(np.argmax(counts))
    out = (lab == idx).astype(np.uint8)
    return out, int(counts[idx])


def touches(a_u8, b_u8, dilate_iter=1):
    a = (np.asarray(a_u8) > 0).astype(np.uint8)
    b = (np.asarray(b_u8) > 0).astype(np.uint8)
    if a.sum() == 0 or b.sum() == 0:
        return False
    a_d = ndi.binary_dilation(
        a,
        structure=ndi.generate_binary_structure(3, 1),
        iterations=int(dilate_iter),
    ).astype(np.uint8)
    return bool(((a_d > 0) & (b > 0)).any())


def print_inference_report(result):
    bridge_m = binary_metrics(result["pred_bridge_mask"], result["gt_bridge_target"])
    grow_m = binary_metrics(result["pred_grow_mask"], result["gt_start_completion"])

    print("=== Bridge Metrics ===")
    for k, v in bridge_m.items():
        print(f"{k}: {v}")
    print("bridge_target_cover:", connectivity_to_target(result["pred_bridge_mask"], result["gt_bridge_target"]))

    print("\n=== Grow Metrics ===")
    for k, v in grow_m.items():
        print(f"{k}: {v}")
    print("grow_target_cover:", connectivity_to_target(result["pred_grow_mask"], result["gt_start_completion"]))

    print("\n=== Node Counts ===")
    print("bridge_node_count:", result["bridge_node_count"])
    print("grow_node_count:", result["grow_node_count"])

    bridge_cc, bridge_cc_size = largest_component(result["pred_bridge_mask"])
    grow_cc, grow_cc_size = largest_component(result["pred_grow_mask"])

    print("\n=== Topology Checks ===")
    print("bridge touches current_frontier:", touches(result["pred_bridge_mask"], result["current_frontier"], dilate_iter=1))
    print("bridge touches start_seed:", touches(result["pred_bridge_mask"], result["start_frontier_seed"], dilate_iter=1))
    print("grow touches bridge:", touches(result["pred_grow_mask"], result["pred_bridge_mask"], dilate_iter=1))
    print("grow touches start_seed:", touches(result["pred_grow_mask"], result["start_frontier_seed"], dilate_iter=1))
    print("largest bridge CC size:", bridge_cc_size)
    print("largest grow CC size:", grow_cc_size)


def write_points_rgb_ply(points_xyz, colors_rgb, save_path):
    print("Start Writtting PLY")
    points_xyz = np.asarray(points_xyz, dtype=np.float32)
    colors_rgb = np.asarray(colors_rgb, dtype=np.uint8)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    with open(save_path, "w", encoding="utf-8") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(points_xyz)}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("property uchar red\n")
        f.write("property uchar green\n")
        f.write("property uchar blue\n")
        f.write("end_header\n")
        for p, c in zip(points_xyz, colors_rgb):
            f.write(f"{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n")


def export_colored_ply(initial_u8, bridge_u8, grow_u8, save_path, offset=None):
    initial = (np.asarray(initial_u8) > 0).astype(np.uint8)
    bridge = (np.asarray(bridge_u8) > 0).astype(np.uint8)
    grow = (np.asarray(grow_u8) > 0).astype(np.uint8)

    # 1. 建立標籤矩陣
    label = np.zeros_like(initial, dtype=np.uint8)
    label[initial > 0] = 1
    label[bridge > 0] = 2
    label[grow > 0] = 3

    # 2. 獲取所有非零點的原始座標 (只執行一次！)
    raw_indices = np.argwhere(label > 0)
    pts = raw_indices.astype(np.float32)
    
    # 3. 獲取對應的標籤值 (一次過攞晒所有點嘅 v)
    v_values = label[raw_indices[:, 0], raw_indices[:, 1], raw_indices[:, 2]]

    # 4. 應用座標偏移
    if offset is not None:
        pts += np.asarray(offset, dtype=np.float32)

    # 5. 向量化設定顏色 (取代 slow for loop)
    colors = np.zeros((pts.shape[0], 3), dtype=np.uint8)
    colors[v_values == 1] = [200, 200, 200]  # initial
    colors[v_values == 2] = [0, 200, 200]    # bridge
    colors[v_values == 3] = [255, 80, 80]    # grow

    # 6. 寫入檔案
    write_points_rgb_ply(pts, colors, save_path)


def export_rollout_result_plys(result, out_dir="rollout_outputs", prefix="sample"):
    os.makedirs(out_dir, exist_ok=True)

    pred_path = os.path.join(out_dir, f"{prefix}_pred.ply")
    gt_path = os.path.join(out_dir, f"{prefix}_gt.ply")

    export_colored_ply(
        result["initial_known"],
        result["pred_bridge_mask"],
        result["pred_grow_mask"],
        pred_path,
    )
    export_colored_ply(
        result["initial_known"],
        result["gt_bridge_target"],
        result["gt_start_completion"],
        gt_path,
    )

    return {
        "pred_ply": pred_path,
        "gt_ply": gt_path,
    }

def print_bridge_grow_metrics(result):
    bridge_m = binary_metrics(result["pred_bridge_mask"], result["gt_bridge_target"])
    grow_m = binary_metrics(result["pred_grow_mask"], result["gt_start_completion"])

    bridge_conn = connectivity_to_target(result["pred_bridge_mask"], result["gt_bridge_target"])
    grow_conn = connectivity_to_target(result["pred_grow_mask"], result["gt_start_completion"])

    print("=== Bridge Metrics ===")
    for k, v in bridge_m.items():
        print(f"{k}: {v}")
    print("bridge_target_cover:", bridge_conn)

    print("\n=== Grow Metrics ===")
    for k, v in grow_m.items():
        print(f"{k}: {v}")
    print("grow_target_cover:", grow_conn)

    print("\n=== Node Counts ===")
    print("bridge_node_count:", result["bridge_node_count"])
    print("grow_node_count:", result["grow_node_count"])

In [41]:
# %% [markdown]
# # Rewritten Bridge Rollout Inference

# ==========================================
# 1. 神經網絡主導的 Bridge Inference
# ==========================================
@torch.no_grad()
def bridge_neural_inference(
    model,
    infer_batch,
    bridge_threshold=0.15,
    max_bridge_steps=10,
):
    model.eval()
    known_init = (infer_batch["known"] > 0.5).float().clone()
    known_curr = known_init.clone()

    # start: pattern tail seed
    frontier_curr = (infer_batch["start_frontier_seed"] > 0.5).float().clone()
    # target: existing building frontier
    target_frontier = (infer_batch["current_frontier"] > 0.5).float().clone()
    pattern_start_frontier = (infer_batch["pattern_start_frontier"] > 0.5).float().clone()

    grow_direction = infer_batch["grow_direction"].float()
    bridge_direction = grow_direction
    n_shift = infer_batch["n_shift"].float()

    bridge_accum = torch.zeros_like(known_curr)
    trace = []

    frontier_band_for_touch = F.max_pool3d(target_frontier, kernel_size=3, stride=1, padding=1)

    for step_idx in range(int(max_bridge_steps)):
        empty_mask_curr = (known_curr <= 0.5).float()

        # ── 1. prepare batch ─────────────────────────────────────────────────
        bridge_batch = prepare_growth_batch_from_bridge_torch(
            known_after_bridge=known_curr,
            bridge_frontier_seed=frontier_curr,
            grow_direction=-bridge_direction,
            pattern_start_frontier=pattern_start_frontier,
            n_shift=n_shift,
            frontier_threshold=0.05,
            phase_mode="bridge",
        )
        if bridge_batch["node_valid"].sum().item() < 1:
            break

        # ── 2. model forward ──────────────────────────────────────────────────
        out_bridge = model(
            state=bridge_batch["state"],
            node_patches=bridge_batch["node_patches"],
            node_pos=bridge_batch["node_pos"],
            node_valid=bridge_batch["node_valid"],
            grow_direction=grow_direction,
            n_shift=bridge_batch["n_shift_norm"],
            phase_mode=bridge_batch["phase_mode"],
        )

        # ── 3. decode to 3D volume ────────────────────────────────────────────
        bridge_prob_vol = decode_node_probs_to_volume_torch(
            torch.sigmoid(out_bridge["bridge_logits"]),
            bridge_batch["node_pos"],
            bridge_batch["node_valid"],
            vol_shape=known_init.shape[2:],
        )

        # ── 4. corridor mask + pattern pull (identical to training) ───────────
        corridor_mask = corridor_mask_from_seed_to_frontier_torch(
            start_seed_t=frontier_curr,
            current_frontier_t=target_frontier,
            empty_mask_t=empty_mask_curr,
            grow_direction_t=grow_direction,
            start_dilate_iters=6,
            frontier_dilate_iters=6,
            tube_radius=3.0,
            mix_keep_max=True,
        ).clamp(0.0, 1.0)

        if corridor_mask.sum().item() <= 0:
            break

        pattern_pull = (
            F.max_pool3d(pattern_start_frontier.float(), kernel_size=5, stride=1, padding=2)
            * empty_mask_curr
        )

        raw_bridge_field = bridge_prob_vol * (0.65 + 0.35 * pattern_pull)

        # corridor-masked field — used ONLY for rollout state, frontier update,
        # and touch detection. Matches exactly what training sees.
        bridge_field = (raw_bridge_field * corridor_mask).clamp(0.0, 1.0)

        # ── 5. output accumulation: raw predictions (supervised by bridge_main) ─
        # The model learned the correct cross-section shape from bridge_main loss
        # which operates on raw logits BEFORE corridor masking.
        # Do NOT apply corridor_mask here — that's what was making output always tubular.
        bridge_step_out = (bridge_prob_vol > bridge_threshold).float() * empty_mask_curr
        out_voxels = int(bridge_step_out.sum().item())

        trace.append({
            "step": step_idx,
            "new_voxels": out_voxels,
            "max_prob": float(bridge_prob_vol.max().item()),
        })

        if out_voxels <= 0:
            break

        # ── 6. accumulate output from raw predictions ─────────────────────────
        bridge_accum = soft_union_torch(bridge_accum, bridge_step_out).clamp(0.0, 1.0)

        # ── 7. rollout state update: corridor-masked soft field (matches training) ─
        # known_curr is updated from bridge_field (tube-constrained) so that the
        # next step's frontier stays well-behaved, exactly as in training.
        bridge_step_rollout = bridge_field * empty_mask_curr
        known_curr = soft_union_torch(known_curr, bridge_step_rollout).clamp(0.0, 1.0)

        # ── 8. frontier update (same threshold as training) ───────────────────
        dilated = _dilate3d_binary_torch((bridge_step_rollout > 0.10).float(), iters=2)
        new_frontier = (dilated * (known_curr <= 0.5).float()).clamp(0.0, 1.0)

        has_active = (new_frontier > 0.5).float().flatten(1).sum(dim=1)
        for b in range(new_frontier.shape[0]):
            if has_active[b] < 1.0:
                new_frontier[b] = frontier_curr[b].clone()
        frontier_curr = new_frontier

        # ── 9. early stop when rollout touches target frontier ────────────────
        if (bridge_step_rollout > 0.10).float().mul(frontier_band_for_touch).sum().item() > 0:
            break

    return {
        "known_after_bridge":   known_curr,
        "bridge_frontier_seed": frontier_curr,
        "pred_bridge_mask":     bridge_accum,   # shape from raw predictions
        "bridge_steps_used":    len(trace),
        "bridge_trace":         trace,
    }


# ==========================================
# 2. 移除 GT 依賴的 Grow Inference
# ==========================================
@torch.no_grad()
def grow_neural_inference(
    model,
    infer_batch,
    known_after_bridge,
    bridge_frontier_seed,
    grow_threshold=0.20,
    frontier_update_thresh=0.10,
    max_grow_steps=6,
    target_dilate_kernel=5,
):
    model.eval()
    known_curr = known_after_bridge.float().clone()
    frontier_curr = bridge_frontier_seed.float().clone()

    if frontier_curr.sum().item() < 1:
        frontier_curr = (infer_batch["start_frontier_seed"] > 0.5).float().clone()

    grow_direction = infer_batch["grow_direction"].float()
    n_shift = infer_batch["n_shift"].float()
    
    # 👉 【修改】讀取 preview_known 作為新的生長範圍引導
    preview_known_mask = (infer_batch["preview_known"] > 0.5).float().clone()

    grow_accum = torch.zeros_like(known_curr)
    trace = []

    for step_idx in range(int(max_grow_steps)):
        empty_mask_curr = (known_curr <= 0.5).float()

        grow_batch = prepare_growth_batch_from_bridge_torch(
            known_after_bridge=known_curr,          
            bridge_frontier_seed=frontier_curr,     
            grow_direction=grow_direction,
            pattern_start_frontier=infer_batch["pattern_start_frontier"].float().clone(), # 保持傳入 tensor
            n_shift=n_shift,
            frontier_threshold=0.05, 
            phase_mode="grow"
        )

        if grow_batch["node_valid"].sum().item() < 1:
            break

        out_grow = model(
            state=grow_batch["state"],
            node_patches=grow_batch["node_patches"],
            node_pos=grow_batch["node_pos"],
            node_valid=grow_batch["node_valid"],
            grow_direction=grow_batch["grow_direction"],
            n_shift=grow_batch["n_shift_norm"],
            phase_mode="grow", 
        )

        grow_prob_vol = decode_node_probs_to_volume_torch(
            torch.sigmoid(out_grow["grow_logits"]),
            grow_batch["node_pos"],
            grow_batch["node_valid"],
            vol_shape=known_curr.shape[2:],
        )

        # 👉 【修改】這裡的 target_band 改用 preview_known_mask 來進行膨脹引導
        target_band = F.max_pool3d(
            preview_known_mask, 
            kernel_size=target_dilate_kernel,
            stride=1,
            padding=target_dilate_kernel // 2,
        ) * empty_mask_curr

        #grow_field = (grow_prob_vol * target_band).clamp(0.0, 1.0)
        grow_field = grow_prob_vol
        grow_step = ((grow_field > grow_threshold).float() * empty_mask_curr)
        step_voxels = int(grow_step.sum().item())

        trace.append({
            "step": step_idx,
            "new_voxels": step_voxels,
            "max_prob": float(grow_field.max().item()) if grow_field.numel() > 0 else 0.0,
        })

        if step_voxels <= 0:
            break

        grow_accum = torch.maximum(grow_accum, grow_step)
        known_curr = torch.maximum(known_curr, grow_step)

        dilated = _dilate3d_binary_torch((grow_step > frontier_update_thresh).float(), iters=1)
        frontier_curr = (dilated * (known_curr <= 0.5).float()).float()

        if frontier_curr.sum().item() < 1:
            break

    return {
        "final_known": known_curr,
        "pred_grow_mask": grow_accum, # 這個依然是完美的 Delta 總和！
        "grow_steps_used": len(trace),
        "grow_trace": trace,
    }


# ==========================================
# 3. 整合主推論函數
# ==========================================
@torch.no_grad()
def run_rewritten_inference(
    model,
    sample,
    pattern_encoder=None,        # 新增
    pattern_bank=None,           # 新增
    bank_embeddings=None,        # 新增
    bridge_threshold=0.15,
    grow_threshold=0.20,
    max_bridge_steps=10,
    max_grow_steps=6,
):
    model.eval()
    if pattern_encoder is not None:
        pattern_encoder.eval()

    # 1. 準備基礎 Tensor 資料 (此時裡面還混有 Ground Truth)
    infer_batch = prepare_infer_batch_from_sample(sample, device=device)

    # ==========================================
    # 🌟 RAG 動態檢索 (取代 Ground Truth)
    # ==========================================
    if pattern_encoder is not None and pattern_bank is not None and bank_embeddings is not None:
        curr_frontier_np = _to_np_u8(infer_batch["current_frontier"])
        grow_dir_np = sample["grow_direction"]
        initial_known_np = _to_np_u8(infer_batch["known"])

        # 1. 使用神經網路檢索最適合的 Top-1 Pattern
        topk_items = retrieve_topk_neural(
            current_frontier_u8=curr_frontier_np,
            grow_direction_np=grow_dir_np,
            pattern_bank=pattern_bank,
            bank_embeddings_t=bank_embeddings,
            encoder=pattern_encoder,
            device=device,
            top_k=1
        )
        print(f"[RAG] get item len {len(topk_items)}")

        if len(topk_items) > 0:
            best_item = topk_items[0]

            # 2. 實例化 Pattern (旋轉對齊)
            pattern_start_base = instantiate_pattern_from_bank_item(
                best_item,
                ref_shape=initial_known_np.shape,
                grow_direction=grow_dir_np,
            ).astype(np.uint8)

            # 3. 計算前進推移量 (Shift)
            shift_n = compute_pattern_shift_along_direction(
                pattern_start_base,
                initial_known_np,
                grow_dir_np,
                clearance=RAG_PATTERN_CLEARANCE,
                overlap_thresh=RAG_PATTERN_OVERLAP_THRESH,
            )

            # 4. 計算側向對齊 (Lateral Offset)
            curr_coords = np.argwhere(curr_frontier_np > 0)
            target_center = curr_coords.mean(axis=0) if curr_coords.shape[0] > 0 else np.array(initial_known_np.shape) / 2.0

            temp_shifted = shift_volume_combined(pattern_start_base, shift_n, grow_dir_np)
            temp_delta = ((temp_shifted > 0) & (initial_known_np == 0)).astype(np.uint8)
            temp_seed = _select_start_frontier_seed_band_np(temp_delta, grow_dir_np, q=RAG_START_BAND_Q)

            seed_coords = np.argwhere(temp_seed > 0)
            source_center = seed_coords.mean(axis=0) if seed_coords.shape[0] > 0 else target_center

            lateral_offset = target_center - source_center
            g_unit = np.asarray(grow_dir_np, dtype=np.float64)
            g_unit = g_unit / (np.linalg.norm(g_unit) + 1e-8)
            lateral_offset = lateral_offset - np.dot(lateral_offset, g_unit) * g_unit

            # =========================================================
            # 🌟 新增：底面 (Z軸) 絕對對齊邏輯 (同步訓練時的設定)
            # =========================================================
            #取得現有建築的最低 Z 座標
            known_pts = np.argwhere(initial_known_np > 0)
            min_z_known = known_pts[:, 2].min() if known_pts.shape[0] > 0 else 0

            # 取得圖案目前的最低 Z 座標
            pattern_pts = np.argwhere(temp_shifted > 0)
            min_z_pattern = pattern_pts[:, 2].min() if pattern_pts.shape[0] > 0 else 0

            # 如果主要是「水平生長」，則強制覆寫 Z 軸偏移量，讓底面對齊
            if abs(g_unit[2]) < 0.5: 
                lateral_offset[2] = min_z_known - min_z_pattern

            # =========================================================
            # 🌟 更新：中心 (Z軸) 絕對對齊邏輯 (從 Bottom 改為 Center)
            # =========================================================
            # 1. 取得現有建築的 Z 軸中心
            # known_pts = np.argwhere(initial_known_np > 0)
            # if known_pts.shape[0] > 0:
            #     z_min_k, z_max_k = known_pts[:, 2].min(), known_pts[:, 2].max()
            #     center_z_known = (z_min_k + z_max_k) / 2.0
            # else:
            #     center_z_known = initial_known_np.shape[2] / 2.0

            # # 2. 取得 Pattern 目前的 Z 軸中心 (基於 temp_shifted)
            # pattern_pts = np.argwhere(temp_shifted > 0)
            # if pattern_pts.shape[0] > 0:
            #     z_min_p, z_max_p = pattern_pts[:, 2].min(), pattern_pts[:, 2].max()
            #     center_z_pattern = (z_min_p + z_max_p) / 2.0
            # else:
            #     center_z_pattern = initial_known_np.shape[2] / 2.0

            # # 3. 如果主要是「水平生長」，則強制覆寫 Z 軸偏移量，讓中心對齊
            # # 這裡計算的是讓兩者 Z 軸中心重合所需的位移
            # if abs(g_unit[2]) < 0.5: 
            #     lateral_offset[2] = center_z_known - center_z_pattern
            # =========================================================
            # =========================================================

            # 5. 執行最終的 3D 移動
            pattern_start_shifted = shift_volume_combined(
                pattern_start_base,
                shift_voxels=shift_n,
                grow_direction_np=grow_dir_np,
                lateral_offset_np=lateral_offset,
            )
            pattern_start_delta_shifted = ((pattern_start_shifted > 0) & (initial_known_np == 0)).astype(np.uint8)

            # 6. 擷取新的 Start Frontier Seed
            start_frontier_seed = _select_start_frontier_seed_band_np(
                pattern_start_u8=pattern_start_delta_shifted,
                grow_direction_np=grow_dir_np,
                q=RAG_START_BAND_Q,
            )

            # 反向橋接來限制 start_frontier_seed
            _, _, reverse_bridge_seed, _, _ = build_reverse_bridge_field_targets_np(
                initial_known_u8=initial_known_np,
                input_frontier_u8=curr_frontier_np,
                pattern_start_u8=pattern_start_delta_shifted,
                grow_direction_np=grow_dir_np,
                step_dilate=BRIDGE_STEP_DILATE,
            )
            start_frontier_seed = matching_start_frontier(
                start_frontier_seed,
                reverse_bridge_seed,
                grow_dir_np,
            )

            # 🌟 覆寫 infer_batch 中的 Ground Truth，改用 RAG 檢索出的結果！
            infer_batch["pattern_start_mask"] = _vol_np_to_torch(pattern_start_delta_shifted, device)
            infer_batch["pattern_start_frontier"] = _vol_np_to_torch(start_frontier_seed, device)
            infer_batch["start_frontier_seed"] = _vol_np_to_torch(start_frontier_seed, device)
            infer_batch["preview_known"] = _vol_np_to_torch(pattern_start_shifted, device)
            infer_batch["n_shift"] = torch.tensor([float(shift_n)], dtype=torch.float32, device=device)

            # 在 run_rewritten_inference 函數中更新 RAG 邏輯的部分
            # 獲取 build_reverse_bridge_field_targets_np 的所有回傳值
            (
                rev_bridge_field,      # 局部場
                rev_bridge_full,       # 🌟 這裡就是您隨機圖案產生的「真實橋接路徑」
                rev_seed, 
                _, _ 
            ) = build_reverse_bridge_field_targets_np(
                initial_known_u8=initial_known_np,
                input_frontier_u8=curr_frontier_np,
                pattern_start_u8=pattern_start_delta_shifted,
                grow_direction_np=grow_dir_np,
                step_dilate=BRIDGE_STEP_DILATE,
            )

            # 🌟 關鍵修正：將 GT Target 更新為隨機圖案計算出的結果
            infer_batch["bridge_target_full"] = _vol_np_to_torch(rev_bridge_full, device)
            infer_batch["start_completion_target"] = _vol_np_to_torch(pattern_start_delta_shifted, device)


            
    # 2. 執行神經網絡 Bridge Rollout (此時依賴的是 RAG 的目標)
    bridge_info = bridge_neural_inference(
        model=model,
        infer_batch=infer_batch,
        bridge_threshold=bridge_threshold,
        max_bridge_steps=max_bridge_steps,
    )

    # 3. 執行神經網絡 Grow Rollout (此時依賴的是 RAG 的空間引導)
    grow_info = grow_neural_inference(
        model=model,
        infer_batch=infer_batch,
        known_after_bridge=bridge_info["known_after_bridge"],
        bridge_frontier_seed=bridge_info["bridge_frontier_seed"],
        grow_threshold=grow_threshold,
        max_grow_steps=max_grow_steps,
    )

    # 4. 封裝結果
    result = {
        "initial_known": _to_np_u8(infer_batch["known"]),
        "current_frontier": _to_np_u8(infer_batch["current_frontier"]),
        "start_frontier_seed": _to_np_u8(infer_batch["start_frontier_seed"]),
        "pattern_start_frontier": _to_np_u8(infer_batch["pattern_start_frontier"]),
        "pattern_start_mask": _to_np_u8(infer_batch["pattern_start_mask"]),
        "known_after_bridge": _to_np_u8(bridge_info["known_after_bridge"]),
        "final_known": _to_np_u8(grow_info["final_known"]),

        "pred_bridge_mask": _to_np_u8(bridge_info["pred_bridge_mask"]),
        "pred_grow_mask": _to_np_u8(grow_info["pred_grow_mask"]),

        "gt_bridge_field_target": _to_np_u8(infer_batch["bridge_target"]),
        "gt_bridge_target": _to_np_u8(infer_batch["bridge_target_full"]),
        "gt_start_completion": _to_np_u8(infer_batch["start_completion_target"]),

        # "gt_bridge_target": _to_np_u8(rev_bridge_full),
        # "gt_start_completion": _to_np_u8(pattern_start_delta_shifted),

        "bridge_steps_used": bridge_info["bridge_steps_used"],
        "grow_steps_used": grow_info["grow_steps_used"],
        "bridge_trace": bridge_info["bridge_trace"],
        "grow_trace": grow_info["grow_trace"],

        "bridge_node_count": int((_to_np_u8(bridge_info["pred_bridge_mask"]) > 0).sum()),
        "grow_node_count": int((_to_np_u8(grow_info["pred_grow_mask"]) > 0).sum()),
    }

    return result

In [42]:
def export_multi_colored_ply(layers_dict, save_path):
    """
    支援傳入多個 mask 並自定義顏色。
    layers_dict 格式: { "層名稱": (mask_u8, [R, G, B]) } (RGB 數值為 0.0~1.0)
    """
    all_pts = []
    all_colors = []
    
    for name, (mask_u8, color_float) in layers_dict.items():
        mask = (np.asarray(mask_u8) > 0).astype(np.uint8)
        pts = np.argwhere(mask > 0).astype(np.float32)
        if pts.shape[0] > 0:
            c = np.array(color_float, dtype=np.float32) * 255.0
            c = c.astype(np.uint8)
            colors = np.tile(c, (pts.shape[0], 1))
            all_pts.append(pts)
            all_colors.append(colors)

    if len(all_pts) > 0:
        final_pts = np.concatenate(all_pts, axis=0)
        final_colors = np.concatenate(all_colors, axis=0)
        write_points_rgb_ply(final_pts, final_colors, save_path)
    else:
        print(f"[WARN] No points to export for {save_path}")

def export_bridge_only_ply(result, out_dir, prefix="sample"):
    export_multi_colored_ply(
        {
            "initial": (result["initial_known"], [0.80, 0.80, 0.80]),
            "current_frontier": (result["current_frontier"], [1.00, 0.40, 0.10]),
            "start_seed": (result["start_frontier_seed"], [0.10, 0.90, 0.10]),
            "pred_bridge": (result["pred_bridge_mask"], [0.00, 0.80, 0.80]),
            "gt_bridge": (result["gt_bridge_target"], [0.00, 0.20, 1.00]),
        },
        os.path.join(out_dir, f"{prefix}_bridge_only.ply"),
    )

def export_grow_only_ply(result, out_dir, prefix="sample"):
    export_multi_colored_ply(
        {
            "initial": (result["initial_known"], [0.80, 0.80, 0.80]),
            "pred_bridge": (result["pred_bridge_mask"], [0.00, 0.80, 0.80]),
            "start_seed": (result["start_frontier_seed"], [0.10, 0.90, 0.10]),
            "pred_grow": (result["pred_grow_mask"], [0.85, 0.00, 0.85]),
            "gt_grow": (result["gt_start_completion"], [1.00, 0.50, 0.90]),
        },
        os.path.join(out_dir, f"{prefix}_grow_only.ply"),
    )

def export_inference_result_plys(result, out_dir, prefix="sample"):
    os.makedirs(out_dir, exist_ok=True)
    pred_path = os.path.join(out_dir, f"{prefix}_pred.ply")
    gt_path = os.path.join(out_dir, f"{prefix}_gt.ply")

    export_colored_ply(
        result["initial_known"],
        result["pred_bridge_mask"],
        result["pred_grow_mask"],
        pred_path,
    )
    export_colored_ply(
        result["initial_known"],
        result["gt_bridge_target"],
        result["gt_start_completion"],
        gt_path,
    )
    return {"pred_ply": pred_path, "gt_ply": gt_path}

def get_ply_bbox_range(file_path, sample_size=1000):
    filesize = os.path.getsize(file_path)
    pts = []
    
    with open(file_path, 'rb') as f:
        # 跳過 header 邏輯同上...
        f.read(1024) 
        
        # 隨機跳轉到文件不同的位置讀取一小塊數據
        for _ in range(sample_size):
            pos = random.randint(1024, filesize - 128)
            f.seek(pos)
            f.readline() # 跳過可能不完整的一行
            line = f.readline().decode('utf-8', errors='ignore')
            parts = line.split()
            if len(parts) >= 3:
                pts.append([float(parts[0]), float(parts[1]), float(parts[2])])
    
    pts = np.array(pts)
    return pts.min(axis=0), pts.max(axis=0)

# 修改後的函數定義，加入 search_prefix 參數
def export_inference_result_plys_Split(result, out_dir, prefix="sample", mode="gt", i_t=False, grow_dir=None, search_prefix="manual_chain"):
    os.makedirs(out_dir, exist_ok=True)
    
    global_max_hi = np.array([-np.inf, -np.inf, -np.inf])
    found_files = False
    
    for file in os.listdir(out_dir):
        full_path = os.path.join(out_dir, file)
        if os.path.isfile(full_path):
            # 🌟 修正：用 search_prefix 搜尋目錄下所有相關嘅步數檔案
            if file.startswith(search_prefix) and file.endswith(".ply"):
                lo, hi = get_ply_bbox_range(full_path)
                if hi is not None:
                    global_max_hi = np.maximum(global_max_hi, hi)
                    found_files = True
    
    # --- B. 計算偏移量 ---
    offset_vec = np.zeros(3)
    if found_files and grow_dir is not None:
        # print 依家應該會出現喇
        print(f"計算到現有 PLY 的最大邊界: {global_max_hi}, 生長方向: {grow_dir}")
        g_dir = np.array(grow_dir)
        # 確保生長方向係單位向量
        g_dir = g_dir / (np.linalg.norm(g_dir) + 1e-8)
        
        # 💡 關鍵邏輯：搵出喺生長方向上最遠嘅投影值
        proj_max = (global_max_hi//2) @ g_dir
        # 偏移 = 方向 * (最大投影值 + 20cm 間距)
        offset_vec = g_dir * (proj_max)

    # --- C. 寫入 PLY (需修改 export_colored_ply 支援 offset) ---
    initial_path = os.path.join(out_dir, f"{prefix}_initial.ply")
    pred_path = os.path.join(out_dir, f"{prefix}_pred.ply")
    gt_path = os.path.join(out_dir, f"{prefix}_gt.ply")

    # 注意：這裡假設你已經修改了 export_colored_ply 以接受 offset 參數
    # 如果 export_colored_ply 不支援，你需要手動對 result 內的座標加 offset_vec
    print("进入实际write ply阶段")
    
    if not i_t:
        export_colored_ply(
            result["initial_known"],
            np.zeros_like(result["initial_known"]),
            np.zeros_like(result["initial_known"]),
            initial_path,
            offset=offset_vec # 傳入偏移
        )

    if mode in ["gt", "Mix"]:
        export_colored_ply(
            np.zeros_like(result["initial_known"]),
            result["gt_bridge_target"],
            result["gt_start_completion"],
            gt_path,
            offset=offset_vec
        )

    if mode in ["pred", "Mix"]:
        export_colored_ply(
            np.zeros_like(result["initial_known"]),
            result["pred_bridge_mask"],
            result["pred_grow_mask"],
            pred_path,
            offset=offset_vec
        )

    return {"pred_ply": pred_path, "gt_ply": gt_path, "initial_ply": initial_path}

def print_bridge_grow_metrics(result):
    bridge_m = binary_metrics(result["pred_bridge_mask"], result["gt_bridge_target"])
    grow_m = binary_metrics(result["pred_grow_mask"], result["gt_start_completion"])

    bridge_conn = connectivity_to_target(result["pred_bridge_mask"], result["gt_bridge_target"])
    grow_conn = connectivity_to_target(result["pred_grow_mask"], result["gt_start_completion"])

    print("\n=== Bridge Metrics ===")
    for k, v in bridge_m.items():
        print(f"{k}: {v}")
    print("bridge_target_cover:", bridge_conn)

    print("\n=== Grow Metrics ===")
    for k, v in grow_m.items():
        print(f"{k}: {v}")
    print("grow_target_cover:", grow_conn)

    print("\n=== Node Counts ===")
    print("bridge_node_count:", result["bridge_node_count"])
    print("grow_node_count:", result["grow_node_count"])


def evaluate_samples(
    dataset, 
    sample_ids, 
    grow_threshold=0.50, 
    max_bridge_steps=40,
    pattern_encoder=None,    # 新增
    pattern_bank=None,       # 新增
    bank_embeddings=None     # 新增
):
    rows = []
    for sid in sample_ids:
        if sid < 0 or sid >= len(dataset):
            print(f"[skip] sample id out of range: {sid} (len={len(dataset)})")
            continue

        sample = dataset[sid]
        
        # 傳遞 RAG 參數給推論函數
        result = run_rewritten_inference(
            model=model,
            sample=sample,
            pattern_encoder=pattern_encoder,
            pattern_bank=pattern_bank,
            bank_embeddings=bank_embeddings,
            grow_threshold=grow_threshold,
            max_bridge_steps=max_bridge_steps,
        )

        bridge_m = binary_metrics(result["pred_bridge_mask"], result["gt_bridge_target"])
        grow_m = binary_metrics(result["pred_grow_mask"], result["gt_start_completion"])

        rows.append({
            "sid": sid,
            "bridge_iou": bridge_m["iou"],
            "bridge_dice": bridge_m["dice"],
            "bridge_precision": bridge_m["precision"],
            "bridge_recall": bridge_m["recall"],
            "grow_iou": grow_m["iou"],
            "grow_dice": grow_m["dice"],
            "grow_precision": grow_m["precision"],
            "grow_recall": grow_m["recall"],
            "bridge_touch_frontier": int(touches(result["pred_bridge_mask"], result["current_frontier"])),
            "bridge_touch_seed": int(touches(result["pred_bridge_mask"], result["start_frontier_seed"])),
            "grow_touch_bridge": int(touches(result["pred_grow_mask"], result["pred_bridge_mask"])),
            "grow_touch_seed": int(touches(result["pred_grow_mask"], result["start_frontier_seed"])),
            "bridge_nodes": result["bridge_node_count"],
            "grow_nodes": result["grow_node_count"],
        })
    return rows

if pattern_encoder is not None:
    BANK_EMBEDDINGS = update_pattern_bank_embeddings(PATTERN_BANK, pattern_encoder, device)

In [102]:
# 確保在推論前建立最新的全局 Embeddings


RETR_DEBUG_DIR = os.path.join(ROOT_DIR, "inference_debug_outputs")
os.makedirs(RETR_DEBUG_DIR, exist_ok=True)

#sample_idx = 10
sample_idx = 0
sample = val_ds[sample_idx]

# 執行修正後的 RAG 推論
result = run_rewritten_inference(
    model=model,
    sample=sample,
    pattern_encoder=pattern_encoder,     # 新增
    pattern_bank=PATTERN_BANK,           # 新增
    bank_embeddings=BANK_EMBEDDINGS,     # 新增
    grow_threshold=0.20,
    max_bridge_steps=40,
    max_grow_steps=20,
)

print_bridge_grow_metrics(result)

# 導出可視化 PLY
export_bridge_only_ply(result, out_dir=RETR_DEBUG_DIR, prefix=f"val_{sample_idx}")
export_grow_only_ply(result, out_dir=RETR_DEBUG_DIR, prefix=f"val_{sample_idx}")
export_inference_result_plys(result, out_dir=RETR_DEBUG_DIR, prefix=f"val_{sample_idx}_full")

[RAG] get item len 1

=== Bridge Metrics ===
tp: 3634
fp: 1062
fn: 6869
precision: 0.7738500851788757
recall: 0.3459963819860992
iou: 0.3142239515780372
dice: 0.4781893545628002
bridge_target_cover: 0.3459963819860992

=== Grow Metrics ===
tp: 1109
fp: 6359
fn: 3786
precision: 0.14850026780931977
recall: 0.22655771195097038
iou: 0.09854274035898347
dice: 0.17940629297096175
grow_target_cover: 0.22655771195097038

=== Node Counts ===
bridge_node_count: 4696
grow_node_count: 7468


{'pred_ply': '.\\inference_debug_outputs\\val_0_full_pred.ply',
 'gt_ply': '.\\inference_debug_outputs\\val_0_full_gt.ply'}

In [51]:
import joblib
import os

def save_pattern_bank(bank, file_path):
    """
    將 Pattern Bank 保存到磁盤。
    使用 compress=3 可以在保持讀寫速度的同時，顯著減小 3D 矩陣佔用的空間。
    """
    print(f"Saving pattern bank with {len(bank)} items to {file_path}...")
    # 使用 joblib 保存，compress 參數範圍 0-9，3 通常是性能與大小的平衡點
    joblib.dump(bank, file_path, compress=3)
    print("Save complete.")

# 執行保存
BANK_FILE_PATH = os.path.join(OUT_DIR, "pattern_bank_v1.pkl")
save_pattern_bank(PATTERN_BANK, BANK_FILE_PATH)

Saving pattern bank with 1002 items to .\FRONTIER_EDGE_STATE_RUN\pattern_bank_v1.pkl...
Save complete.


In [43]:
def extract_sample_to_npy(dataset, idx, save_path):
    # 1. 獲取 dataset 中的 sample
    sample = dataset[idx]
    
    # 2. 獲取 initial_known 的體素座標
    grid_coords = np.argwhere(sample["initial_known"] > 0)
    
    # 3. 透過 crop_lo 还原到全局 Voxel 坐標
    global_vox_coords = grid_coords + sample["crop_lo"]
    
    # 4. 乘以 VOX_CM 轉回厘米 (cm)
    pts_cm = global_vox_coords * VOX_CM
    
    # 5. 保存為外部輸入格式
    np.save(save_path, pts_cm)
    print(f"Successfully extracted Sample {idx} to {save_path}. Shape: {pts_cm.shape}")

# 執行提取
extract_sample_to_npy(val_ds, idx=0, save_path="test_input_pc.npy")

Successfully extracted Sample 0 to test_input_pc.npy. Shape: (7495, 3)


Sequence Inference

In [43]:
def run_sequential_growth_prediction_only(initial_sample, directions_list, model, pattern_encoder, pattern_bank, bank_embeddings):
    current_sample = initial_sample.copy()
    history_results = []

    for i, d in enumerate(directions_list):
        print(f"\n>>> Link-Chain Step {i+1}/{len(directions_list)} | Direction: {d}")
        
        # 1. 更新方向
        g_dir = np.array(d, dtype=np.float32)
        g_dir /= (np.linalg.norm(g_dir) + 1e-8)
        current_sample["grow_direction"] = g_dir

        # 2. 執行推論
        result = run_rewritten_inference(
            model=model,
            sample=current_sample,
            pattern_encoder=pattern_encoder,
            pattern_bank=pattern_bank,
            bank_embeddings=bank_embeddings,
            grow_threshold=0.40,
            max_bridge_steps=30,
            max_grow_steps=6
        )
        history_results.append(result)

        # --- 核心修改部分：只提取預測結果 ---
        # 提取第 n 步產生的 Bridge 同 Grow 部分 (Boolean Mask)
        # 注意：唔好加入 result["initial_known"]
        step_prediction = (result["pred_bridge_mask"] > 0) | (result["pred_grow_mask"] > 0)
        
        # 將 Boolean 轉返做 uint8 體素格式
        next_known = step_prediction.astype(np.uint8)

        # 檢查：如果呢步乜都生唔出，就要中斷，否則下一步會報錯
        if next_known.sum() == 0:
            print(f"!!! Warning: Step {i+1} produced no voxels. Sequential growth stopped.")
            break

        # 3. 準備下一步嘅 Sample
        # 新嘅 Frontier 係由呢嚿「新預測出嚟嘅結構」計算出嚟
        next_frontier = frontier_from_known_3d(next_known)

        current_sample = {
            "initial_known": next_known,         # 下一步嘅「地基」係今步嘅「結果」
            "current_frontier": next_frontier,
            "grow_direction": g_dir, 
            "grow_direction_valid": 1.0,
            "n_shift": 0.0,
            "start_frontier_seed": next_frontier,
            
            # Dummy 補全
            "pattern_start_frontier": np.zeros_like(next_known),
            "preview_known": np.zeros_like(next_known),
            "bridge_target": np.zeros_like(next_known),
            "bridge_target_full": np.zeros_like(next_known),
            "start_completion_target": np.zeros_like(next_known),
        }

        # 導出
        export_inference_result_plys(result, out_dir="./chain_results", prefix=f"chain_step_{i+1}")

    return history_results

In [46]:
def build_npy_to_input(file_path):

    pts_cm = np.load(file_path).astype(np.float32)
    def pad_to_shape(grid, target_shape):
        out = np.zeros(target_shape, dtype=np.uint8)
        sx, sy, sz = grid.shape[:3]
        tx, ty, tz = target_shape
        if sx > tx or sy > ty or sz > tz:
            raise ValueError("grid larger than target_shape")
        # 先用左上角放置，测试最直接
        out[:sx, :sy, :sz] = grid
        return out

    grid, meta = build_voxel_grid_from_centers(pts_cm, margin=GRID_MARGIN)
    print(f"DEBUG: Grid actual shape: {grid.shape}")

    grid = pad_to_shape(grid, (500, 500, 82))   # 先用 val shape 做对照测试
    frontier = frontier_from_known_3d(grid)


    dummy = np.zeros_like(grid, dtype=np.uint8)

    current_sample_if = {
        "initial_known": grid,
        "current_frontier": frontier,

        "start_frontier_seed": frontier,
        "pattern_start_frontier": dummy,

        "preview_known": grid.copy(),

        "pattern_start_mask": dummy,
        "start_completion_target": dummy,

        "bridge_target": dummy,
        "bridge_target_full": dummy,

        "grow_direction": np.array([1,0,0],dtype=np.float32),
        "grow_direction_valid": 1.0,

        "n_shift": 0.0,
    }
    return current_sample_if

In [51]:
#npy_file_path = r".\test_input_pc.npy"
npy_file_path = r".\Test4.npy"
pts_cm = np.load(npy_file_path).astype(np.float32)
print(pts_cm.shape)



(97636, 6)


In [48]:
# --- 初始化 ---
# 1. 攞一個初始起點 (可以是 val_ds 或者是外部 npy)

#npy_file_path = r".\test_input_pc.npy"
npy_file_path = r".\Test4.npy"
current_sample_NPY = build_npy_to_input(npy_file_path)
current_sample = current_sample_NPY.copy()


#current_sample = val_ds[0].copy() 

step_idx = 0

current_sample_if = current_sample.copy()


print("=== 互動式連鎖生長模式啟動 ===")
print("指令格式: x y z (例如: 1 0 0 代表向正X生長; 0 0 1 代表向上生長)")
print("輸入 'q' 退出, 輸入 'r' 重置起點")

Inital_Handle = False

while True:
    # 2. 手動輸入方向
    user_input = input(f"\n[Step {step_idx}] 請輸入生長方向 (x y z): ").strip().lower()

    if user_input == 'q':
        print("停止生成。")
        break
    
    if user_input == 'r':
        current_sample_if = current_sample.copy()
        step_idx = 0
        print("已重置回初始狀態。")
        continue

    try:
        # 解析輸入坐標
        d = [float(x) for x in user_input.split()]
        if len(d) != 3:
            print("錯誤: 請輸入三個數字，以空格分開。")
            continue
        
        g_dir = np.array(d, dtype=np.float32)
        norm = np.linalg.norm(g_dir)
        if norm < 1e-8:
            print("錯誤: 方向向量長度不能為 0。")
            continue
        g_dir /= norm
    except ValueError:
        print("錯誤: 請輸入有效的數字格式。")
        continue

    # 3. 執行單步推論
    print(f"正在執行推論... 方向: {g_dir}")
    
    current_sample_if["grow_direction"] = g_dir

    # 調用原本的推論函數
    result = run_rewritten_inference(
        model=model,
        sample=current_sample_if,
        pattern_encoder=pattern_encoder,
        pattern_bank=PATTERN_BANK,
        bank_embeddings=BANK_EMBEDDINGS,
        grow_threshold=0.48,
        max_bridge_steps=10,
        max_grow_steps=10
    )

    # 4. 檢查結果並提取 Prediction Only
    step_prediction = (result["pred_bridge_mask"] > 0) | (result["pred_grow_mask"] > 0)
    next_known = step_prediction.astype(np.uint8)

    if next_known.sum() == 0:
        print("!!! 警告: 呢步乜都生唔出 (模型信心值太低)。請試下換個方向。")
        continue

    # 5. 更新狀態 (準備下一步)
    next_frontier = frontier_from_known_3d(next_known)
    
    # 建立下一輪的 Sample
    current_sample = {
        "initial_known": next_known,
        "current_frontier": next_frontier,
        "grow_direction": g_dir, 
        "grow_direction_valid": 1.0,
        "n_shift": 0.0,
        "start_frontier_seed": next_frontier,
        
        # 補全 Dummy 欄位
        "pattern_start_frontier": np.zeros_like(next_known),
        "preview_known": np.zeros_like(next_known),
        "bridge_target": np.zeros_like(next_known),
        "bridge_target_full": np.zeros_like(next_known),
        "start_completion_target": np.zeros_like(next_known),
    }

    # 6. 保存與反饋
    step_idx += 1
    out_name = f"manual_chain_step_{step_idx}"
    print(f"[Step {step_idx}] 成功生長 {next_known.sum()} 個新體素。正在導出 PLY...")
    export_inference_result_plys_Split(result, out_dir="./manual_results", prefix=out_name,mode = "Mix",i_t= Inital_Handle,grow_dir=g_dir)
    Inital_Handle = True
    print(f"[OK] 第 {step_idx} 步完成。已導出至 ./manual_results/{out_name}.ply")
    print(f"新增體素量: {next_known.sum()}")

DEBUG: Grid actual shape: (103, 85, 56)
=== 互動式連鎖生長模式啟動 ===
指令格式: x y z (例如: 1 0 0 代表向正X生長; 0 0 1 代表向上生長)
輸入 'q' 退出, 輸入 'r' 重置起點
正在執行推論... 方向: [0. 1. 0.]
[RAG] get item len 1
[Step 1] 成功生長 5919 個新體素。正在導出 PLY...
进入实际write ply阶段
Start Writtting PLY
Start Writtting PLY
Start Writtting PLY
[OK] 第 1 步完成。已導出至 ./manual_results/manual_chain_step_1.ply
新增體素量: 5919
正在執行推論... 方向: [0. 1. 0.]
[RAG] get item len 1
[Step 2] 成功生長 6191 個新體素。正在導出 PLY...
計算到現有 PLY 的最大邊界: [ 99. 142.  52.], 生長方向: [0. 1. 0.]
进入实际write ply阶段
Start Writtting PLY
Start Writtting PLY
[OK] 第 2 步完成。已導出至 ./manual_results/manual_chain_step_2.ply
新增體素量: 6191
停止生成。


In [2]:
import numpy as np
def inspect_npy_content(file_path):
    """
    解析並打印 NPY 文件嘅詳細結構，用於 Debug 比對
    """
    print(f"{'='*20} Inspecting: {file_path} {'='*20}")
    try:
        data = np.load(file_path, allow_pickle=True)
        
        # 1. 基本信息
        print(f"1. Basic Info:")
        print(f"   - Array Shape: {data.shape}")
        print(f"   - Data Type:   {data.dtype}")
        
        # 2. 數值範圍 (如果是點雲或 Voxel)
        if len(data.shape) >= 2 and data.shape[1] >= 3:
            print(f"2. Geometric Range (Min/Max):")
            print(f"   - X: {data[:, 0].min():.2f} to {data[:, 0].max():.2f}")
            print(f"   - Y: {data[:, 1].min():.2f} to {data[:, 1].max():.2f}")
            print(f"   - Z: {data[:, 2].min():.2f} to {data[:, 2].max():.2f}")
            
            # 3. 顏色/額外屬性信息 (如果有)
            if data.shape[1] > 3:
                print(f"3. Attribute Range (Columns 4-6):")
                print(f"   - Col 4: {data[:, 3].min():.2f} to {data[:, 3].max():.2f}")
                print(f"   - Mean Values: {np.mean(data, axis=0)}")
        
        # 4. 數據樣本
        print(f"4. First 2 Rows Sample:\n{data[:2]}")
        
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
    print(f"{'='*50}\n")

inspect_npy_content(r".\test_input_pc.npy")
inspect_npy_content(r".\Test4.npy")

==================== Inspecting: .\test_input_pc.npy ====================
1. Basic Info:
   - Array Shape: (7495, 3)
   - Data Type:   float64
2. Geometric Range (Min/Max):
   - X: 2200.00 to 3860.00
   - Y: 1720.00 to 3140.00
   - Z: 260.00 to 880.00
4. First 2 Rows Sample:
[[2200. 2360.  320.]
 [2200. 2360.  340.]]

==================== Inspecting: .\Test4.npy ====================
1. Basic Info:
   - Array Shape: (20423, 3)
   - Data Type:   float32
2. Geometric Range (Min/Max):
   - X: -1530.00 to 1730.00
   - Y: 290.00 to 2990.00
   - Z: -390.00 to 350.00
4. First 2 Rows Sample:
[[-1530.  2930.  -130.]
 [-1490.  2910.    30.]]



In [48]:
import numpy as np
import os

def export_different_resolutions_from_npz(npz_path, resolutions_cm, vox_cm=20.0, output_dir="res_comparison"):
    """
    讀取 QF_2.py 產出的 .npz 檔案，還原點雲後根據多種體素尺寸導出 PLY。
    
    Args:
        npz_path: .npz 檔案路徑。
        resolutions_cm: list, 想要輸出的體素大小（例如 [10, 20, 40]）。
        vox_cm: 原始製作快取時使用的 VOX_CM (QF_2.py 預設為 20.0)。
        output_dir: 儲存目錄。
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # 1. 載入 .npz 並檢查是否為空
    data = np.load(npz_path, allow_pickle=True)
    if "empty" in data.files:
        print(f"[Skip] {npz_path} 是空幀快取。")
        return

    # 2. 還原點雲座標
    # 根據 QF_2.py 的邏輯：Global_Pts = centers + (local_pts * (vox_cm / 2))
    centers = data['centers_cm']  # Shape: (V, 3)
    local_pts = data['local_pts'].astype(np.float32)  # Shape: (V, 256, 3)
    half = vox_cm / 2.0
    
    # 使用 Broadcasting 將中心點與偏移量相加
    raw_pts = centers[:, np.newaxis, :] + (local_pts * half)
    raw_pts = raw_pts.reshape(-1, 3) # 打平回 (N, 3)
    
    print(f"[Info] 從 NPZ 還原成功，點數: {len(raw_pts)}")

    for res in resolutions_cm:
        # 3. 進行體素化 (Voxelization)
        vox_indices = np.floor(raw_pts / res).astype(np.int32)
        unique_voxels = np.unique(vox_indices, axis=0)
        
        # 4. 座標對齊與還原
        # 讓最小點從 0 開始，方便在 3D 軟體預覽
        min_bound = unique_voxels.min(axis=0)
        aligned_voxels = unique_voxels - min_bound
        export_pts = (aligned_voxels + 0.5) * res
        
        # 5. 顏色設定
        color_val = int(np.clip(res * 5, 0, 255))
        colors = np.tile([255 - color_val, 150, color_val], (len(export_pts), 1)).astype(np.uint8)
        
        # 6. 導出 PLY
        file_name = f"res_{res}cm_from_npz_{len(export_pts)}.ply"
        save_path = os.path.join(output_dir, file_name)
        
        # 呼叫你現有的寫入函數
        write_points_rgb_ply(export_pts, colors, save_path)
        print(f"[Done] 解析度 {res}cm -> {save_path}")

# === 使用範例 ===
# 假設你的快取在 LearnCache_seq_fast_Kidasaki/frame_cache/000000.npz
path = r".\Frames\LearnCache_seq_fast_Kidoshina\frame_cache\000004.npz"
resolutions = [10, 20,30, 40]
export_different_resolutions_from_npz(path, resolutions)

[Info] 從 NPZ 還原成功，點數: 828160
[Done] 解析度 10cm -> res_comparison\res_10cm_from_npz_9762.ply
[Done] 解析度 20cm -> res_comparison\res_20cm_from_npz_3235.ply
[Done] 解析度 30cm -> res_comparison\res_30cm_from_npz_1648.ply
[Done] 解析度 40cm -> res_comparison\res_40cm_from_npz_969.ply
